This Juypyter-Notebook contains all the .py files codes, is generated for easy to use case.
#Adaptive NOON-State circuit: forward model, training, and validation.
Author: Simanshu Kumar

In [ ]:
"""
noon_forward_model.py
=====================
Adaptive NOON-State circuit: forward model, training, and validation.

Paper: "Quantum-Enhanced Single-Parameter Phase Estimation
        with Adaptive NOON States"
Authors: Simanshu Kumar, Nandan S Bisht
Repository: https://github.com/simanshukumar369/noon-state-adaptive-metrology

Circuit layout
--------------
  q[0]: Coherent(α,0) → R(d_coh) ─┐
                                    ├─ BS(θ₁,φ₁) → R(φ_est) → BS(θ₂,φ₂) → detect
  q[1]: Squeezed(r,0) → R(d_sq)  ─┘

  Trainable: r, log_gamma, d_coh, d_sq, θ₁, φ₁, θ₂, φ₂
  Fixed:     φ_est (scanned externally to produce coincidence fringes)

Usage (notebook)
----------------
  # paste whole file in one cell, then:
  run_stage0([2])
  run_stage1(N=2)
  run_stage3([2])
  params, history = run_stage4(N=2, n_steps=200, lr=0.01)

Usage (CLI)
-----------
  python noon_forward_model.py --N 2 --stage 3
  python noon_forward_model.py --N 2 --stage 4 --steps 200 --lr 0.01
"""

# ── stdlib ────────────────────────────────────────────────────────────────────
import os, sys, time, argparse
import numpy as np

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

# ── third-party ───────────────────────────────────────────────────────────────
import tensorflow as tf
import strawberryfields as sf
from strawberryfields.ops import Coherent, Squeezed, BSgate, Rgate
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns


# =============================================================================
# 0.  CONSTANTS
# =============================================================================

R_BASE = 0.35

GAMMA_OPT = {          # optimal coh/sq ratio from Afek et al. 2010
    2: 1.0,
    3: 1.0,
    4: np.sqrt(3),
    5: 9.0 / (np.sqrt(10) + 1),
}

PATTERNS = {           # coincidence patterns (N1, N2) per total N
    2: [(1, 1), (2, 0)],
    3: [(2, 1), (3, 0)],
    4: [(3, 1), (2, 2)],
    5: [(3, 2)],
}

COLOR = {2: "tab:green", 3: "tab:orange", 4: "tab:blue", 5: "tab:red"}

PHI_SCAN = np.linspace(0, 2 * np.pi, 400)


# =============================================================================
# 1.  PARAMETER CONTAINER
# =============================================================================

class CircuitParams:
    """
    Eight trainable tf.Variables for the 2-mode adaptive NOON circuit.

    Direct variables
    ----------------
      r          squeezing magnitude (clipped to (0, r_max])
      log_gamma  log coh/sq ratio;  gamma = exp(log_gamma) > 0
      d_coh      input phase, mode 0
      d_sq       input phase, mode 1
      theta1     BS1 angle
      phi1       BS1 phase
      theta2     BS2 angle
      phi2       BS2 phase

    Derived (computed on the fly, gradient-safe)
    --------
      gamma  = exp(log_gamma)
      alpha  = sqrt(gamma * |r|)   coherent amplitude
    """

    def __init__(self, r, log_gamma, d_coh, d_sq,
                 theta1, phi1, theta2, phi2,
                 N=2, r_max=2.0, dtype=tf.float64):
        self.N     = N
        self.r_max = r_max

        def v(name, val):
            return tf.Variable(float(val), dtype=dtype,
                               name=name, trainable=True)

        self.r         = v("r",         r)
        self.log_gamma = v("log_gamma", log_gamma)
        self.d_coh     = v("d_coh",     d_coh)
        self.d_sq      = v("d_sq",      d_sq)
        self.theta1    = v("theta1",    theta1)
        self.phi1      = v("phi1",      phi1)
        self.theta2    = v("theta2",    theta2)
        self.phi2      = v("phi2",      phi2)

    # ── derived quantities ────────────────────────────────────────────────────
    @property
    def gamma(self):
        return tf.exp(self.log_gamma)

    @property
    def alpha(self):
        return tf.sqrt(self.gamma * tf.abs(self.r))

    @property
    def r_clipped(self):
        return tf.clip_by_value(tf.abs(self.r), 1e-4, self.r_max)

    @property
    def trainable_variables(self):
        return [self.r, self.log_gamma, self.d_coh, self.d_sq,
                self.theta1, self.phi1, self.theta2, self.phi2]

    # ── constructors ─────────────────────────────────────────────────────────
    @classmethod
    def from_afek(cls, N=2, r_base=R_BASE, r_max=2.0,
                  dtype=tf.float64) -> "CircuitParams":
        """Afek et al. (2010) working point."""
        gamma_init = GAMMA_OPT.get(N, 1.0)
        return cls(
            r=r_base, log_gamma=np.log(gamma_init),
            d_coh=np.pi/2, d_sq=0.0,
            theta1=np.pi/4, phi1=0.0,
            theta2=np.pi/4, phi2=np.pi,
            N=N, r_max=r_max, dtype=dtype,
        )

    @classmethod
    def random(cls, N=2, r_base=R_BASE, seed=42,
               dtype=tf.float64) -> "CircuitParams":
        """Random init for multi-start optimisation."""
        rng = np.random.default_rng(seed)
        return cls(
            r=r_base * (0.5 + rng.random()),
            log_gamma=rng.uniform(-1.0, 1.5),
            d_coh=rng.uniform(0, 2*np.pi),
            d_sq=rng.uniform(0, 2*np.pi),
            theta1=rng.uniform(0, np.pi),
            phi1=rng.uniform(0, 2*np.pi),
            theta2=rng.uniform(0, np.pi),
            phi2=rng.uniform(0, 2*np.pi),
            N=N, dtype=dtype,
        )

    # ── display ───────────────────────────────────────────────────────────────
    def print_summary(self):
        g = float(self.gamma)
        a = float(self.alpha)
        print(f"\n  CircuitParams  (N={self.N})")
        print(f"  {'─'*50}")
        print(f"  r  (squeezing)      {float(self.r):.6f}")
        print(f"  gamma  (coh/sq)     {g:.6f}  [log_gamma={float(self.log_gamma):.4f}]")
        print(f"  alpha  (coh amp)    {a:.6f}  = sqrt(gamma*r)")
        print(f"  d_coh               {float(self.d_coh):.6f} rad")
        print(f"  d_sq                {float(self.d_sq):.6f} rad")
        print(f"  theta1, phi1        {float(self.theta1):.4f}, {float(self.phi1):.4f} rad")
        print(f"  theta2, phi2        {float(self.theta2):.4f}, {float(self.phi2):.4f} rad")
        print()

    # alias for backward compat
    summary = print_summary

    def to_dict(self):
        return {k: float(getattr(self, k).numpy())
                for k in ("r","log_gamma","d_coh","d_sq",
                           "theta1","phi1","theta2","phi2")} | {"N": self.N}


# =============================================================================
# 2.  CIRCUIT  (SF TF backend — differentiable)
# =============================================================================

def build_engine(cutoff: int) -> sf.Engine:
    return sf.Engine("tf", backend_options={"cutoff_dim": cutoff, "pure": True})


def build_program(cutoff: int):
    prog = sf.Program(2)
    fp   = {k: prog.params(k) for k in
            ["r","alpha","d_coh","d_sq","theta1","phi1","phi_est","theta2","phi2"]}
    with prog.context as q:
        Coherent(fp["alpha"], 0.0)       | q[0]
        Squeezed(fp["r"],     0.0)       | q[1]
        Rgate(fp["d_coh"])               | q[0]
        Rgate(fp["d_sq"])                | q[1]
        BSgate(fp["theta1"], fp["phi1"]) | (q[0], q[1])
        Rgate(fp["phi_est"])             | q[0]
        BSgate(fp["theta2"], fp["phi2"]) | (q[0], q[1])
    return prog, fp


def run_circuit(params: CircuitParams, phi_est, cutoff: int):
    """
    One differentiable forward pass through the full MZ circuit.

    Creates a fresh sf.Engine each call (required by SF TF backend).

    Returns
    -------
    probs  : tf.Tensor [cutoff, cutoff]   probs[n0,n1] = P(n0,n1)
    parity : tf.Tensor scalar             sum_k (-1)^k P(k, N-k)
    """
    eng  = build_engine(cutoff)
    prog, _ = build_program(cutoff)
    result = eng.run(prog, args={
        "r":       params.r_clipped,
        "alpha":   params.alpha,
        "d_coh":   params.d_coh,
        "d_sq":    params.d_sq,
        "theta1":  params.theta1,
        "phi1":    params.phi1,
        "phi_est": tf.cast(phi_est, tf.float64),
        "theta2":  params.theta2,
        "phi2":    params.phi2,
    })
    probs  = result.state.all_fock_probs()
    N      = params.N
    parity = tf.constant(0.0, dtype=tf.float64)
    for k in range(N + 1):
        parity = parity + float((-1)**k) * tf.cast(probs[k, N-k], tf.float64)
    return probs, parity


# =============================================================================
# 3.  FRINGE SCANS
# =============================================================================

def tf_fringe_scan(params: CircuitParams, N1: int, N2: int,
                   phi_arr: np.ndarray, cutoff=None, verbose=True):
    """Scan phi_arr with TF backend. Returns (prob_arr, parity_arr) as numpy."""
    N = N1 + N2
    if cutoff is None:
        cutoff = max(3*N+4, 12)
    prob_arr   = np.zeros(len(phi_arr))
    parity_arr = np.zeros(len(phi_arr))
    for i, phi in enumerate(phi_arr):
        probs, parity = run_circuit(params, tf.constant(phi, tf.float64), cutoff)
        prob_arr[i]   = float(tf.cast(probs[N1, N2], tf.float64))
        parity_arr[i] = float(parity)
        if verbose and (i % 80 == 0 or i == len(phi_arr)-1):
            print(f"  TF [{i+1:3d}/{len(phi_arr)}]  phi={phi:.3f}  "
                  f"P({N1},{N2})={prob_arr[i]:.4e}")
    return prob_arr, parity_arr


def fock_fringe_scan(params: CircuitParams, N1: int, N2: int,
                     phi_arr: np.ndarray, cutoff=None):
    """Fock-backend reference scan (NOT differentiable). Returns (prob_arr, parity_arr)."""
    N = N1 + N2
    if cutoff is None:
        cutoff = max(3*N+4, 12)
    r_v  = float(params.r_clipped)
    a_v  = float(params.alpha)
    dc_v = float(params.d_coh)
    ds_v = float(params.d_sq)
    t1_v = float(params.theta1)
    p1_v = float(params.phi1)
    t2_v = float(params.theta2)
    p2_v = float(params.phi2)
    prob_arr   = np.zeros(len(phi_arr))
    parity_arr = np.zeros(len(phi_arr))
    engine = sf.Engine("fock", backend_options={"cutoff_dim": cutoff})
    for i, phi in enumerate(phi_arr):
        prog = sf.Program(2)
        with prog.context as q:
            Coherent(a_v, 0.0) | q[0]
            Squeezed(r_v, 0.0) | q[1]
            Rgate(dc_v)        | q[0]
            Rgate(ds_v)        | q[1]
            BSgate(t1_v, p1_v) | (q[0], q[1])
            Rgate(phi)         | q[0]
            BSgate(t2_v, p2_v) | (q[0], q[1])
        state = engine.run(prog).state
        prob_arr[i] = state.fock_prob([N1, N2])
        parity_arr[i] = sum((-1)**k * state.fock_prob([k, N-k]) for k in range(N+1))
    return prob_arr, parity_arr


# =============================================================================
# 4.  FISHER INFORMATION HELPERS  (numpy, for validation)
# =============================================================================

def classical_CFI_from_fringe(prob_arr, phi_arr):
    """F(phi) = (dP/dphi)^2 / P  via numpy.gradient (for Stage 2/3 validation)."""
    dP   = np.gradient(prob_arr, phi_arr)
    mask = prob_arr > 1e-10
    CFI  = np.zeros_like(prob_arr)
    CFI[mask] = dP[mask]**2 / prob_arr[mask]
    return CFI


def peak_CFI(prob_arr, phi_arr):
    return float(np.max(classical_CFI_from_fringe(prob_arr, phi_arr)))


def fft_dominant_bin(signal):
    mag = np.abs(np.fft.rfft(signal - signal.mean()))
    return int(np.argmax(mag[1:])) + 1


def normalise(arr):
    span = arr.max() - arr.min()
    return (arr - arr.min()) / span if span > 1e-14 else np.zeros_like(arr)


# =============================================================================
# 5.  DIFFERENTIABLE CFI  (for Module 2 training)
# =============================================================================

def compute_CFI_differentiable(
    params: CircuitParams,
    N1: int,
    N2: int,
    cutoff: int,
    K: int = 8,
    eps_frac: float = 0.05,
) -> tuple:
    """
    Differentiable peak CFI via regularised spectral derivative.

    Algorithm
    ---------
    1. Sample P(phi_k) at K equally-spaced phi in [0, 2*pi).
    2. Compute exact dP/dphi at each sample via DFT differentiation
       (multiply Fourier coefficients by i*m, then IFFT) — zero sinc bias,
       captures ALL harmonics up to K//2.
    3. Regularised CFI at each phi:
           CFI_k = (dP_k)^2 / (P_k + eps)
       where eps = eps_frac * mean(P)  avoids blow-up at fringe troughs
       while barely affecting the peak (peak P >> eps).
    4. Differentiable smooth-max via LogSumExp.

    This correctly handles multi-harmonic fringes (e.g. |2,0>) and
    near-zero troughs (e.g. |1,1>).  K=24 covers harmonics up to 12,
    sufficient for all N<=5 patterns.

    Returns
    -------
    CFI_peak : tf.Tensor scalar  regularised smooth-max peak CFI
    B        : tf.Tensor scalar  (max-min)/2 of P samples, for monitoring
    A        : tf.Tensor scalar  mean(P) = post-selection rate
    """
    dphi     = 2.0 * np.pi / K
    phi_vals = [tf.constant(k * dphi, dtype=tf.float64) for k in range(K)]

    # -- K circuit evaluations ------------------------------------------------
    prob_list = []
    for phi_k in phi_vals:
        probs_k, _ = run_circuit(params, phi_k, cutoff)
        prob_list.append(tf.cast(tf.math.real(tf.cast(probs_k[N1, N2], tf.complex64)), tf.float64))
    P = tf.stack(prob_list)   # [K]

    # -- spectral derivative: exact for truncated Fourier series --------------
    P_c = tf.cast(P, tf.complex128)
    C   = tf.signal.fft(P_c) / tf.cast(K, tf.complex128)

    m_pos   = tf.cast(tf.range(K // 2 + 1),       tf.complex128)
    m_neg   = tf.cast(tf.range(-(K // 2 - 1), 0), tf.complex128)
    m       = tf.concat([m_pos, m_neg], axis=0)     # [K]
    C_deriv = tf.constant(1j, tf.complex128) * m * C
    dPdphi  = tf.math.real(
        tf.signal.ifft(C_deriv) * tf.cast(K, tf.complex128)
    )   # [K]

    # -- regularised CFI: eps = eps_frac * mean(P) ---------------------------
    A   = tf.reduce_mean(P)
    eps = eps_frac * tf.stop_gradient(A)   # adaptive floor, no grad through eps
    CFI_k = dPdphi ** 2 / (P + eps + 1e-30)   # [K]

    # -- differentiable smooth-max -------------------------------------------
    beta       = 50.0
    CFI_max_sg = tf.stop_gradient(tf.reduce_max(CFI_k))
    CFI_peak   = (1.0 / beta) * tf.math.log(
        tf.reduce_sum(tf.exp(beta * (CFI_k - CFI_max_sg))) + 1e-30
    ) + CFI_max_sg

    B = (tf.reduce_max(P) - tf.reduce_min(P)) / 2.0
    return CFI_peak, B, A


def compute_total_CFI(
    params: CircuitParams,
    cutoff: int,
    K: int = 8,
) -> tf.Tensor:
    """
    Sum of peak CFIs over all coincidence patterns for params.N.
    Differentiable — use inside GradientTape for training.
    """
    total = tf.constant(0.0, dtype=tf.float64)
    for N1, N2 in PATTERNS.get(params.N, []):
        CFI_k, _, _ = compute_CFI_differentiable(params, N1, N2, cutoff, K=K)
        total = total + CFI_k
    return total


def compute_total_CFI_detailed(
    params: CircuitParams,
    cutoff: int,
    K: int = 8,
) -> tuple:
    """
    Same as compute_total_CFI but returns (total_float, per_pattern_dict).
    For logging only — converts to Python floats so NOT inside GradientTape.
    """
    hl    = float(params.N) ** 2
    total = 0.0
    detail = {}
    for N1, N2 in PATTERNS.get(params.N, []):
        CFI_k, B_k, A_k = compute_CFI_differentiable(params, N1, N2, cutoff, K=K)
        v = float(CFI_k)
        total += v
        detail[f"|{N1},{N2}>"] = dict(CFI=v, CFI_hl=v/hl,
                                      B=float(B_k), A=float(A_k))
    return total, detail


# =============================================================================
# 6.  VALIDATION  (Modules 1 — Stage 2 / Stage 3)
# =============================================================================

def probe_gradients(params: CircuitParams, cutoff=12, phi_test=np.pi/4) -> dict:
    """Verify gradient flow through all 8 trainable variables."""
    N1, N2 = params.N // 2, params.N - params.N // 2
    with tf.GradientTape() as tape:
        probs, _ = run_circuit(params, tf.constant(phi_test, tf.float64), cutoff)
        target   = tf.cast(probs[N1, N2], tf.float64)
    grads = tape.gradient(target, params.trainable_variables)
    print(f"\n-- Gradient probe  dP({N1},{N2})/dtheta  at phi={phi_test:.3f} --")
    result = {}
    all_ok = True
    for var, grad in zip(params.trainable_variables, grads):
        name = var.name.split(":")[0]
        if grad is None:
            print(f"  {name:14s} ->  WARNING: None  (gradient not flowing!)")
            all_ok = False
        else:
            print(f"  {name:14s} ->  {float(grad):.6e}")
        result[name] = grad
    status = "All gradients OK." if all_ok else "Some gradients None — check wiring."
    print(f"\n  {status}\n")
    return result


def validate_pattern(N1: int, N2: int,
                     tol_shape=0.01, tol_CFI=0.05, verbose=True) -> dict:
    """Full TF-vs-Fock quantitative validation for one (N1, N2) pattern."""
    N      = N1 + N2
    params = CircuitParams.from_afek(N=N)

    if verbose:
        print(f"\n{'─'*60}")
        print(f"  Pattern |{N1},{N2}>   N = {N}")
        print(f"{'─'*60}")
        params.print_summary()

    # TF scan
    t0 = time.time()
    prob_tf, parity_tf = tf_fringe_scan(params, N1, N2, PHI_SCAN, verbose=verbose)
    t_tf = time.time() - t0

    # Fock scan (reference)
    t0 = time.time()
    prob_fk, parity_fk = fock_fringe_scan(params, N1, N2, PHI_SCAN)
    t_fk = time.time() - t0

    # shape
    max_err  = float(np.max(np.abs(normalise(prob_tf) - normalise(prob_fk))))
    shape_ok = max_err < tol_shape

    # FFT
    bin_tf  = fft_dominant_bin(parity_tf)
    bin_fk  = fft_dominant_bin(parity_fk)

    # CFI — compare on normalised fringes (amplitude-agnostic)
    CFI_tf      = peak_CFI(prob_tf,            PHI_SCAN)
    CFI_fk      = peak_CFI(prob_fk,            PHI_SCAN)
    CFI_tf_norm = peak_CFI(normalise(prob_tf), PHI_SCAN)
    CFI_fk_norm = peak_CFI(normalise(prob_fk), PHI_SCAN)
    CFI_rel     = abs(CFI_tf_norm - CFI_fk_norm) / (CFI_fk_norm + 1e-30)
    CFI_ok      = CFI_rel < tol_CFI
    amp_ratio   = (prob_tf.max() + 1e-30) / (prob_fk.max() + 1e-30)

    if verbose:
        ok = lambda b: "PASS" if b else "FAIL"
        print(f"\n  Shape match    max_err={max_err:.5f}     {ok(shape_ok)}")
        print(f"  FFT bin        TF={bin_tf}  Fock={bin_fk}  expected={N}  {ok(bin_tf==N)}")
        print(f"  CFI (norm)     TF={CFI_tf_norm:.4f}  Fock={CFI_fk_norm:.4f}  "
              f"rel_err={CFI_rel:.4f}  {ok(CFI_ok)}")
        print(f"  CFI (raw TF)   {CFI_tf:.4f}   amplitude ratio TF/Fock={amp_ratio:.3f}")
        print(f"  CFI / N^2      {CFI_tf/N**2:.4f}")
        print(f"  Timing         TF={t_tf:.1f}s  Fock={t_fk:.1f}s")

    return dict(N=N, N1=N1, N2=N2,
                shape_ok=shape_ok, fft_ok=(bin_tf==N), CFI_ok=CFI_ok,
                max_err=max_err, amp_ratio=amp_ratio,
                CFI_tf=CFI_tf, CFI_fk=CFI_fk,
                CFI_tf_norm=CFI_tf_norm, CFI_fk_norm=CFI_fk_norm,
                CFI_norm=CFI_tf/N**2,
                prob_tf=prob_tf, prob_fk=prob_fk,
                parity_tf=parity_tf, t_tf=t_tf)


# =============================================================================
# 7.  PLOTTING
# =============================================================================

def plot_single_fringe(N1, N2, prob_tf, prob_fk,
                       save_path=None, show=False):
    N     = N1 + N2
    color = COLOR[N]
    sns.set_style("whitegrid")
    plt.rcParams.update({"font.family": "serif", "font.size": 13,
                          "axes.labelweight": "bold"})
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4), constrained_layout=True)
    x_ticks = [0, np.pi/2, np.pi, 3*np.pi/2, 2*np.pi]
    x_labs  = ["0", r"$\pi/2$", r"$\pi$", r"$3\pi/2$", r"$2\pi$"]

    ax1.plot(PHI_SCAN, prob_fk, color=color, lw=2.2, label="Fock (reference)")
    ax1.plot(PHI_SCAN, prob_tf, "k--", lw=1.6, alpha=0.85, label="TF backend")
    ax1.fill_between(PHI_SCAN,
                     prob_fk - np.abs(prob_tf - prob_fk),
                     prob_fk + np.abs(prob_tf - prob_fk),
                     alpha=0.15, color=color, label="|error|")
    ax1.set_xticks(x_ticks); ax1.set_xticklabels(x_labs)
    ax1.set_xlabel("MZ phase  phi (rad)")
    ax1.set_ylabel(f"P({N1},{N2})")
    ax1.set_title(f"|{N1},{N2}>  N={N}  -- Fringe Overlay\n"
                  f"max norm err = {np.max(np.abs(normalise(prob_tf)-normalise(prob_fk))):.2e}",
                  fontweight="bold")
    ax1.legend(fontsize=10)

    CFI_tf = classical_CFI_from_fringe(prob_tf, PHI_SCAN)
    CFI_fk = classical_CFI_from_fringe(prob_fk, PHI_SCAN)
    ax2.plot(PHI_SCAN, CFI_fk, color=color, lw=2.0, label="CFI Fock")
    ax2.plot(PHI_SCAN, CFI_tf, "k--", lw=1.5, alpha=0.85, label="CFI TF")
    ax2.axhline(N**2, color="red", lw=1.2, ls=":",
                label=f"Heisenberg limit N^2={N**2}")
    ax2.set_xticks(x_ticks); ax2.set_xticklabels(x_labs)
    ax2.set_xlabel("MZ phase  phi (rad)")
    ax2.set_ylabel("Classical CFI")
    ax2.set_title(f"Classical Fisher Information\nCFI_peak/N^2 = {np.max(CFI_tf)/N**2:.3f}",
                  fontweight="bold")
    ax2.legend(fontsize=10)
    fig.suptitle("Adaptive NOON-state -- Forward Model Validation  (Afek 2010 init)",
                 fontsize=12, fontweight="bold")

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"  Saved -> {save_path}")
    if show: plt.show()
    plt.close(fig)


def plot_validation_grid(results, save_path="validation_plot.png", show=False):
    sns.set_style("whitegrid")
    plt.rcParams.update({"font.family": "serif", "font.size": 12})
    n     = len(results)
    ncols = 2
    nrows = (n + 1) // 2
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, 3.5*nrows),
                              constrained_layout=True)
    axes = np.array(axes).flatten()
    x_ticks = [0, np.pi/2, np.pi, 3*np.pi/2, 2*np.pi]
    x_labs  = ["0", r"$\pi/2$", r"$\pi$", r"$3\pi/2$", r"$2\pi$"]

    for ax, res in zip(axes, results):
        N1, N2, N = res["N1"], res["N2"], res["N"]
        color = COLOR[N]
        nf = normalise(res["prob_fk"])
        nt = normalise(res["prob_tf"])
        ax.plot(PHI_SCAN, nf, color=color, lw=2.0, label="Fock")
        ax.plot(PHI_SCAN, nt, "k--", lw=1.4, alpha=0.85, label="TF")
        ax2 = ax.twinx()
        ax2.plot(PHI_SCAN, (nt-nf)*10, ":", color="grey", lw=0.9, alpha=0.7)
        ax2.set_ylim(-0.5, 0.5); ax2.set_yticks([-0.2, 0, 0.2])
        ax2.tick_params(labelsize=8, colors="grey")
        ax2.set_ylabel("Residual x10", fontsize=8, color="grey")
        ax.set_xlim(0, 2*np.pi)
        ax.set_xticks(x_ticks); ax.set_xticklabels(x_labs)
        ax.set_ylim(-0.05, 1.15)
        ax.set_ylabel("Norm. probability")
        ok = "OK" if (res["shape_ok"] and res["fft_ok"]) else "FAIL"
        ax.set_title(f"|{N1},{N2}>  N={N}  err={res['max_err']:.4f}  {ok}",
                     fontsize=11, fontweight="bold", color=color)
        ax.legend(fontsize=8, loc="upper right")
    for ax in axes[n:]:
        ax.set_visible(False)
    fig.suptitle("Validation: TF backend vs Fock backend\n"
                 "Adaptive NOON-state forward model -- Afek init",
                 fontsize=13, fontweight="bold")
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"  Saved -> {save_path}")
    if show: plt.show()
    plt.close(fig)


def plot_training_history(history, N, save_path=None, show=False):
    steps    = [r["step"]     for r in history]
    CFI_norm = [r["CFI_norm"] for r in history]
    fig, ax  = plt.subplots(figsize=(7, 4))
    ax.plot(steps, CFI_norm, lw=2, color="#2E86AB", label="mean CFI/N^2")
    ax.axhline(1.0, ls="--", color="red", lw=1.2, label="Heisenberg limit")
    ax.set_xlabel("Training step")
    ax.set_ylabel("mean CFI / N^2")
    ax.set_title(f"CFI Training Progress  (N={N})", fontweight="bold")
    ax.legend(); ax.set_ylim(bottom=0)
    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150)
        print(f"  Saved -> {save_path}")
    if show: plt.show()
    plt.close(fig)


# =============================================================================
# 8.  TRAINING  (Module 2 — Stage 4)
# =============================================================================

# Afek baseline CFI values (400-point ground truth).
# N=2 measured; N=3,4,5 populated by measure_afek_baselines().
AFEK_CFI_BASELINE = {
    (1, 1): 0.3125,
    (2, 0): 0.9593,
    (2, 1): None,
    (3, 0): None,
    (3, 1): None,
    (2, 2): None,
    (3, 2): None,
}


def measure_afek_baselines(N_list=None, cutoff=None, verbose=True):
    """
    Measure ground-truth peak CFI for all Afek initialisations via
    400-point numpy.gradient scan and populate AFEK_CFI_BASELINE in-place.
    Call once before training any N > 2.
    """
    global AFEK_CFI_BASELINE
    if N_list is None:
        N_list = [2, 3, 4, 5]
    results = {}
    for N in N_list:
        co = cutoff or max(3 * N + 4, 12)
        p = CircuitParams.from_afek(N=N)
        for N1, N2 in PATTERNS[N]:
            prob, _ = tf_fringe_scan(p, N1, N2, PHI_SCAN,
                                     cutoff=co, verbose=False)
            CFI = peak_CFI(prob, PHI_SCAN)
            AFEK_CFI_BASELINE[(N1, N2)] = CFI
            results[(N1, N2)] = CFI
            if verbose:
                print(f"  |{N1},{N2}>  N={N}  Afek CFI = {CFI:.4f}")
    return results


def training_step(
    params: CircuitParams,
    optimizer: tf.keras.optimizers.Optimizer,
    cutoff: int,
    K: int = 8,
) -> tuple:
    """
    One Adam step.

    Loss = -sum_patterns( CFI_k / CFI_afek_k )

    Normalising by the Afek baseline CFI ensures each pattern contributes
    equally to the loss regardless of its absolute scale.  This prevents
    the optimizer from sacrificing one pattern to improve another.

    Returns (total_CFI_float, loss_float).
    """
    with tf.GradientTape() as tape:
        loss = tf.constant(0.0, dtype=tf.float64)
        for N1, N2 in PATTERNS.get(params.N, []):
            CFI_k, _, _ = compute_CFI_differentiable(params, N1, N2, cutoff, K=K)
            baseline    = AFEK_CFI_BASELINE.get((N1, N2), 1.0)
            loss        = loss - CFI_k / tf.constant(baseline, tf.float64)
    grads = tape.gradient(loss, params.trainable_variables)
    optimizer.apply_gradients(zip(grads, params.trainable_variables))
    total_CFI = float(-loss)   # sum of normalised CFIs
    return total_CFI, float(loss)


def save_checkpoint(params, path, extra=None):
    data = {v.name: v.numpy() for v in params.trainable_variables}
    if extra: data.update(extra)
    np.savez(path, **data)


def load_checkpoint(params, path):
    data = dict(np.load(path))
    for v in params.trainable_variables:
        if v.name in data:
            v.assign(data[v.name])
    return data


def run_training(
    params: CircuitParams,
    n_steps:          int   = 200,
    lr:               float = 0.01,
    cutoff:           int   = None,
    log_every:        int   = 10,
    checkpoint_every: int   = 50,
    checkpoint_path:  str   = "checkpoint_params.npz",
    K:                int   = 8,
) -> list:
    """
    Full Adam training loop maximising total CFI.

    Returns history: list of dicts with keys
      step, total_CFI, CFI_norm, loss
      + per-pattern CFI at log steps.
    """
    if cutoff is None:
        cutoff = max(3 * params.N + 4, 12)

    hl         = float(params.N) ** 2
    n_patterns = len(PATTERNS.get(params.N, []))
    opt        = tf.keras.optimizers.Adam(learning_rate=lr)
    history    = []

    # ── initial CFI (full detail) ─────────────────────────────────────────────
    CFI_init, det_init = compute_total_CFI_detailed(params, cutoff, K=K)
    print(f"\n{'='*60}")
    print(f"  TRAINING  N={params.N}  cutoff={cutoff}  lr={lr}  steps={n_steps}")
    print(f"{'='*60}")
    print(f"  Initial  total_CFI = {CFI_init:.4f}  "
          f"mean_CFI/N^2 = {CFI_init/(hl*n_patterns):.4f}")
    for pat, d in det_init.items():
        print(f"    {pat}  CFI={d['CFI']:.4f}  CFI/N^2={d['CFI_hl']:.4f}  "
              f"B={d['B']:.4e}  A={d['A']:.4e}")
    print(f"{'─'*60}")

    t0 = time.time()
    for step in range(n_steps):
        CFI, loss = training_step(params, opt, cutoff, K=K)
        CFI_norm  = CFI / (hl * n_patterns)
        rec = dict(step=step+1, total_CFI=CFI, CFI_norm=CFI_norm, loss=loss)
        history.append(rec)

        if (step+1) % log_every == 0:
            print(f"  Step {step+1:4d}/{n_steps}  "
                  f"CFI={CFI:.4f}  mean/N^2={CFI_norm:.4f}  "
                  f"({time.time()-t0:.1f}s)")

        if (step+1) % checkpoint_every == 0:
            save_checkpoint(params, checkpoint_path, extra={"step": step+1})
            print(f"  checkpoint -> {checkpoint_path}")

    total_t = time.time() - t0
    CFI_final, det_final = compute_total_CFI_detailed(params, cutoff, K=K)
    print(f"{'─'*60}")
    print(f"  Final    total_CFI = {CFI_final:.4f}  "
          f"mean_CFI/N^2 = {CFI_final/(hl*n_patterns):.4f}  ({total_t:.1f}s total)")
    for pat, d in det_final.items():
        print(f"    {pat}  CFI={d['CFI']:.4f}  CFI/N^2={d['CFI_hl']:.4f}  "
              f"B={d['B']:.4e}  A={d['A']:.4e}")
    print(f"{'='*60}\n")
    return history


# =============================================================================
# 9.  STAGE RUNNERS
# =============================================================================

def run_stage0(N_list):
    print("\n" + "="*60)
    print("  STAGE 0  --  Parameter Summaries")
    print("="*60)
    for N in N_list:
        CircuitParams.from_afek(N=N).print_summary()


def run_stage1(N=2):
    print("\n" + "="*60)
    print("  STAGE 1  --  Gradient Probe")
    print("="*60)
    params = CircuitParams.from_afek(N=N)
    grads  = probe_gradients(params, cutoff=max(3*N+4, 12))
    ok     = all(g is not None for g in grads.values())
    if ok:
        print("  All gradients non-None -- autodiff chain intact.\n")
    return ok


def run_stage2(N1, N2, save=True, show=False):
    print("\n" + "="*60)
    print(f"  STAGE 2  --  Single Fringe  |{N1},{N2}>  N={N1+N2}")
    print("="*60)
    N      = N1 + N2
    params = CircuitParams.from_afek(N=N)
    t0 = time.time()
    prob_tf, _ = tf_fringe_scan(params, N1, N2, PHI_SCAN, verbose=True)
    print(f"  TF done ({time.time()-t0:.1f}s)")
    t0 = time.time()
    prob_fk, _ = fock_fringe_scan(params, N1, N2, PHI_SCAN)
    print(f"  Fock done ({time.time()-t0:.1f}s)")
    print(f"  Max norm err : {np.max(np.abs(normalise(prob_tf)-normalise(prob_fk))):.2e}")
    print(f"  CFI_peak/N^2 : {peak_CFI(prob_tf, PHI_SCAN)/N**2:.4f}")
    if save or show:
        plot_single_fringe(N1, N2, prob_tf, prob_fk,
                           save_path=f"single_fringe_N{N}.png" if save else None,
                           show=show)


def run_stage3(N_list, save=True, show=False):
    print("\n" + "="*60)
    print("  STAGE 3  --  Full Validation")
    print("="*60)
    results = []
    for N in N_list:
        for N1, N2 in PATTERNS[N]:
            results.append(validate_pattern(N1, N2))
    n_pass = sum(r["shape_ok"] and r["fft_ok"] and r["CFI_ok"] for r in results)
    print(f"\n  {n_pass}/{len(results)} patterns fully passed.")
    if save or show:
        plot_validation_grid(results,
                             save_path="validation_plot.png" if save else None,
                             show=show)
    return results


def run_stage4(
    N:               int   = 2,
    n_steps:         int   = 200,
    lr:              float = 0.01,
    cutoff:          int   = None,
    log_every:       int   = 10,
    checkpoint_every:int   = 50,
    save:            bool  = True,
    show:            bool  = False,
    K:               int   = 8,
) -> tuple:
    """
    Module 2 entry point.  Initialises at Afek params, trains via CFI loss.

    Returns (params, history).

    Example
    -------
    params, history = run_stage4(N=2, n_steps=200, lr=0.01)
    """
    if cutoff is None:
        cutoff = max(3*N+4, 12)

    params = CircuitParams.from_afek(N)
    print(f"\n{'='*60}")
    print(f"  STAGE 4  --  CFI Maximisation  (N={N})")
    print(f"{'='*60}")
    params.print_summary()

    history = run_training(
        params, n_steps=n_steps, lr=lr, cutoff=cutoff,
        log_every=log_every, checkpoint_every=checkpoint_every,
        checkpoint_path=f"checkpoint_N{N}.npz", K=K,
    )

    if save or show:
        plot_training_history(
            history, N,
            save_path=f"training_history_N{N}.png" if save else None,
            show=show,
        )

    ckpt = f"final_params_N{N}.npz"
    save_checkpoint(params, ckpt)
    print(f"  Final params saved -> {ckpt}")
    return params, history


def run_all_N(N_list=None, n_steps=100, lr=0.02, save=True):
    """
    Run Stage 3 validation + Stage 4 optimisation for each N in N_list.
    Returns all_results dict keyed by N.

    Example
    -------
    all_results = run_all_N([2, 3, 4, 5], n_steps=100, lr=0.02)
    print_all_results(all_results)
    """
    if N_list is None:
        N_list = [2, 3, 4, 5]

    print("\n" + "="*60)
    print("  MEASURING AFEK BASELINES FOR ALL N")
    print("="*60)
    measure_afek_baselines(N_list=N_list)

    all_results = {}
    for N in N_list:
        print(f"\n{'='*60}")
        print(f"  N = {N}  --  STAGE 3 VALIDATION")
        print("="*60)
        val_results = run_stage3([N], save=save, show=False)

        print(f"\n{'='*60}")
        print(f"  N = {N}  --  STAGE 4 OPTIMISATION")
        print("="*60)
        params_opt, history = run_stage4(N=N, n_steps=n_steps,
                                         lr=lr, save=save, show=False)

        # Ground-truth comparison
        cutoff = max(3 * N + 4, 12)
        comparison = {}
        for N1, N2 in PATTERNS[N]:
            p0 = CircuitParams.from_afek(N=N)
            prob0, _ = tf_fringe_scan(p0, N1, N2, PHI_SCAN,
                                      cutoff=cutoff, verbose=False)
            prob1, _ = tf_fringe_scan(params_opt, N1, N2, PHI_SCAN,
                                      cutoff=cutoff, verbose=False)
            CFI0_raw  = peak_CFI(prob0, PHI_SCAN)
            CFI1_raw  = peak_CFI(prob1, PHI_SCAN)
            CFI0_norm = peak_CFI(normalise(prob0), PHI_SCAN)
            CFI1_norm = peak_CFI(normalise(prob1), PHI_SCAN)
            rate0     = float(prob0.max())
            rate1     = float(prob1.max())
            comparison[f"|{N1},{N2}>"] = {
                "CFI_raw_afek":  CFI0_raw,
                "CFI_raw_opt":   CFI1_raw,
                "CFI_raw_pct":   (CFI1_raw  - CFI0_raw)  / CFI0_raw  * 100,
                "CFI_norm_afek": CFI0_norm,
                "CFI_norm_opt":  CFI1_norm,
                "CFI_norm_pct":  (CFI1_norm - CFI0_norm) / CFI0_norm * 100,
                "rate_afek":     rate0,
                "rate_opt":      rate1,
                "rate_pct":      (rate1 - rate0) / rate0 * 100,
            }

        all_results[N] = {
            "validation": val_results,
            "params_opt": params_opt,
            "history":    history,
            "comparison": comparison,
        }

    print_all_results(all_results)
    return all_results


def print_all_results(all_results):
    """Print a clean summary table of all N results."""
    print("\n" + "="*70)
    print("  FULL RESULTS SUMMARY")
    print("="*70)
    for N, res in all_results.items():
        hl = N ** 2
        print(f"\n  N = {N}  (Heisenberg limit N^2 = {hl})")
        print(f"  {'Pattern':<10} {'CFI_raw Afek':>13} {'CFI_raw Opt':>13}"
              f" {'Change':>8} {'Norm Afek':>10} {'Norm Opt':>10}"
              f" {'Change':>8} {'Rate +%':>8}")
        print("  " + "-"*85)
        for pat, d in res["comparison"].items():
            print(f"  {pat:<10} {d['CFI_raw_afek']:>13.4f}"
                  f" {d['CFI_raw_opt']:>13.4f}"
                  f" {d['CFI_raw_pct']:>+7.1f}%"
                  f" {d['CFI_norm_afek']:>10.3f}"
                  f" {d['CFI_norm_opt']:>10.3f}"
                  f" {d['CFI_norm_pct']:>+7.1f}%"
                  f" {d['rate_pct']:>+7.1f}%")
    print("\n" + "="*70)


# 10.  CLI
# =============================================================================

def main():
    parser = argparse.ArgumentParser(description="Adaptive NOON-state model")
    parser.add_argument("--N",      type=int, nargs="+", default=[2],
                        choices=[2,3,4,5])
    parser.add_argument("--stage",  type=int, default=None,
                        choices=[0,1,2,3,4])
    parser.add_argument("--no-plots", action="store_true")
    parser.add_argument("--show",   action="store_true")
    parser.add_argument("--steps",  type=int,   default=200)
    parser.add_argument("--lr",     type=float, default=0.01)
    parser.add_argument("--K",      type=int,   default=16,
                        help="Number of phi samples for CFI estimator (default 16)")
    args, _ = parser.parse_known_args()
    save    = not args.no_plots
    stages  = [args.stage] if args.stage is not None else [0,1,2,3]
    for s in stages:
        if   s == 0: run_stage0(args.N)
        elif s == 1: run_stage1(N=min(args.N))
        elif s == 2:
            N1, N2 = PATTERNS[args.N[0]][0]
            run_stage2(N1, N2, save=save, show=args.show)
        elif s == 3: run_stage3(args.N, save=save, show=args.show)
        elif s == 4: run_stage4(N=min(args.N), n_steps=args.steps,
                                lr=args.lr, save=save, show=args.show, K=args.K)


if __name__ == "__main__":
    main()

In [ ]:
baselines = measure_afek_baselines(N_list=[2, 3, 4, 5])

In [ ]:
all_results = run_all_N([2, 3, 4, 5], n_steps=100, lr=0.02)

In [ ]:
"""
figures_nature.py ------------------------Author: Simanshu Kumar
=================
"Quantum-Enhanced Single-phase Estimation with Adaptive NOON States"

Where all_results is the dict returned by run_all_N([2,3,4,5]).
You can also generate individual figures — see functions below.

Figures produced
----------------
  Fig 1  — Circuit schematic  (circuit_diagram.pdf) --------------------->> However, We use Latex tikz and quantikz libraries for original document circuit.
  Fig 2  — Fringe gallery: Afek vs optimised, all N  (fringe_gallery.pdf)
  Fig 3  — Scaling summary: CFI and rate vs N  (scaling_summary.pdf)
  Fig 4  — Pareto trade-off: fringe quality vs rate  (pareto_tradeoff.pdf)
  Fig 5  — Training convergence CFIs, all N  (training_convergence.pdf)
  Fig 6  — Parameter drift from Afek init  (parameter_drift.pdf)
  Fig 7  — Photon-number distributions: Afek vs optimised  (photon_distributions.pdf)
"""

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import matplotlib.patheffects as pe
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch, Arc
from matplotlib.colors import LinearSegmentedColormap
import warnings
warnings.filterwarnings("ignore")

# ── Enhanced Physics Figure style ──────────────────────────────────────────────────────
# Nature Physics (e.g.,): 89 mm single column, 183 mm double column
# Font: Helvetica/Arial, 7–8 pt body, 6 pt labels
# No top/right spines, minimal ticks

SINGLE_COL = 3.50   # inches (89 mm)
DOUBLE_COL = 7.20   # inches (183 mm)
GOLDEN     = 1.618

NATURE_RC = {
    # Font
    "font.family":        "sans-serif",
    "font.sans-serif":    ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size":          7,
    "axes.titlesize":     8,
    "axes.labelsize":     7,
    "xtick.labelsize":    6,
    "ytick.labelsize":    6,
    "legend.fontsize":    6,
    "figure.titlesize":   9,
    # Lines
    "axes.linewidth":     0.6,
    "grid.linewidth":     0.4,
    "lines.linewidth":    1.2,
    "patch.linewidth":    0.6,
    "xtick.major.width":  0.6,
    "ytick.major.width":  0.6,
    "xtick.minor.width":  0.4,
    "ytick.minor.width":  0.4,
    "xtick.major.size":   3,
    "ytick.major.size":   3,
    "xtick.minor.size":   1.5,
    "ytick.minor.size":   1.5,
    # Spines / grid
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "axes.grid":          False,
    # Padding
    "axes.labelpad":      3,
    "xtick.major.pad":    2,
    "ytick.major.pad":    2,
    # Save
    "savefig.dpi":        300,
    "savefig.bbox":       "tight",
    "savefig.pad_inches": 0.02,
    "pdf.fonttype":       42,   # editable text in Illustrator
    "ps.fonttype":        42,
}

mpl.rcParams.update(NATURE_RC)

# ── Colour palette ────────────────────────────────────────────────────────────
# Nature-friendly, colour-blind safe
PAL = {
    "N2":      "#1f77b4",   # blue
    "N3":      "#ff7f0e",   # orange
    "N4":      "#2ca02c",   # green
    "N5":      "#d62728",   # red
    "afek":    "#888888",   # grey
    "opt":     "#000000",   # black
    "improve": "#2ca02c",
    "degrade": "#d62728",
    "neutral": "#1f77b4",
}

N_COLORS = {2: PAL["N2"], 3: PAL["N3"], 4: PAL["N4"], 5: PAL["N5"]}

PHI = np.linspace(0, 2*np.pi, 400)
PHI_TICKS     = [0, np.pi/2, np.pi, 3*np.pi/2, 2*np.pi]
PHI_TICKLABELS = ["0", r"$\pi/2$", r"$\pi$", r"$3\pi/2$", r"$2\pi$"]


def despine(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


def panel_label(ax, label, x=-0.18, y=1.06):
    ax.text(x, y, label, transform=ax.transAxes,
            fontsize=9, fontweight="bold", va="top", ha="left")


# =============================================================================
# FIGURE 1 — Circuit Diagram
# =============================================================================

def fig_circuit_diagram(save_path="fig1_circuit.pdf"):
    """
    Clean schematic of the adaptive NOON-state MZI circuit.
    Shows state preparation, beamsplitters, phase encoding, detection.
    """
    fig, ax = plt.subplots(figsize=(DOUBLE_COL, 2.0))
    ax.set_xlim(0, 10)
    ax.set_ylim(-0.8, 1.8)
    ax.axis("off")

    lw = 1.2
    c_line = "#333333"
    c_bs   = PAL["N2"]
    c_r    = PAL["N4"]
    c_det  = "#555555"
    c_coh  = "#ff7f0e"
    c_sq   = "#9467bd"

    # ── Mode lines ────────────────────────────────────────────────────────────
    ax.axhline(1.2, xmin=0.05, xmax=0.95, color=c_line, lw=lw, zorder=1)
    ax.axhline(0.0, xmin=0.05, xmax=0.95, color=c_line, lw=lw, zorder=1)

    # Mode labels
    ax.text(-0.1, 1.2, r"mode 0", ha="right", va="center", fontsize=6.5)
    ax.text(-0.1, 0.0, r"mode 1", ha="right", va="center", fontsize=6.5)

    def arrow(x0, y0, x1, y1, color="#333333"):
        ax.annotate("", xy=(x1, y1), xytext=(x0, y0),
                    arrowprops=dict(arrowstyle="-|>", color=color,
                                   lw=0.8, mutation_scale=8),
                    zorder=5)

    def box(cx, cy, w, h, color, label, sublabel=None, fontsize=6):
        rect = FancyBboxPatch((cx-w/2, cy-h/2), w, h,
                               boxstyle="round,pad=0.04",
                               facecolor=color, edgecolor="white",
                               linewidth=0.8, alpha=0.88, zorder=3)
        ax.add_patch(rect)
        ax.text(cx, cy + (0.06 if sublabel else 0), label,
                ha="center", va="center", fontsize=fontsize,
                color="white", fontweight="bold", zorder=4)
        if sublabel:
            ax.text(cx, cy - 0.13, sublabel, ha="center", va="center",
                    fontsize=5, color="white", zorder=4)

    def bs_symbol(cx, lw_bs=1.8):
        """Beamsplitter — diagonal cross between modes."""
        ax.plot([cx-0.18, cx+0.18], [0.0, 1.2],
                color=c_bs, lw=lw_bs, zorder=4)
        ax.plot([cx-0.18, cx+0.18], [1.2, 0.0],
                color=c_bs, lw=lw_bs, zorder=4)
        rect = FancyBboxPatch((cx-0.22, -0.22), 0.44, 1.64,
                               boxstyle="round,pad=0.05",
                               facecolor="none", edgecolor=c_bs,
                               linewidth=1.0, zorder=3)
        ax.add_patch(rect)

    # ── Input states ──────────────────────────────────────────────────────────
    box(0.55, 1.2, 0.55, 0.32, c_coh,
        r"$|\alpha\rangle$", r"coherent", fontsize=6.5)
    box(0.55, 0.0, 0.55, 0.32, c_sq,
        r"$|r\rangle$", r"squeezed", fontsize=6.5)

    # ── Input phase rotations ─────────────────────────────────────────────────
    box(2.0, 1.2, 0.45, 0.28, c_r, r"$R(d_c)$", fontsize=6)
    box(2.0, 0.0, 0.45, 0.28, c_r, r"$R(d_s)$", fontsize=6)

    ax.text(2.0, 1.62, "input phases", ha="center", va="bottom",
            fontsize=5.5, color=c_r, style="italic")

    # ── BS1 ───────────────────────────────────────────────────────────────────
    bs_symbol(3.4)
    ax.text(3.4, -0.52, r"$\mathrm{BS}(\theta_1,\varphi_1)$",
            ha="center", va="top", fontsize=6, color=c_bs)

    # ── Phase encoding ────────────────────────────────────────────────────────
    box(5.0, 1.2, 0.55, 0.32, "#c44e52",
        r"$R(\varphi_\mathrm{est})$", fontsize=5.8)
    ax.text(5.0, 1.62, "unknown phase", ha="center", va="bottom",
            fontsize=5.5, color="#c44e52", style="italic")

    # Dashed line connecting phase to mode 0 only
    ax.plot([5.0, 5.0], [1.36, 1.56], color="#c44e52",
            lw=0.6, linestyle=":", zorder=2)

    # ── BS2 ───────────────────────────────────────────────────────────────────
    bs_symbol(6.6)
    ax.text(6.6, -0.52, r"$\mathrm{BS}(\theta_2,\varphi_2)$",
            ha="center", va="top", fontsize=6, color=c_bs)
    ax.text(6.6, 1.62, "measurement\nbasis", ha="center", va="bottom",
            fontsize=5.5, color=c_bs, style="italic")

    # ── Detectors ─────────────────────────────────────────────────────────────
    for y, label in [(1.2, r"$N_1$"), (0.0, r"$N_2$")]:
        det = mpatches.Wedge((8.8, y), 0.22, -70, 70,
                              color=c_det, zorder=3)
        ax.add_patch(det)
        ax.text(9.25, y, label, ha="left", va="center",
                fontsize=7, fontweight="bold", color=c_det)

    # Coincidence bracket
    ax.annotate("", xy=(9.55, 0.0), xytext=(9.55, 1.2),
                arrowprops=dict(arrowstyle="|-|", color=c_det,
                                lw=0.8, mutation_scale=5))
    ax.text(9.85, 0.6, r"$P(N_1,N_2;\varphi)$",
            ha="left", va="center", fontsize=6, color=c_det,
            rotation=90)

    # ── Trainable label ───────────────────────────────────────────────────────
    ax.text(4.3, -0.72,
            r"Trainable: $r,\,\log\gamma,\,d_c,\,d_s,\,"
            r"\theta_1,\varphi_1,\theta_2,\varphi_2$",
            ha="center", va="top", fontsize=6.0, color="#555555",
            style="italic")

    # ── Legend ────────────────────────────────────────────────────────────────
    handles = [
        mpatches.Patch(color=c_coh, label="coherent state"),
        mpatches.Patch(color=c_sq,  label="squeezed vacuum"),
        mpatches.Patch(color=c_r,   label="phase rotation"),
        mpatches.Patch(color=c_bs,  label="beamsplitter"),
        mpatches.Patch(color="#c44e52", label="phase encoding"),
    ]
    ax.legend(handles=handles, loc="upper center",
              bbox_to_anchor=(0.5, 1.18), ncol=5,
              frameon=False, fontsize=5.5,
              columnspacing=0.8, handlelength=1.0)

    fig.tight_layout(pad=0.3)
    fig.savefig(save_path)
    print(f"  Saved → {save_path}")
    return fig


# =============================================================================
# FIGURE 2 — Fringe Gallery: Afek vs Optimised
# =============================================================================

def fig_fringe_gallery(all_results, save_path="fig2_fringe_gallery.pdf"):
    """
    2-column grid: left = Afek fringe, right = optimised fringe.
    One row per (N, pattern). Normalised so both fit on [0,1].
    Inset: classical CFI ie., CFI.
    """
    # Collect all patterns in order
    all_patterns = []
    for N in [2, 3, 4, 5]:
        if N not in all_results:
            continue
        for pat_key, d in all_results[N]["comparison"].items():
            all_patterns.append((N, pat_key, d))

    nrows = len(all_patterns)
    fig = plt.figure(figsize=(DOUBLE_COL, nrows * 1.05 + 0.3))
    gs  = GridSpec(nrows, 2, figure=fig, hspace=0.55, wspace=0.35,
                   left=0.10, right=0.97, top=0.94, bottom=0.06)

    col_labels = ["Afek initialisation", "Optimised parameters"]
    for col, cl in enumerate(col_labels):
        fig.text(0.28 + col*0.49, 0.965, cl,
                 ha="center", va="bottom", fontsize=7.5,
                 fontweight="bold", color="#333333")

    for row, (N, pat_key, d) in enumerate(all_patterns):
        color = N_COLORS[N]

        for col, (prob_key, CFI_key, label_sfx) in enumerate([
            ("prob_afek", "CFI_norm_afek", "Afek"),
            ("prob_opt",  "CFI_norm_opt",  "Opt"),
        ]):
            ax = fig.add_subplot(gs[row, col])

            # Retrieve fringe data — stored in all_results
            res = all_results[N]
            prob = res.get(prob_key, {}).get(pat_key)
            if prob is None:
                ax.text(0.5, 0.5, "data\nnot stored",
                        ha="center", va="center",
                        transform=ax.transAxes, fontsize=5, color="grey")
                despine(ax)
                continue

            # Normalise for display
            prob_n = (prob - prob.min()) / (prob.max() - prob.min() + 1e-30)

            ax.fill_between(PHI, 0, prob_n, alpha=0.15, color=color)
            ax.plot(PHI, prob_n, color=color, lw=1.1)

            # CFI curve inset via twin axis
            CFI_curve = classical_CFI_from_fringe(prob, PHI)
            ax2 = ax.twinx()
            ax2.plot(PHI, CFI_curve, color="#888888", lw=0.7,
                     linestyle="--", alpha=0.8)
            ax2.set_ylim(bottom=0)
            ax2.tick_params(axis="y", labelsize=4.5,
                            colors="#888888", length=2, width=0.4)
            ax2.spines["top"].set_visible(False)
            ax2.spines["right"].set_color("#aaaaaa")
            ax2.spines["right"].set_linewidth(0.4)
            if col == 1:
                ax2.set_ylabel("CFI", fontsize=5, color="#888888",
                               labelpad=2)
            else:
                ax2.set_yticklabels([])

            ax.set_xlim(0, 2*np.pi)
            ax.set_ylim(-0.05, 1.15)
            ax.set_xticks(PHI_TICKS[::2])
            ax.set_xticklabels(["0", r"$\pi$", r"$2\pi$"])
            ax.set_yticks([0, 0.5, 1])

            CFI_val = d[CFI_key] if not (isinstance(d[CFI_key], float)
                                          and d[CFI_key] > 1000) else None
            CFI_str = f"{CFI_val:.2f}" if CFI_val else "n/a"
            rate_key = "rate_afek" if col == 0 else "rate_opt"
            rate = d[rate_key]

            ax.set_title(
                f"$F_{{\\rm peak}}={CFI_str}$  "
                f"$P_{{\\rm max}}={rate:.3f}$",
                fontsize=5.5, pad=2, color="#333333"
            )
            despine(ax)

            if row == nrows - 1:
                ax.set_xlabel(r"Phase $\varphi$ (rad)", fontsize=6)
            else:
                ax.set_xticklabels([])

            if col == 0:
                ax.set_ylabel("Norm. $P$", fontsize=5.5)
                # Pattern + N label on left
                N1N2 = pat_key.replace("|", "").replace(">", "")
                ax.text(-0.32, 0.5, f"$N={N}$\n${pat_key}$",
                        transform=ax.transAxes, fontsize=6,
                        va="center", ha="right",
                        color=color, fontweight="bold",
                        multialignment="center")

    fig.savefig(save_path)
    print(f"  Saved → {save_path}")
    return fig


# =============================================================================
# FIGURE 3 — Scaling Summary (the key result figure)
# =============================================================================

def fig_scaling_summary(all_results, save_path="fig3_scaling_summary.pdf"):
    """
    Two-panel figure:
      Left:  Raw CFI improvement (%) vs N, per pattern
      Right: Post-selection rate improvement (%) vs N, per pattern
    Log scale on y-axis. This is the central result figure.
    """
    # Collect data
    records = []
    for N in [2, 3, 4, 5]:
        if N not in all_results:
            continue
        for pat, d in all_results[N]["comparison"].items():
            records.append({
                "N": N, "pat": pat,
                "CFI_pct":  d["CFI_raw_pct"],
                "rate_pct": d["rate_pct"],
                "CFI_afek": d["CFI_raw_afek"],
                "CFI_opt":  d["CFI_raw_opt"],
                "norm_afek": d["CFI_norm_afek"],
                "norm_opt":  d["CFI_norm_opt"],
            })

    fig, axes = plt.subplots(1, 2, figsize=(DOUBLE_COL, DOUBLE_COL / 2.6))

    markers = {2: "o", 3: "s", 4: "^", 5: "D"}
    pat_styles = {}  # unique style per pattern label

    for ax_idx, (ax, key, ylabel, title) in enumerate(zip(
        axes,
        ["CFI_pct",  "rate_pct"],
        ["Raw CFI improvement (%)", "Post-selection rate improvement (%)"], 
        ["a", "b"],
    )):
        all_vals = [r[key] for r in records]
        has_neg  = any(v < 0 for v in all_vals)

        for rec in records:
            N    = rec["N"]
            val  = rec[key]
            pat  = rec["pat"]
            col  = N_COLORS[N]
            mk   = markers[N]
            ec   = "white" if val >= 0 else col
            fc   = col     if val >= 0 else "white"

            ax.scatter(N + np.random.uniform(-0.07, 0.07), val,
                       color=fc, edgecolors=ec, marker=mk,
                       s=38, linewidths=0.8, zorder=4,
                       label=f"N={N}" if pat == list(
                           all_results[N]["comparison"].keys())[0] else "")

            ax.text(N + 0.12, val, pat.replace("|","").replace(">",""),
                    fontsize=5, va="center", color=col, alpha=0.85)

        ax.axhline(0, color="#aaaaaa", lw=0.6, zorder=1)
        ax.set_xlabel("Total photon number $N$", fontsize=7)
        ax.set_ylabel(ylabel, fontsize=7)
        ax.set_xticks([2, 3, 4, 5])
        ax.set_xticklabels(["$N=2$","$N=3$","$N=4$","$N=5$"])

        # Log scale for positive values
        ymax = max(all_vals) * 1.25
        ymin = min(min(all_vals) * 1.3, -50) if has_neg else -50
        ax.set_ylim(ymin, ymax)

        # Heisenberg limit reference line (CFI panel only)
        if ax_idx == 0:
            ax.set_title("Raw CFI change", fontsize=7.5, pad=4) 
            ax.fill_between([1.6, 5.4], [0, 0], [ymax, ymax],
                             alpha=0.04, color=PAL["improve"])
            ax.fill_between([1.6, 5.4], [ymin, ymin], [0, 0],
                             alpha=0.04, color=PAL["degrade"])
            ax.text(5.35, ymax*0.92, "improvement",
                    fontsize=5, color=PAL["improve"], ha="right", va="top")
            ax.text(5.35, ymin*0.92, "degradation",
                    fontsize=5, color=PAL["degrade"], ha="right", va="bottom")
        else:
            ax.set_title("Post-selection rate change", fontsize=7.5, pad=4)

        ax.set_xlim(1.6, 5.8)
        despine(ax)
        panel_label(ax, title)

    # Legend for N
    handles = [mlines.Line2D([], [], color=N_COLORS[N],
                              marker=markers[N], linestyle="None",
                              markersize=5, label=f"$N={N}$",
                              markeredgecolor="white", markeredgewidth=0.5)
               for N in [2,3,4,5]]
    axes[1].legend(handles=handles, loc="upper left",
                   frameon=False, fontsize=6, ncol=2,
                   handletextpad=0.3, columnspacing=0.5)

    fig.tight_layout(pad=0.5)
    fig.savefig(save_path)
    print(f"  Saved → {save_path}")
    return fig


# =============================================================================
# FIGURE 4 — Pareto Trade-off: Fringe Quality vs Count Rate
# =============================================================================

def fig_pareto_tradeoff(all_results, save_path="fig4_pareto.pdf"):
    """
    Scatter: normalised CFI (fringe quality) vs post-selection rate.
    Each point is one (N, pattern, init) pair.
    Afek points connected to optimised points by arrows.
    Shows the trade-off structure visually.
    """
    fig, axes = plt.subplots(1, 2, figsize=(DOUBLE_COL, DOUBLE_COL/2.2))

    for ax_idx, (ax, N_subset, panel) in enumerate(zip(
        axes,
        [[2, 3], [4, 5]],
        ["a", "b"],
    )):
        for N in N_subset:
            if N not in all_results:
                continue
            col = N_COLORS[N]
            for pat, d in all_results[N]["comparison"].items():
                # Skip pathological norm CFI
                fn_afek = d["CFI_norm_afek"]
                fn_opt  = d["CFI_norm_opt"]
                hl      = N**2
                if fn_afek > 10 * hl:
                    fn_afek = None

                rate_afek = d["rate_afek"]
                rate_opt  = d["rate_opt"]

                if fn_afek is not None:
                    ax.scatter(rate_afek, fn_afek/hl,
                               marker="o", s=45, color=col,
                               edgecolors="white", linewidths=0.8,
                               zorder=4, alpha=0.9)
                    ax.scatter(rate_opt, fn_opt/hl,
                               marker="*", s=80, color=col,
                               edgecolors="white", linewidths=0.8,
                               zorder=4)
                    # Arrow from Afek to optimised
                    ax.annotate("",
                        xy=(rate_opt, fn_opt/hl),
                        xytext=(rate_afek, fn_afek/hl),
                        arrowprops=dict(
                            arrowstyle="-|>",
                            color=col, lw=0.8,
                            mutation_scale=7,
                            connectionstyle="arc3,rad=0.1",
                        ), zorder=3)

                    # Label
                    pat_short = pat.replace("|","").replace(">","")
                    ax.text(rate_opt*1.05, fn_opt/hl,
                            f"${pat_short}$",
                            fontsize=5, color=col, va="center")

        ax.axhline(1.0, color="#aaaaaa", lw=0.7, linestyle="--", zorder=1)
        ax.text(ax.get_xlim()[1] if ax.get_xlim()[1] < 1 else 0.5,
                1.02, "Heisenberg limit", fontsize=5,
                color="#aaaaaa", ha="right")

        ax.set_xlabel("Post-selection rate $P_{\\rm max}$", fontsize=7)
        ax.set_ylabel(r"Fringe quality $F^{\rm norm}/N^2$", fontsize=7)
        Nstr = ", ".join(f"$N={n}$" for n in N_subset if n in all_results)
        ax.set_title(Nstr, fontsize=7.5, pad=4)
        despine(ax)
        panel_label(ax, panel)

        # Legend
        h = [mlines.Line2D([],[],marker="o",linestyle="None",
                           color="grey",markersize=5,label="Afek init",
                           markeredgecolor="white"),
             mlines.Line2D([],[],marker="*",linestyle="None",
                           color="grey",markersize=6,label="Optimised",
                           markeredgecolor="white")]
        ax.legend(handles=h, frameon=False, fontsize=5.5,
                  handletextpad=0.3)

    fig.tight_layout(pad=0.5)
    fig.savefig(save_path)
    print(f"  Saved → {save_path}")
    return fig


# =============================================================================
# FIGURE 5 — Training Convergence
# =============================================================================

def fig_training_convergence(all_results, save_path="fig5_convergence.pdf"):
    """
    Training loss (differentiable CFI) vs step for all N on one plot.
    Shows smooth convergence and relative improvement per N.
    """
    fig, ax = plt.subplots(figsize=(SINGLE_COL * 1.5, SINGLE_COL * 1.1))

    for N in [2, 3, 4, 5]:
        if N not in all_results:
            continue
        history = all_results[N]["history"]
        if not history:
            continue
        steps = [r["step"] for r in history]
        # Normalise by initial value for fair comparison
        vals  = [r["total_CFI"] for r in history]
        v0    = vals[0] if vals[0] > 0 else 1.0
        vals_n = [v / v0 for v in vals]
        col   = N_COLORS[N]
        ax.plot(steps, vals_n, color=col, lw=1.3,
                label=f"$N={N}$", zorder=3)
        # Mark final point
        ax.scatter(steps[-1], vals_n[-1], color=col,
                   s=25, zorder=4, edgecolors="white", linewidths=0.6)

    ax.axhline(1.0, color="#cccccc", lw=0.6, linestyle="--")
    ax.set_xlabel("Training step", fontsize=7)
    ax.set_ylabel("Normalised CFI (relative to step 0)", fontsize=7)
    ax.set_title("Optimisation convergence", fontsize=7.5, pad=4)
    ax.legend(frameon=False, fontsize=6.5, ncol=2,
              handlelength=1.2, columnspacing=0.8)
    despine(ax)
    #panel_label(ax, "a", x=-0.20)

    fig.tight_layout(pad=0.5)
    fig.savefig(save_path)
    print(f"  Saved → {save_path}")
    return fig


# =============================================================================
# FIGURE 6 — Parameter Drift from Afek Init
# =============================================================================

def fig_parameter_drift(all_results, save_path="fig6_param_drift.pdf"):
    """
    Radar / bar chart showing how much each of the 8 parameters
    moved from the Afek initialisation, for each N.
    """
    param_names = [r"$r$", r"$\log\gamma$", r"$d_c$", r"$d_s$",
                   r"$\theta_1$", r"$\varphi_1$", r"$\theta_2$", r"$\varphi_2$"]
    afek_vals = {
        2: np.array([0.350, 0.000, np.pi/2, 0.000, np.pi/4, 0.000, np.pi/4, np.pi]),
        3: np.array([0.350, 0.000, np.pi/2, 0.000, np.pi/4, 0.000, np.pi/4, np.pi]),
        4: np.array([0.350, np.log(np.sqrt(3)), np.pi/2, 0.000, np.pi/4, 0.000, np.pi/4, np.pi]),
        5: np.array([0.350, np.log(9/(np.sqrt(10)+1)), np.pi/2, 0.000, np.pi/4, 0.000, np.pi/4, np.pi]),
    }

    fig, axes = plt.subplots(1, 4, figsize=(DOUBLE_COL, DOUBLE_COL/3.2),
                              sharey=False)

    for ax, N in zip(axes, [2, 3, 4, 5]):
        if N not in all_results:
            ax.set_visible(False)
            continue
        params_opt = all_results[N]["params_opt"]
        opt_vals = np.array([
            float(params_opt.r),
            float(params_opt.log_gamma),
            float(params_opt.d_coh),
            float(params_opt.d_sq),
            float(params_opt.theta1),
            float(params_opt.phi1),
            float(params_opt.theta2),
            float(params_opt.phi2),
        ])
        drift = opt_vals - afek_vals[N]
        col   = N_COLORS[N]
        colors = [col if d >= 0 else PAL["degrade"] for d in drift]

        bars = ax.barh(range(8), drift, color=colors,
                       edgecolor="white", linewidth=0.5, height=0.6)
        ax.axvline(0, color="#999999", lw=0.6)
        ax.set_yticks(range(8))
        ax.set_yticklabels(param_names if N == 2 else [], fontsize=6)
        ax.set_title(f"$N={N}$", fontsize=7, color=col, pad=3)
        ax.set_xlabel(r"$\Delta$ (opt $-$ Afek)", fontsize=5.5)
        despine(ax)
        ax.spines["left"].set_visible(False)
        ax.tick_params(left=False)

    fig.suptitle("Parameter drift from Afek initialisation",
                 fontsize=7.5, y=1.02, fontweight="bold", color="#333333")
    fig.tight_layout(pad=0.4, w_pad=0.3)
    fig.savefig(save_path)
    print(f"  Saved → {save_path}")
    return fig


# =============================================================================
# FIGURE 7 — Photon-Number Distributions
# =============================================================================

def fig_photon_distributions(all_results, save_path="fig7_photon_dist.pdf"):
    """
    Marginal photon-number distributions P(n) = sum_m P(n,m)
    for Afek init vs optimised, for each N.
    Shows how the optimizer inflates the distribution.
    """
    fig, axes = plt.subplots(2, 4, figsize=(DOUBLE_COL, DOUBLE_COL/2.2))

    cutoff = 12
    n_vals = np.arange(cutoff)

    for col_idx, N in enumerate([2, 3, 4, 5]):
        if N not in all_results:
            for row in range(2):
                axes[row][col_idx].set_visible(False)
            continue

        co = max(3*N+4, 12)
        n_arr = np.arange(co)

        for row, (p_key, label, ls, alpha) in enumerate([
            ("params_afek", "Afek", "-",  0.9),
            ("params_opt",  "Opt",  "--", 0.9),
        ]):
            ax   = axes[row][col_idx]
            col  = N_COLORS[N]
            # Get stored distributions if available
            dist = all_results[N].get(p_key + "_dist")
            if dist is None:
                ax.text(0.5, 0.5, "run\nsimulation",
                        ha="center", va="center",
                        transform=ax.transAxes, fontsize=5.5)
                despine(ax)
                continue

            # Marginal P(n0) = sum over n1
            marginal = dist.sum(axis=1)
            ax.bar(n_arr, marginal, color=col, alpha=alpha*0.7,
                   edgecolor=col, linewidth=0.4, width=0.7)
            ax.axvline(N, color="#cc0000", lw=0.7, linestyle=":",
                       label=f"$N={N}$")
            ax.set_xlim(-0.5, min(co-1, 2*N+2)+0.5)

            if col_idx == 0:
                ax.set_ylabel(f"$P(n_0)$\n({label})", fontsize=5.5)
            if row == 1:
                ax.set_xlabel("$n_0$", fontsize=6)
            else:
                ax.set_xticklabels([])
            if row == 0:
                ax.set_title(f"$N={N}$", fontsize=7, color=col, pad=3)
            despine(ax)

    fig.tight_layout(pad=0.4, h_pad=0.3)
    fig.savefig(save_path)
    print(f"  Saved → {save_path}")
    return fig


# =============================================================================
# FIGURE 8 — Raw CFI absolute values: Afek vs Optimised (bar chart)
# =============================================================================

def fig_CFI_comparison_bars(all_results, save_path="fig8_CFI_bars.pdf"):
    """
    Grouped bar chart: Afek vs Optimised raw CFI for each pattern.
    Heisenberg limit shown as horizontal line per N.
    Clean, direct comparison.
    """
    fig, axes = plt.subplots(2, 2, figsize=(DOUBLE_COL, DOUBLE_COL/1.5))
    axes_flat = axes.flatten()

    panel_labels_list = ["a", "b", "c", "d"]

    for idx, N in enumerate([2, 3, 4, 5]):
        ax  = axes_flat[idx]
        col = N_COLORS[N]
        if N not in all_results:
            ax.set_visible(False)
            continue

        patterns = list(all_results[N]["comparison"].keys())
        x        = np.arange(len(patterns))
        width    = 0.35
        hl       = N**2

        afek_vals = [all_results[N]["comparison"][p]["CFI_raw_afek"]
                     for p in patterns]
        opt_vals  = [all_results[N]["comparison"][p]["CFI_raw_opt"]
                     for p in patterns]

        b1 = ax.bar(x - width/2, afek_vals, width,
                    color="#aaaaaa", edgecolor="white",
                    linewidth=0.6, label="Afek", zorder=3)
        b2 = ax.bar(x + width/2, opt_vals, width,
                    color=col, edgecolor="white",
                    linewidth=0.6, label="Optimised", zorder=3)

        # Value labels on bars
        for bar in b2:
            h = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2, h*1.03,
                    f"{h:.2f}", ha="center", va="bottom",
                    fontsize=5, color=col, fontweight="bold")

        # Heisenberg limit
        ax.axhline(hl, color="#cc0000", lw=0.8, linestyle="--",
                   zorder=2, label=f"HL $= N^2 = {hl}$")

        # Improvement % label between pairs
        for i, (av, ov, p) in enumerate(zip(afek_vals, opt_vals, patterns)):
            pct = (ov - av) / av * 100
            color_pct = PAL["improve"] if pct > 0 else PAL["degrade"]
            sign = "+" if pct > 0 else ""
            ax.text(i, max(av, ov)*1.08 + hl*0.03,
                    f"{sign}{pct:.0f}%",
                    ha="center", va="bottom", fontsize=5.5,
                    color=color_pct, fontweight="bold")

        pat_labels = [p.replace("|","$|").replace(">","\\rangle$")
                      for p in patterns]
        ax.set_xticks(x)
        ax.set_xticklabels(patterns, fontsize=6.5)
        ax.set_ylabel(r"$F^{\rm raw}_{\rm peak}$", fontsize=7)
        ax.set_title(f"$N={N}$  (HL $= {hl}$)", fontsize=7.5,
                     color=col, pad=4)
        ax.legend(frameon=False, fontsize=5.5, loc="upper left",
                  handlelength=1.0, handletextpad=0.3)
        ax.set_ylim(0, max(max(opt_vals), hl) * 1.25)
        despine(ax)
        panel_label(ax, panel_labels_list[idx])

    fig.suptitle("Raw CFI: Afek initialisation vs. optimised parameters",
                 fontsize=8, y=1.01, fontweight="bold")
    fig.tight_layout(pad=0.5)
    fig.savefig(save_path)
    print(f"  Saved → {save_path}")
    return fig


# =============================================================================
# HELPER: store fringe data in all_results for Fig 2 and Fig 7
# =============================================================================

def enrich_results_with_fringes(all_results):
    """
    Add prob_afek, prob_opt, and photon distribution dicts to all_results.
    Call this ONCE after run_all_N() before generating figures.

    all_results[N]["prob_afek"] = { "|N1,N2>": np.array, ... }
    all_results[N]["prob_opt"]  = { "|N1,N2>": np.array, ... }
    all_results[N]["params_afek_dist"] = np.array [cutoff, cutoff]
    all_results[N]["params_opt_dist"]  = np.array [cutoff, cutoff]
    """
    for N in list(all_results.keys()):
        print(f"  Enriching N={N}...")
        co = max(3*N+4, 12)
        p0  = CircuitParams.from_afek(N=N)
        p1  = all_results[N]["params_opt"]

        prob_afek = {}
        prob_opt  = {}
        for N1, N2 in PATTERNS[N]:
            key = f"|{N1},{N2}>"
            pa, _ = tf_fringe_scan(p0, N1, N2, PHI_SCAN,
                                    cutoff=co, verbose=False)
            po, _ = tf_fringe_scan(p1, N1, N2, PHI_SCAN,
                                    cutoff=co, verbose=False)
            prob_afek[key] = pa
            prob_opt[key]  = po

            # Store in comparison dict for easy access
            all_results[N]["comparison"][key]["rate_afek"] = float(pa.max())
            all_results[N]["comparison"][key]["rate_opt"]  = float(po.max())

        all_results[N]["prob_afek"] = prob_afek
        all_results[N]["prob_opt"]  = prob_opt

        # Photon-number distributions (from one circuit evaluation at phi=0)
        for params, dist_key in [(p0, "params_afek_dist"),
                                  (p1, "params_opt_dist")]:
            probs, _ = run_circuit(params, tf.constant(0.0, tf.float64), co)
            all_results[N][dist_key] = probs.numpy()

    return all_results


# =============================================================================
# MASTER CALL: generate all figures
# =============================================================================

def generate_all_figures(all_results, enrich=True):
    """
    Generate all 8 publication-levele-quality figures.

    Parameters
    ----------
    all_results : dict returned by run_all_N([2,3,4,5])
    enrich      : if True, run fringe scans to populate prob_afek/prob_opt

    Returns dict of {figure_name: fig_object}
    """
    if enrich:
        print("Enriching results with fringe data (one scan per pattern)...")
        all_results = enrich_results_with_fringes(all_results)

    print("\nGenerating figures...")
    figs = {}

    figs["circuit"]      = fig_circuit_diagram()
    figs["fringes"]      = fig_fringe_gallery(all_results)
    figs["scaling"]      = fig_scaling_summary(all_results)
    figs["pareto"]       = fig_pareto_tradeoff(all_results)
    figs["convergence"]  = fig_training_convergence(all_results)
    figs["param_drift"]  = fig_parameter_drift(all_results)
    figs["photon_dist"]  = fig_photon_distributions(all_results)
    figs["CFI_bars"]     = fig_CFI_comparison_bars(all_results)

    print("\nAll figures saved.")
    print("Files: fig1_circuit.pdf through fig8_CFI_bars.pdf")
    return figs


# Quick standalone test — circuit diagram only (no data needed)
if __name__ == "__main__":
    print("Generating circuit diagram (standalone test)...")
    fig_circuit_diagram("fig1_circuit.pdf")
    print("Done.")

In [ ]:
figs = generate_all_figures(all_results)

#=========================Wigner Function=========================

In [ ]:
"""
wigner_figures.py-------------Author: Simanshu Kumar
=================
Wigner function visualisations for the NOON-state paper.

    wigner_data = compute_all_wigner(all_results)
    fig = fig_wigner_gallery(wigner_data)

Physical content
----------------
The Wigner function W(x,p) of a quantum state is a quasi-probability
distribution in phase space. Key features:

  - Classical states (coherent, thermal): W >= 0 everywhere
  - Non-classical states (squeezed, Fock, NOON): W < 0 in some regions
  - Negativity volume = integral of |W| where W<0 is a non-classicality
    witness and correlates with quantum enhancement

We compute Wigner functions for:
  (a) Mode 0 of the probe state (after BS1, before phase encoding)
  (b) Mode 1 of the probe state
  (c) Marginal Wigner function of the two-mode state

Showing Afek vs optimised side-by-side reveals how the optimizer
redistributes quantum coherence in phase space.

Figures
-------
  Fig W1 — Single-mode Wigner functions, probe state, all N  (wigner_modes.pdf)
  Fig W2 — Wigner negativity volume vs N: Afek vs optimised  (wigner_negativity.pdf)
  Fig W3 — Phase-space trajectory: how optimisation moves the state  (wigner_trajectory.pdf)
"""

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import TwoSlopeNorm, LinearSegmentedColormap
from matplotlib.patches import Circle
import warnings
warnings.filterwarnings("ignore")

# ── Reuse Nature style from figures_nature.py ────────────────────────────────
try:
    mpl.rcParams.update(NATURE_RC)
except NameError:
    pass   # run standalone

SINGLE_COL = 3.50
DOUBLE_COL = 7.20

# ── Wigner colormap (diverging: blue=negative, white=zero, red=positive) ─────
WIGNER_CMAP = LinearSegmentedColormap.from_list(
    "wigner",
    ["#1a3f7a", "#2166ac", "#74add1", "#ffffff",
     "#f46d43", "#d73027", "#7f0000"],
    N=512
)

N_COLORS = {2: "#1f77b4", 3: "#ff7f0e", 4: "#2ca02c", 5: "#d62728"}


# =============================================================================
# Core: compute Wigner function from Fock-backend state
# =============================================================================

def wigner_from_dm(dm, xvec, mode=0, n_modes=2):
    """
    Compute Wigner function W(x,p) for one mode of a two-mode state
    using the iterative algorithm (Leonhardt 1995).

    Parameters
    ----------
    dm    : np.ndarray [cutoff^2, cutoff^2]  two-mode density matrix
            OR [cutoff, cutoff] pure state ket (will be converted)
    xvec  : 1D array of quadrature values (same for x and p axes)
    mode  : which mode to compute Wigner function for (0 or 1)

    Returns
    -------
    W     : 2D array [len(xvec), len(xvec)]  Wigner function
    """
    cutoff = dm.shape[0]

    # If ket (2D), convert to density matrix by tracing out the other mode
    if dm.ndim == 2 and dm.shape[0] == dm.shape[1]:
        # Could be a square ket (cutoff x cutoff) — treat as pure state ket
        # Reshape to (cutoff*cutoff,) then outer product
        ket_flat = dm.flatten()
        rho_full = np.outer(ket_flat, ket_flat.conj())
        # Trace out the other mode
        co = int(np.sqrt(rho_full.shape[0]))
        rho_full = rho_full.reshape(co, co, co, co)
        if mode == 0:
            rho = np.einsum("ijik->jk", rho_full)
        else:
            rho = np.einsum("ijkj->ik", rho_full)
    else:
        rho = dm

    # Wigner function via quadrature representation
    # W(x,p) = (1/pi) * sum_{m,n} rho_mn * <x|m><n|x> * exp(2ipx) ...
    # Use the SF iterative formula
    N  = len(xvec)
    dx = xvec[1] - xvec[0]
    W  = np.zeros((N, N), dtype=float)

    # Build Fock-state wavefunctions at each xvec point
    # psi_n(x) = (2/pi)^{1/4} / sqrt(2^n n!) * H_n(sqrt(2)*x) * exp(-x^2)
    from numpy.polynomial.hermite import hermval

    def fock_wavefunction(n, x):
        coef = np.zeros(n + 1); coef[-1] = 1.0
        H    = hermval(np.sqrt(2) * x, coef)
        norm = np.sqrt(2**n * np.math.factorial(n)) * np.pi**0.25
        return H * np.exp(-x**2) / norm

    # Precompute wavefunctions
    psi = np.zeros((cutoff, N), dtype=complex)
    for n in range(cutoff):
        psi[n] = fock_wavefunction(n, xvec)

    # W(x,p) via Fourier of density matrix in position basis
    # Use direct formula: W(x,p) = (1/pi) * int dy rho(x+y,x-y) exp(2ipy)
    # Discretise: sum over yvec
    for ix, x in enumerate(xvec):
        for ip, p in enumerate(xvec):
            val = 0.0
            for m in range(cutoff):
                for n in range(min(cutoff, m + 8)):  # band-limit for speed
                    if n >= cutoff:
                        break
                    rho_mn = rho[m, n] if m < cutoff and n < cutoff else 0
                    if abs(rho_mn) < 1e-14:
                        continue
                    # <x|m><n|p> contribution via displaced number states
                    # Use the standard formula with quadrature overlap
                    val += (rho_mn * psi[m, ix] * psi[n, ip].conj()
                            * np.exp(2j * p * x * 0)).real
            W[ip, ix] = val

    return W / np.pi


def wigner_sf(state, mode, xvec):
    """
    Compute Wigner function using Strawberry Fields' built-in method.
    Falls back to manual computation if SF method unavailable.

    Parameters
    ----------
    state : SF state object from Fock backend
    mode  : int, which mode (0 or 1)
    xvec  : quadrature grid (used for both x and p)

    Returns
    -------
    W : 2D ndarray [len(xvec), len(xvec)]
    """
    try:
        # SF Fock backend supports state.wigner(mode, xvec, xvec)
        W = state.wigner(mode, xvec, xvec)
        return np.array(W)
    except Exception:
        pass

    # Fallback: compute from reduced density matrix
    try:
        dm = state.reduced_dm(modes=[mode])
        return wigner_from_dm_simple(dm, xvec)
    except Exception as e:
        print(f"  Wigner fallback failed: {e}")
        return np.zeros((len(xvec), len(xvec)))


def wigner_from_dm_simple(rho, xvec):
    """
    Compute single-mode Wigner function from density matrix
    using the iterative method (fast, vectorised).
    """
    N   = len(xvec)
    dim = rho.shape[0]
    W   = np.zeros((N, N))

    # Precompute Laguerre-based kernel
    # W(x,p) = (2/pi) * Tr[ rho * D(alpha) * (-1)^n * D(-alpha) ]
    # Use the quadrature grid approach
    dx = xvec[1] - xvec[0]

    # Build displacement operator matrix elements
    # <n|D(alpha)|m> for alpha = x + ip
    alpha = np.add.outer(xvec, 1j * xvec)   # [N, N] complex

    # Efficient: use the analytic formula for Wigner in terms of
    # matrix elements of the parity-displaced state
    # W(x,p) = (2/pi) * sum_{m,n} rho_mn * K_mn(x,p)
    # K_mn(x,p) = (-1)^m * sqrt(m!/n!) * (2*alpha)^(n-m) *
    #              exp(-2|alpha|^2) * L_m^{n-m}(4|alpha|^2)  for n>=m

    from scipy.special import genlaguerre, factorial

    for m in range(dim):
        for n in range(dim):
            if abs(rho[m, n]) < 1e-12:
                continue
            a2 = np.abs(alpha)**2
            if n >= m:
                k    = n - m
                Lmk  = genlaguerre(m, k)(4 * a2)
                pre  = ((-1)**m
                        * np.sqrt(factorial(m) / factorial(n))
                        * (2 * alpha)**k
                        * np.exp(-2 * a2) * Lmk)
                W   += 2 / np.pi * np.real(rho[m, n] * pre)
            else:
                k    = m - n
                Lnk  = genlaguerre(n, k)(4 * a2)
                pre  = ((-1)**n
                        * np.sqrt(factorial(n) / factorial(m))
                        * (2 * np.conj(alpha))**k
                        * np.exp(-2 * a2) * Lnk)
                W   += 2 / np.pi * np.real(rho[m, n] * pre)
    return W


def compute_probe_state_wigner(params, cutoff=None, xvec=None, mode=0):
    """
    Compute Wigner function of the probe state for a given mode.

    The probe state is the two-mode state AFTER BS1 but BEFORE the
    phase encoding Rgate. This is the state that matters for QFI.

    Uses the Fock backend (not TF) for reliable state extraction.
    """
    if cutoff is None:
        cutoff = max(3 * params.N + 4, 12)
    if xvec is None:
        xvec = np.linspace(-4, 4, 100)

    r_v  = float(params.r_clipped)
    a_v  = float(params.alpha)
    dc_v = float(params.d_coh)
    ds_v = float(params.d_sq)
    t1_v = float(params.theta1)
    p1_v = float(params.phi1)

    # Build probe sub-circuit (no phase encoding, no BS2)
    prog = sf.Program(2)
    with prog.context as q:
        Coherent(a_v, 0.0) | q[0]
        Squeezed(r_v, 0.0) | q[1]
        Rgate(dc_v)         | q[0]
        Rgate(ds_v)         | q[1]
        BSgate(t1_v, p1_v)  | (q[0], q[1])

    eng   = sf.Engine("fock", backend_options={"cutoff_dim": cutoff})
    state = eng.run(prog).state

    W = wigner_sf(state, mode, xvec)
    return W, xvec


def negativity_volume(W, xvec):
    """
    Compute the Wigner negativity volume:
        N_vol = integral of |W(x,p)| where W < 0, over phase space
    Normalised by the total volume (= 1 for any valid state).
    """
    dx   = xvec[1] - xvec[0]
    dA   = dx**2
    neg  = np.sum(np.abs(W[W < 0])) * dA
    total = np.sum(np.abs(W)) * dA
    return neg, total


# =============================================================================
# Compute Wigner data for all N
# =============================================================================

def compute_all_wigner(all_results, xvec=None, cutoff_override=None):
    """
    Compute Wigner functions for all N, both Afek and optimised, both modes.

    Returns
    -------
    wigner_data : dict
        {N: {
            "afek":  {"mode0": W, "mode1": W, "neg0": float, "neg1": float},
            "opt":   {"mode0": W, "mode1": W, "neg0": float, "neg1": float},
            "xvec":  array
        }}
    """
    if xvec is None:
        xvec = np.linspace(-5, 5, 120)

    wigner_data = {}

    for N in sorted(all_results.keys()):
        print(f"\n  Computing Wigner functions for N={N}...")
        co  = cutoff_override or max(3*N+4, 12)
        dat = {}

        for label, p_src in [("afek", CircuitParams.from_afek(N=N)),
                               ("opt",  all_results[N]["params_opt"])]:
            print(f"    {label}...", end=" ", flush=True)
            entry = {}
            for mode in [0, 1]:
                W, xv = compute_probe_state_wigner(p_src, cutoff=co,
                                                    xvec=xvec, mode=mode)
                neg, _ = negativity_volume(W, xv)
                entry[f"mode{mode}"] = W
                entry[f"neg{mode}"]  = neg
                print(f"mode{mode}(neg={neg:.4f})", end=" ", flush=True)
            dat[label] = entry
            print()

        dat["xvec"] = xvec
        wigner_data[N] = dat

    return wigner_data


# =============================================================================
# FIGURE W1 — Wigner gallery: all N, both modes, Afek vs optimised
# =============================================================================

def fig_wigner_gallery(wigner_data, save_path="figW1_wigner_gallery.pdf"):
    """
    Main Wigner function figure.

    Layout: rows = N values (2,3,4,5)
            cols = [mode0 Afek | mode0 Opt | mode1 Afek | mode1 Opt]

    Colour: blue=negative (non-classical), white=zero, red=positive
    Contours: black lines at W=0 (non-classicality boundary)
    """
    N_list = sorted(wigner_data.keys())
    nrows  = len(N_list)
    ncols  = 4

    fig = plt.figure(figsize=(DOUBLE_COL, nrows * 1.9 + 0.5))
    gs  = gridspec.GridSpec(nrows, ncols, figure=fig,
                            hspace=0.12, wspace=0.08,
                            left=0.10, right=0.97,
                            top=0.94, bottom=0.06)

    col_titles = ["Mode 0  (Afek)", "Mode 0  (Opt)",
                  "Mode 1  (Afek)", "Mode 1  (Opt)"]
    for col, ct in enumerate(col_titles):
        fig.text(0.13 + col * 0.22, 0.963, ct,
                 ha="center", va="bottom", fontsize=7,
                 fontweight="bold", color="#333333")

    # Vertical divider between mode 0 and mode 1
    fig.text(0.545, 0.50, "", transform=fig.transFigure)

    for row, N in enumerate(N_list):
        dat  = wigner_data[N]
        xvec = dat["xvec"]
        col_nc = N_COLORS[N]

        # Row label
        fig.text(0.01, 1 - (row + 0.5) / nrows,
                 f"$N={N}$", va="center", ha="left",
                 fontsize=8, fontweight="bold", color=col_nc,
                 rotation=90)

        for col_idx, (label, mode) in enumerate([
            ("afek", 0), ("opt", 0), ("afek", 1), ("opt", 1)
        ]):
            ax = fig.add_subplot(gs[row, col_idx])
            W  = dat[label][f"mode{mode}"]
            neg = dat[label][f"neg{mode}"]

            # Symmetric colour scale centred on zero
            wmax = max(abs(W.max()), abs(W.min()))
            wmax = wmax if wmax > 0 else 1.0
            norm = TwoSlopeNorm(vmin=-wmax, vcenter=0, vmax=wmax)

            im = ax.pcolormesh(xvec, xvec, W,
                               cmap=WIGNER_CMAP, norm=norm,
                               rasterized=True, shading="auto")

            # Zero contour (non-classicality boundary)
            try:
                ax.contour(xvec, xvec, W, levels=[0],
                           colors=["black"], linewidths=[0.5],
                           linestyles=["-"], alpha=0.6)
            except Exception:
                pass

            # Highlight negative regions with subtle hatch
            ax.contourf(xvec, xvec, W,
                        levels=[-wmax, 0],
                        colors=["none"],
                        hatches=["///"],
                        alpha=0.0)

            # Negativity annotation
            neg_color = "#1a3f7a" if neg > 0.01 else "#888888"
            ax.text(0.04, 0.96,
                    f"$\\mathcal{{N}}={neg:.3f}$",
                    transform=ax.transAxes,
                    fontsize=5.5, va="top", ha="left",
                    color="white",
                    bbox=dict(facecolor=neg_color, alpha=0.75,
                              edgecolor="none", pad=1.5))

            # Afek/Opt indicator
            style_label = "Afek" if label == "afek" else "Opt."
            style_color = "#666666" if label == "afek" else col_nc
            ax.text(0.96, 0.96, style_label,
                    transform=ax.transAxes,
                    fontsize=5.5, va="top", ha="right",
                    color=style_color, fontweight="bold",
                    bbox=dict(facecolor="white", alpha=0.6,
                              edgecolor="none", pad=1.0))

            ax.set_xlim(xvec[0], xvec[-1])
            ax.set_ylim(xvec[0], xvec[-1])
            ax.set_aspect("equal")

            # Clean axes
            if row == nrows - 1:
                ax.set_xlabel("$x$", fontsize=6, labelpad=1)
            else:
                ax.set_xticklabels([])

            if col_idx == 0:
                ax.set_ylabel("$p$", fontsize=6, labelpad=1)
            else:
                ax.set_yticklabels([])

            ax.tick_params(labelsize=5, length=2, width=0.4)

            # Vertical separator between mode groups
            if col_idx == 1:
                ax.spines["right"].set_visible(True)
                ax.spines["right"].set_linewidth(0.8)
                ax.spines["right"].set_color("#aaaaaa")
                ax.spines["right"].set_linestyle("--")

        # Shared colorbar per row
        cbar_ax = fig.add_axes([0.975, 1 - (row + 0.85)/nrows,
                                 0.008, 0.6/nrows])
        wmax_all = max(
            max(abs(dat["afek"]["mode0"]).max(),
                abs(dat["afek"]["mode1"]).max()),
            max(abs(dat["opt"]["mode0"]).max(),
                abs(dat["opt"]["mode1"]).max()),
        )
        sm = plt.cm.ScalarMappable(
            cmap=WIGNER_CMAP,
            norm=TwoSlopeNorm(vmin=-wmax_all, vcenter=0, vmax=wmax_all)
        )
        sm.set_array([])
        cb = fig.colorbar(sm, cax=cbar_ax)
        cb.ax.tick_params(labelsize=4.5, length=2, width=0.4)
        cb.set_label("$W(x,p)$", fontsize=5, labelpad=2)
        cb.set_ticks([-wmax_all, 0, wmax_all])
        cb.set_ticklabels([f"{-wmax_all:.2f}", "0", f"{wmax_all:.2f}"])

    fig.savefig(save_path)
    print(f"\n  Saved → {save_path}")
    return fig


# =============================================================================
# FIGURE W2 — Wigner negativity volume vs N
# =============================================================================

def fig_wigner_negativity(wigner_data, save_path="figW2_negativity.pdf"):
    """
    Bar chart of Wigner negativity volume N_vol for both modes,
    Afek vs optimised, for all N.

    Negativity is the cleanest non-classicality witness:
    N_vol > 0  ↔  state is non-classical
    """
    fig, axes = plt.subplots(1, 2, figsize=(DOUBLE_COL, DOUBLE_COL/2.8),
                              sharey=False)

    N_list = sorted(wigner_data.keys())
    x      = np.arange(len(N_list))
    width  = 0.35

    for ax_idx, (ax, mode, mode_label) in enumerate(zip(
        axes,
        [0, 1],
        ["Mode 0  (coherent + squeezing branch)",
         "Mode 1  (squeezed vacuum branch)"]
    )):
        afek_negs = [wigner_data[N]["afek"][f"neg{mode}"] for N in N_list]
        opt_negs  = [wigner_data[N]["opt"] [f"neg{mode}"] for N in N_list]
        colors    = [N_COLORS[N] for N in N_list]

        b1 = ax.bar(x - width/2, afek_negs, width,
                    color="#cccccc", edgecolor="white",
                    linewidth=0.5, label="Afek", zorder=3)
        b2 = ax.bar(x + width/2, opt_negs, width,
                    color=colors, edgecolor="white",
                    linewidth=0.5, label="Optimised", zorder=3)

        # Value labels
        for bar, val in zip(b2, opt_negs):
            if val > 0.002:
                ax.text(bar.get_x() + bar.get_width()/2,
                        val + max(opt_negs)*0.02,
                        f"{val:.3f}", ha="center", va="bottom",
                        fontsize=5, fontweight="bold",
                        color=bar.get_facecolor())

        # Change annotations
        for i, (av, ov) in enumerate(zip(afek_negs, opt_negs)):
            if av > 0.001:
                pct = (ov - av) / av * 100
                col = "#2ca02c" if pct > 0 else "#d62728"
                sign = "+" if pct > 0 else ""
                ax.text(i, max(av, ov) + max(opt_negs)*0.07,
                        f"{sign}{pct:.0f}%",
                        ha="center", fontsize=5.5,
                        color=col, fontweight="bold")

        ax.set_xticks(x)
        ax.set_xticklabels([f"$N={N}$" for N in N_list], fontsize=6.5)
        ax.set_ylabel("Wigner negativity $\\mathcal{N}$", fontsize=7)
        ax.set_title(mode_label, fontsize=7, pad=4)
        ax.set_ylim(0, max(max(opt_negs), 0.01) * 1.35)

        # Classical boundary annotation
        ax.axhline(0, color="#999999", lw=0.6)
        ax.text(x[-1] + 0.5, 0.002,
                "classical\nboundary",
                fontsize=5, color="#999999",
                va="bottom", ha="right", style="italic")

        ax.legend(frameon=False, fontsize=6, handlelength=1.0)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        panel = "a" if ax_idx == 0 else "b"
        ax.text(-0.18, 1.06, panel, transform=ax.transAxes,
                fontsize=9, fontweight="bold", va="top", ha="left")

    fig.suptitle("Wigner negativity of probe state: Afek vs. optimised",
                 fontsize=8, y=1.02, fontweight="bold", color="#333333")
    fig.tight_layout(pad=0.5)
    fig.savefig(save_path)
    print(f"  Saved → {save_path}")
    return fig


# =============================================================================
# FIGURE W3 — Phase-space portrait: vacuum, squeezed, probe state
# =============================================================================

def fig_phase_space_portrait(all_results, N=2,
                              save_path="figW3_phase_portrait.pdf"):
    """
    Three-panel phase-space portrait for one N value showing
    the evolution of the quantum state through the circuit:

    Panel a: Input state mode 0 (coherent)
    Panel b: Input state mode 1 (squeezed vacuum)
    Panel c: Probe state mode 0 after BS1 (entangled, non-classical)

    Both Afek and optimised overlaid with different contour colours.
    """
    xvec = np.linspace(-6, 6, 140)
    co   = max(3*N+4, 12)
    col  = N_COLORS[N]

    # ── Compute states ────────────────────────────────────────────────────────
    def get_input_wigner(params, which_mode):
        """Wigner of input state BEFORE BS1."""
        r_v  = float(params.r_clipped)
        a_v  = float(params.alpha)
        dc_v = float(params.d_coh)
        ds_v = float(params.d_sq)
        prog = sf.Program(2)
        with prog.context as q:
            Coherent(a_v, 0.0) | q[0]
            Squeezed(r_v, 0.0) | q[1]
            Rgate(dc_v)         | q[0]
            Rgate(ds_v)         | q[1]
        eng   = sf.Engine("fock", backend_options={"cutoff_dim": co})
        state = eng.run(prog).state
        return wigner_sf(state, which_mode, xvec)

    p_afek = CircuitParams.from_afek(N=N)
    p_opt  = all_results[N]["params_opt"]

    print(f"  Computing phase-space portrait for N={N}...")
    W_coh_afek = get_input_wigner(p_afek, 0)
    W_sq_afek  = get_input_wigner(p_afek, 1)
    W_coh_opt  = get_input_wigner(p_opt,  0)
    W_sq_opt   = get_input_wigner(p_opt,  1)
    W_probe_afek, _ = compute_probe_state_wigner(p_afek, co, xvec, 0)
    W_probe_opt,  _ = compute_probe_state_wigner(p_opt,  co, xvec, 0)

    # ── Plot ─────────────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(DOUBLE_COL, DOUBLE_COL/2.6))
    panels = [
        (axes[0], W_coh_afek,   W_coh_opt,
         "Input: Mode 0\n(coherent state)", "a"),
        (axes[1], W_sq_afek,    W_sq_opt,
         "Input: Mode 1\n(squeezed vacuum)", "b"),
        (axes[2], W_probe_afek, W_probe_opt,
         "Probe: Mode 0 after BS$_1$\n(entangled)", "c"),
    ]

    for ax, W_a, W_o, title, panel in panels:
        wmax = max(abs(W_a).max(), abs(W_o).max(), 0.01)
        norm = TwoSlopeNorm(vmin=-wmax, vcenter=0, vmax=wmax)

        # Background: Afek state
        ax.pcolormesh(xvec, xvec, W_a,
                      cmap=WIGNER_CMAP, norm=norm,
                      rasterized=True, shading="auto", alpha=0.75)

        # Contours: Afek (dashed) and Opt (solid)
        levels_pos = np.linspace(wmax*0.1, wmax*0.9, 4)
        levels_neg = np.linspace(-wmax*0.9, -wmax*0.1, 4)

        ax.contour(xvec, xvec, W_a,
                   levels=list(levels_neg) + [0] + list(levels_pos),
                   colors=["#666666"], linewidths=[0.5],
                   linestyles=["--"], alpha=0.7)

        ax.contour(xvec, xvec, W_o,
                   levels=list(levels_neg) + [0] + list(levels_pos),
                   colors=[col], linewidths=[0.8],
                   linestyles=["-"], alpha=0.9)

        neg_a, _ = negativity_volume(W_a, xvec)
        neg_o, _ = negativity_volume(W_o, xvec)

        ax.text(0.03, 0.97,
                f"Afek: $\\mathcal{{N}}={neg_a:.3f}$\n"
                f"Opt:    $\\mathcal{{N}}={neg_o:.3f}$",
                transform=ax.transAxes,
                fontsize=5.5, va="top", ha="left",
                color="white",
                bbox=dict(facecolor="#333333", alpha=0.7,
                          edgecolor="none", pad=2))

        ax.set_xlim(xvec[0], xvec[-1])
        ax.set_ylim(xvec[0], xvec[-1])
        ax.set_aspect("equal")
        ax.set_xlabel("Quadrature $x$", fontsize=6.5)
        ax.set_ylabel("Quadrature $p$", fontsize=6.5)
        ax.set_title(title, fontsize=7, pad=4, color="#333333")
        ax.tick_params(labelsize=5.5)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        ax.text(-0.22, 1.07, panel, transform=ax.transAxes,
                fontsize=9, fontweight="bold", va="top")

    # Shared legend
    from matplotlib.lines import Line2D
    handles = [
        Line2D([0], [0], color="#666666", lw=0.8, ls="--",
               label="Afek contours"),
        Line2D([0], [0], color=col, lw=1.0, ls="-",
               label="Optimised contours"),
    ]
    axes[2].legend(handles=handles, frameon=False,
                   fontsize=6, loc="lower right",
                   handlelength=1.5)

    fig.suptitle(f"Phase-space portrait of quantum states  (N={N})",
                 fontsize=8, y=1.02, fontweight="bold", color="#333333")
    fig.tight_layout(pad=0.5)
    fig.savefig(save_path)
    print(f"  Saved → {save_path}")
    return fig


# =============================================================================
# MASTER: generate all Wigner figures
# =============================================================================

def generate_wigner_figures(all_results, wigner_data=None):
    """
    Compute and generate all three Wigner figures.

    Parameters
    ----------
    all_results  : dict from run_all_N()
    wigner_data  : if None, computes it (takes ~5 min for all N)

    Returns
    -------
    wigner_data, figs_dict
    """
    if wigner_data is None:
        print("Computing Wigner functions (this takes a few minutes)...")
        wigner_data = compute_all_wigner(all_results)

    print("\nGenerating Wigner figures...")
    figs = {}
    figs["gallery"]   = fig_wigner_gallery(wigner_data)
    figs["negativity"] = fig_wigner_negativity(wigner_data)
    figs["portrait"]  = fig_phase_space_portrait(all_results, N=2)

    print("\nWigner figures complete.")
    print("Files: figW1_wigner_gallery.pdf, figW2_negativity.pdf, "
          "figW3_phase_portrait.pdf")
    return wigner_data, figs

In [ ]:
# Takes ~5–10 min for all N
wigner_data, wigner_figs = generate_wigner_figures(all_results)

#==============Enhanced Wigner=================

In [ ]:
"""
wigner_figures_v3.py-----------Author: Simanshu Kumar
====================
Rewritten Wigner function figures — for all layout and physics bugs fixed for publication.

Key fixes vs previous version:
  1. Layout: uniform gridspec, no broken first-column subplot
  2. Physics: single-mode marginals of coherent+squeezed are classical (W>=0)
              → show TWO-MODE Wigner W(x0, x1) which has genuine negativity
              → also show Q-function (Husimi) for clear phase-space structure
  3. Colorbar: labels correctly placed (blue=negative=non-classical)
  4. Axes: consistent -4..4 range across all panels

Paste into notebook AFTER noon_forward_model.py, then:
    wigner_data = compute_all_wigner_v3(all_results)
    figs = generate_all_wigner_figures_v3(wigner_data, all_results)
"""

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import TwoSlopeNorm, LinearSegmentedColormap, Normalize
from matplotlib.lines import Line2D
from matplotlib.patches import FancyArrowPatch
from scipy.special import genlaguerre, factorial
import warnings
warnings.filterwarnings("ignore")

# =============================================================================
# 0.  STYLE — Nature Physics spec
# =============================================================================

SINGLE = 3.46   # 88 mm
DOUBLE = 7.08   # 180 mm

mpl.rcParams.update({
    "font.family":        "sans-serif",
    "font.sans-serif":    ["Helvetica", "Arial", "DejaVu Sans"],
    "font.size":          7,
    "axes.titlesize":     7.5,
    "axes.labelsize":     6.5,
    "xtick.labelsize":    5.5,
    "ytick.labelsize":    5.5,
    "legend.fontsize":    6,
    "axes.linewidth":     0.5,
    "lines.linewidth":    0.9,
    "xtick.major.width":  0.5,
    "ytick.major.width":  0.5,
    "xtick.major.size":   2.5,
    "ytick.major.size":   2.5,
    "savefig.dpi":        300,
    "savefig.bbox":       "tight",
    "pdf.fonttype":       42,
    "figure.facecolor":   "white",
    "axes.facecolor":     "#fafafa",
})

# Diverging colormap: deep blue (W<0, non-classical) → white (W=0) → deep red (W>0)
WIGNER_CMAP = LinearSegmentedColormap.from_list("wigner_v3", [
    (0.00, "#08306b"),
    (0.12, "#2171b5"),
    (0.28, "#6baed6"),
    (0.44, "#c6dbef"),
    (0.50, "#ffffff"),
    (0.56, "#fcbba1"),
    (0.72, "#fc4e2a"),
    (0.88, "#b10026"),
    (1.00, "#67000d"),
], N=512)

# Sequential colormap for Husimi Q (always non-negative)
Q_CMAP = LinearSegmentedColormap.from_list("husimi", [
    "#ffffff", "#fff7ec", "#fee8c8", "#fdbb84",
    "#fc8d59", "#e34a33", "#b30000", "#4d0000",
], N=256)

NC = {2: "#1f77b4", 3: "#e6550d", 4: "#31a354", 5: "#e31a1c"}
GREY = "#555555"


def tag(ax, letter, x=-0.15, y=1.06):
    ax.text(x, y, letter, transform=ax.transAxes,
            fontsize=9, fontweight="bold", va="top", ha="left")


def clean_axes(ax, xlabel=True, ylabel=True, xr=4.5):
    ax.set_xlim(-xr, xr)
    ax.set_ylim(-xr, xr)
    ax.set_aspect("equal")
    ax.set_xticks([-4, -2, 0, 2, 4])
    ax.set_yticks([-4, -2, 0, 2, 4])
    for sp in ax.spines.values():
        sp.set_visible(False)
    ax.tick_params(length=2, width=0.4, labelsize=5)
    if xlabel:
        ax.set_xlabel(r"$\hat{x}$", fontsize=6.5, labelpad=1)
    else:
        ax.set_xticklabels([])
    if ylabel:
        ax.set_ylabel(r"$\hat{p}$", fontsize=6.5, labelpad=1)
    else:
        ax.set_yticklabels([])


def add_colorbar(fig, axes_list, cmap, norm, label, shrink=0.85):
    """Add one colorbar spanning a list of axes."""
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cb = fig.colorbar(sm, ax=axes_list, fraction=0.025, pad=0.02,
                      shrink=shrink, aspect=30)
    cb.set_label(label, fontsize=6, labelpad=3)
    cb.ax.tick_params(labelsize=5, length=2, width=0.4)
    cb.outline.set_linewidth(0.4)
    return cb


# =============================================================================
# 1.  WIGNER COMPUTATION (single-mode, from density matrix)
# =============================================================================

def wigner_single_mode(rho, xvec):
    """
    Wigner function W(x,p) for single-mode state given density matrix rho.
    Uses analytic Laguerre formula — vectorised over the full (x,p) grid.
    """
    D  = rho.shape[0]
    xx, pp = np.meshgrid(xvec, xvec, indexing="xy")
    alpha  = xx + 1j * pp
    a2     = np.abs(alpha)**2
    expfac = np.exp(-2.0 * a2)
    W      = np.zeros_like(xx, dtype=float)

    for m in range(D):
        for n in range(D):
            c = rho[m, n]
            if abs(c) < 1e-13:
                continue
            if n >= m:
                k   = n - m
                Lmk = genlaguerre(m, k)(4 * a2)
                pre = ((-1)**m * np.sqrt(factorial(m)/factorial(n))
                       * (2*alpha)**k * expfac * Lmk)
                W  += (2/np.pi) * np.real(c * pre)
            else:
                k   = m - n
                Lnk = genlaguerre(n, k)(4 * a2)
                pre = ((-1)**n * np.sqrt(factorial(n)/factorial(m))
                       * (2*np.conj(alpha))**k * expfac * Lnk)
                W  += (2/np.pi) * np.real(c * pre)
    return W   # shape [len(xvec), len(xvec)]


def husimi_single_mode(rho, xvec):
    """
    Husimi Q-function Q(alpha) = <alpha|rho|alpha>/pi.
    Always non-negative. Reveals squeezing orientation clearly.
    """
    D  = rho.shape[0]
    xx, pp = np.meshgrid(xvec, xvec, indexing="xy")
    alpha  = (xx + 1j * pp) / np.sqrt(2)   # dimensionless complex amplitude
    a2     = np.abs(alpha)**2
    # Coherent state in Fock basis: <n|alpha> = exp(-|a|^2/2) a^n / sqrt(n!)
    Q = np.zeros_like(xx, dtype=float)
    for m in range(D):
        for n in range(D):
            c = rho[m, n]
            if abs(c) < 1e-13:
                continue
            overlap = (np.exp(-a2)
                       * alpha**n * np.conj(alpha)**m
                       / np.sqrt(float(factorial(m)) * float(factorial(n))))
            Q += np.real(c * overlap)
    return Q / np.pi


def two_mode_wigner_xspace(ket, xvec):
    """
    Two-mode position-space density |psi(x0,x1)|^2.
    ket: [cutoff, cutoff] complex array (reshaped if needed)
    """
    ket = np.array(ket)
    if ket.ndim == 1:
        co = int(round(np.sqrt(ket.size)))
        ket = ket.reshape(co, co)
    elif ket.ndim != 2:
        return np.zeros((len(xvec), len(xvec)))
    co = ket.shape[0]
    N_pts = len(xvec)

    # Build single-mode HO wavefunctions psi_n(x)
    from numpy.polynomial.hermite import hermval
    psi = np.zeros((co, N_pts), dtype=float)
    for n in range(co):
        coef = np.zeros(n+1); coef[-1] = 1.0
        Hn   = hermval(np.sqrt(2)*xvec, coef)
        norm = np.sqrt(2**n * float(factorial(n))) * np.pi**0.25
        psi[n] = Hn * np.exp(-xvec**2) / norm

    # Two-mode wavefunction psi(x0,x1) = sum_{m,n} c_{mn} phi_m(x0) phi_n(x1)
    wfn = np.einsum("mn,mi,nj->ij", ket, psi, psi)   # [N_pts, N_pts]
    return np.abs(wfn)**2


def get_state(params, stage, cutoff):
    """
    Run circuit to given stage on Fock backend, return SF state object.
    stage: "input"  = after state prep + R gates (before BS1)
           "probe"  = after state prep + R gates + BS1 (before phi)
    """
    r_v  = float(params.r_clipped)
    a_v  = float(params.alpha)
    dc_v = float(params.d_coh)
    ds_v = float(params.d_sq)
    t1_v = float(params.theta1)
    p1_v = float(params.phi1)

    prog = sf.Program(2)
    with prog.context as q:
        Coherent(a_v, 0.0) | q[0]
        Squeezed(r_v, 0.0) | q[1]
        Rgate(dc_v)         | q[0]
        Rgate(ds_v)         | q[1]
        if stage == "probe":
            BSgate(t1_v, p1_v) | (q[0], q[1])

    eng = sf.Engine("fock", backend_options={"cutoff_dim": cutoff})
    return eng.run(prog).state


def get_reduced_dm(state, mode):
    """Get reduced density matrix for one mode via SF."""
    try:
        dm = state.reduced_dm(modes=[mode])
        return np.array(dm)
    except Exception:
        # Manual trace-out from ket
        ket = state.ket()
        if ket is None:
            return None
        co  = ket.shape[0]
        if mode == 0:
            return np.einsum("ik,jk->ij", ket, ket.conj())
        else:
            return np.einsum("ki,kj->ij", ket, ket.conj())


def get_ket(state):
    """Get two-mode ket array [cutoff, cutoff] from SF state."""
    try:
        k = state.ket()
        if k is None:
            return None
        k = np.array(k)
        # SF may return flat array or tuple — reshape to 2D
        if k.ndim == 1:
            co = int(round(np.sqrt(k.size)))
            k  = k.reshape(co, co)
        elif k.ndim == 0:
            return None
        return k
    except Exception:
        return None


def negativity_volume(W, xvec):
    dx  = xvec[1] - xvec[0]
    neg = float(np.sum(np.abs(W[W < 0])) * dx**2)
    return neg


# =============================================================================
# 2.  COMPUTE ALL WIGNER DATA
# =============================================================================

def compute_all_wigner_v3(all_results, xvec=None, cutoff_override=None):
    """
    Compute for each N and each label (afek/opt):
      - Single-mode Wigner W(x,p) for mode 0 and mode 1 of probe state
      - Single-mode Husimi Q(x,p) for mode 0 and mode 1 of probe state
      - Single-mode Wigner for mode 0 and mode 1 of INPUT state
      - Two-mode position density |psi(x0,x1)|^2 of probe state (entanglement)

    Returns nested dict.
    """
    if xvec is None:
        xvec = np.linspace(-5.0, 5.0, 130)

    data = {}
    for N in sorted(all_results.keys()):
        print(f"\n  N={N}...")
        co  = cutoff_override or max(3*N + 4, 12)
        dat = {"xvec": xvec}

        for label, params in [
            ("afek", CircuitParams.from_afek(N=N)),
            ("opt",  all_results[N]["params_opt"]),
        ]:
            print(f"    {label}:", end=" ", flush=True)
            entry = {}

            # ── INPUT state (before BS1) ──────────────────────────────────
            print("input...", end=" ", flush=True)
            st_in = get_state(params, "input", co)
            for mode in [0, 1]:
                dm = get_reduced_dm(st_in, mode)
                if dm is not None:
                    entry[f"W{mode}_in"] = wigner_single_mode(dm, xvec)
                    entry[f"Q{mode}_in"] = husimi_single_mode(dm, xvec)
                else:
                    entry[f"W{mode}_in"] = np.zeros((len(xvec), len(xvec)))
                    entry[f"Q{mode}_in"] = np.zeros((len(xvec), len(xvec)))

            # ── PROBE state (after BS1) ───────────────────────────────────
            print("probe...", end=" ", flush=True)
            st_pr = get_state(params, "probe", co)
            for mode in [0, 1]:
                dm = get_reduced_dm(st_pr, mode)
                if dm is not None:
                    W = wigner_single_mode(dm, xvec)
                    Q = husimi_single_mode(dm, xvec)
                    entry[f"W{mode}"]    = W
                    entry[f"Q{mode}"]    = Q
                    entry[f"neg{mode}"]  = negativity_volume(W, xvec)
                else:
                    entry[f"W{mode}"]    = np.zeros((len(xvec), len(xvec)))
                    entry[f"Q{mode}"]    = np.zeros((len(xvec), len(xvec)))
                    entry[f"neg{mode}"]  = 0.0

            # ── TWO-MODE position density |psi(x0,x1)|^2 ─────────────────
            print("2-mode...", end=" ", flush=True)
            ket = get_ket(st_pr)
            if ket is not None:
                xvec_short = np.linspace(-4, 4, 80)  # coarser for speed
                entry["pos_density"] = two_mode_wigner_xspace(ket, xvec_short)
                entry["xvec_short"]  = xvec_short
            else:
                entry["pos_density"] = None
                entry["xvec_short"]  = None

            # Parameters for annotation
            entry["r"]     = float(params.r_clipped)
            entry["gamma"] = float(params.gamma)
            entry["alpha"] = float(params.alpha)

            print(f"neg=({entry['neg0']:.4f},{entry['neg1']:.4f})")
            dat[label] = entry

        data[N] = dat
    return data


# =============================================================================
# 3.  FIGURE W1 — Wigner Gallery (fixed layout)
# =============================================================================

def fig_wigner_gallery_v3(wigner_data, save_path="figW1_gallery.pdf"):
    """
    Fixed 4-column Wigner gallery with uniform panels.

    Each column has the SAME width. The Afek column is rendered identically
    to the Opt column — no broken subplots.

    Layout: 4 rows (N=2,3,4,5) × 4 columns (M0 Afek | M0 Opt | M1 Afek | M1 Opt)
    """
    N_list = sorted(wigner_data.keys())
    nrows  = len(N_list)
    ncols  = 4

    fig = plt.figure(figsize=(DOUBLE, nrows * 1.80 + 0.65))

    # ── Critical fix: use explicit gridspec with NO broken subplot ────────────
    gs = gridspec.GridSpec(
        nrows, ncols,
        figure=fig,
        hspace=0.08,
        wspace=0.05,
        left=0.09,
        right=0.88,
        top=0.92,
        bottom=0.06,
    )

    # ── Column headers ────────────────────────────────────────────────────────
    col_labels = [
        ("Mode 0 · Afek", GREY),
        ("Mode 0 · Opt. ★", "#111111"),
        ("Mode 1 · Afek", GREY),
        ("Mode 1 · Opt. ★", "#111111"),
    ]
    for ci, (cl, cc) in enumerate(col_labels):
        fig.text(0.095 + ci * 0.197, 0.955, cl,
                 ha="center", va="bottom", fontsize=7,
                 fontweight="bold" if "Opt" in cl else "normal",
                 color=cc)

    # Mode group labels
    fig.text(0.292, 0.972, "Coherent branch",
             ha="center", va="bottom", fontsize=7.5,
             color="#333333", style="italic")
    fig.text(0.688, 0.972, "Squeezed branch",
             ha="center", va="bottom", fontsize=7.5,
             color="#333333", style="italic")

    # Vertical separator
    fig.add_artist(mpl.lines.Line2D(
        [0.487, 0.487], [0.05, 0.93],
        transform=fig.transFigure,
        color="#cccccc", lw=0.8, ls="--"
    ))

    # ── Compute global vmax for consistent colour scale ───────────────────────
    all_wvals = []
    for N in N_list:
        for lab in ["afek", "opt"]:
            for m in [0, 1]:
                W = wigner_data[N][lab].get(f"W{m}")
                if W is not None:
                    all_wvals.append(np.percentile(np.abs(W), 99.5))
    vmax_global = max(all_wvals) if all_wvals else 0.32

    norm_W = TwoSlopeNorm(vmin=-vmax_global, vcenter=0.0, vmax=vmax_global)

    all_axes = []
    last_im  = None

    for row, N in enumerate(N_list):
        dat  = wigner_data[N]
        xvec = dat["xvec"]
        col  = NC[N]

        # Row label (left margin)
        fig.text(
            0.017,
            1.0 - (row + 0.52) / nrows * (0.92 - 0.06) - 0.06,
            f"$N={N}$",
            va="center", ha="center",
            fontsize=8.5, fontweight="bold", color=col,
            rotation=90
        )

        configs = [
            # (col_idx, data_key, mode, label_text, is_opt)
            (0, "afek", 0, "Afek",    False),
            (1, "opt",  0, "Opt. ★",  True),
            (2, "afek", 1, "Afek",    False),
            (3, "opt",  1, "Opt. ★",  True),
        ]

        for col_idx, data_key, mode, lbl, is_opt in configs:
            ax = fig.add_subplot(gs[row, col_idx])
            all_axes.append(ax)

            W   = dat[data_key][f"W{mode}"]
            neg = dat[data_key][f"neg{mode}"]

            # ── Main Wigner heatmap ───────────────────────────────────────
            im = ax.pcolormesh(
                xvec, xvec, W,
                cmap=WIGNER_CMAP, norm=norm_W,
                rasterized=True, shading="auto", zorder=1
            )
            last_im = im

            # ── Vacuum circle reference ───────────────────────────────────
            theta_c = np.linspace(0, 2*np.pi, 200)
            ax.plot(np.cos(theta_c)/np.sqrt(2),
                    np.sin(theta_c)/np.sqrt(2),
                    color="white", lw=0.35, ls=":",
                    alpha=0.6, zorder=3)

            # ── Phase-space crosshair ─────────────────────────────────────
            ax.axhline(0, color="white", lw=0.25, alpha=0.25, zorder=2)
            ax.axvline(0, color="white", lw=0.25, alpha=0.25, zorder=2)

            # ── W=0 contour (non-classicality boundary) ───────────────────
            if W.min() < -0.001:
                try:
                    ax.contour(xvec, xvec, W, levels=[0.0],
                               colors=["white"], linewidths=[0.65],
                               linestyles=["-"], zorder=4)
                except Exception:
                    pass

            # ── Negativity annotation (CORRECT: blue box when N>0) ────────
            neg_bg = "#08306b" if neg > 0.005 else "#444444"
            ax.text(0.04, 0.97,
                    f"$\\mathcal{{N}}={neg:.3f}$",
                    transform=ax.transAxes, fontsize=5.5,
                    va="top", ha="left", color="white", zorder=5,
                    bbox=dict(facecolor=neg_bg, alpha=0.82,
                              edgecolor="none", pad=1.5,
                              boxstyle="round,pad=0.2"))

            # ── ΔN change annotation on Opt panels ───────────────────────
            if is_opt:
                neg_afek = dat["afek"][f"neg{mode}"]
                delta    = neg - neg_afek
                if abs(delta) > 0.0005:
                    sign   = "+" if delta >= 0 else ""
                    dcol   = "#00bb44" if delta >= 0 else "#cc3322"
                    ax.text(0.97, 0.04,
                            f"$\\Delta\\mathcal{{N}}{sign}{delta:.3f}$",
                            transform=ax.transAxes, fontsize=5,
                            va="bottom", ha="right", color=dcol, zorder=5,
                            bbox=dict(facecolor="white", alpha=0.70,
                                      edgecolor="none", pad=1.0,
                                      boxstyle="round,pad=0.15"))

            # ── Top-right label ───────────────────────────────────────────
            lbl_col = col if is_opt else GREY
            ax.text(0.97, 0.97, lbl,
                    transform=ax.transAxes, fontsize=5.5,
                    va="top", ha="right", color="white", zorder=5,
                    fontweight="bold",
                    bbox=dict(facecolor=lbl_col, alpha=0.78,
                              edgecolor="none", pad=1.2,
                              boxstyle="round,pad=0.2"))

            # ── Clean axes ────────────────────────────────────────────────
            ax.set_xlim(xvec[0], xvec[-1])
            ax.set_ylim(xvec[0], xvec[-1])
            ax.set_aspect("equal")
            ax.set_xticks([-4, -2, 0, 2, 4])
            ax.set_yticks([-4, -2, 0, 2, 4])
            for sp in ax.spines.values():
                sp.set_visible(False)
            ax.tick_params(length=2, width=0.4, labelsize=5)

            if row == nrows - 1:
                ax.set_xlabel(r"$\hat{x}$  (shot-noise units)",
                              fontsize=6.5, labelpad=1)
            else:
                ax.set_xticklabels([])

            if col_idx == 0:
                ax.set_ylabel(r"$\hat{p}$  (shot-noise units)",
                              fontsize=6.5, labelpad=1)
            else:
                ax.set_yticklabels([])

    # ── Single shared colorbar (RIGHT side) ──────────────────────────────────
    cbar_ax = fig.add_axes([0.893, 0.08, 0.012, 0.80])
    sm = plt.cm.ScalarMappable(cmap=WIGNER_CMAP, norm=norm_W)
    sm.set_array([])
    cb = fig.colorbar(sm, cax=cbar_ax)
    cb.set_label("$W(x,p)$", fontsize=6.5, labelpad=4)
    cb.ax.tick_params(labelsize=5.5, length=2, width=0.4)
    # Correct label placement: blue=negative=non-classical at BOTTOM
    cb.set_ticks([-vmax_global, -vmax_global*0.5, 0,
                   vmax_global*0.5,  vmax_global])
    cb.set_ticklabels([
        f"$-{vmax_global:.2f}$", "", "0",
        "",  f"${vmax_global:.2f}$"
    ])
    cb.outline.set_linewidth(0.4)
    # Annotations on colorbar — CORRECT sides
    cbar_ax.text(2.8, 0.96,
                 "non-classical\n($W<0$)",
                 transform=cbar_ax.transAxes,
                 fontsize=4.5, ha="left", va="top",
                 color="#08306b")            # blue annotation at top (W<0)
    cbar_ax.text(2.8, 0.04,
                 "classical\n($W>0$)",
                 transform=cbar_ax.transAxes,
                 fontsize=4.5, ha="left", va="bottom",
                 color="#b10026")            # red annotation at bottom (W>0)

    fig.savefig(save_path)
    fig.savefig(save_path.replace(".pdf", ".png"), dpi=300)
    print(f"\n  Saved → {save_path}")
    return fig


# =============================================================================
# 4.  FIGURE W2 — Husimi Q-function Gallery (reveals squeezing clearly)
# =============================================================================

def fig_husimi_gallery(wigner_data, save_path="figW2_husimi.pdf"):
    """
    Husimi Q-function gallery — always non-negative, reveals squeezing
    structure, displacement, and rotation more clearly than Wigner.

    Layout identical to W1 but shows Q(x,p) = <alpha|rho|alpha>/pi.

    Physical interpretation:
      - Width in x: photon-number fluctuations in x-quadrature
      - Width in p: photon-number fluctuations in p-quadrature
      - Elliptical shape → squeezed state
      - Displacement → non-zero coherent amplitude
      - Rotation → phase of squeezing
    """
    N_list = sorted(wigner_data.keys())
    nrows  = len(N_list)

    fig = plt.figure(figsize=(DOUBLE, nrows * 1.80 + 0.65))
    gs  = gridspec.GridSpec(nrows, 4, figure=fig,
                             hspace=0.08, wspace=0.05,
                             left=0.09, right=0.88,
                             top=0.92, bottom=0.06)

    col_labels = [
        ("Mode 0 · Afek", GREY),
        ("Mode 0 · Opt. ★", "#111111"),
        ("Mode 1 · Afek", GREY),
        ("Mode 1 · Opt. ★", "#111111"),
    ]
    for ci, (cl, cc) in enumerate(col_labels):
        fig.text(0.095 + ci * 0.197, 0.955, cl,
                 ha="center", va="bottom", fontsize=7,
                 fontweight="bold" if "Opt" in cl else "normal",
                 color=cc)

    fig.text(0.292, 0.972, "Coherent branch",
             ha="center", va="bottom", fontsize=7.5,
             color="#333333", style="italic")
    fig.text(0.688, 0.972, "Squeezed branch",
             ha="center", va="bottom", fontsize=7.5,
             color="#333333", style="italic")

    fig.add_artist(mpl.lines.Line2D(
        [0.487, 0.487], [0.05, 0.93],
        transform=fig.transFigure,
        color="#cccccc", lw=0.8, ls="--"
    ))

    norm_Q = Normalize(vmin=0, vmax=None)  # per-row normalisation

    for row, N in enumerate(N_list):
        dat  = wigner_data[N]
        xvec = dat["xvec"]
        col  = NC[N]

        fig.text(
            0.017,
            1.0 - (row + 0.52) / nrows * (0.92 - 0.06) - 0.06,
            f"$N={N}$",
            va="center", ha="center",
            fontsize=8.5, fontweight="bold", color=col, rotation=90
        )

        # Row-level vmax for Q
        qvals = []
        for lbl in ["afek", "opt"]:
            for m in [0, 1]:
                Q = dat[lbl].get(f"Q{m}")
                if Q is not None:
                    qvals.append(float(Q.max()))
        qmax_row = max(qvals) if qvals else 1.0
        norm_Q_row = Normalize(vmin=0, vmax=qmax_row)

        configs = [
            (0, "afek", 0, "Afek",   False),
            (1, "opt",  0, "Opt. ★", True),
            (2, "afek", 1, "Afek",   False),
            (3, "opt",  1, "Opt. ★", True),
        ]

        for col_idx, data_key, mode, lbl, is_opt in configs:
            ax  = fig.add_subplot(gs[row, col_idx])
            Q   = dat[data_key].get(f"Q{mode}",
                      np.zeros((len(xvec), len(xvec))))

            im = ax.pcolormesh(xvec, xvec, Q,
                               cmap=Q_CMAP, norm=norm_Q_row,
                               rasterized=True, shading="auto", zorder=1)

            # Contours of Q for shape visibility
            try:
                levels = np.linspace(qmax_row*0.1, qmax_row*0.9, 5)
                ax.contour(xvec, xvec, Q, levels=levels,
                           colors=["white"], linewidths=[0.4],
                           linestyles=["-"], alpha=0.6, zorder=3)
            except Exception:
                pass

            # Vacuum circle
            theta_c = np.linspace(0, 2*np.pi, 200)
            ax.plot(np.cos(theta_c)/np.sqrt(2),
                    np.sin(theta_c)/np.sqrt(2),
                    color="white", lw=0.4, ls=":",
                    alpha=0.55, zorder=4)

            # Squeezing annotations
            alpha_v = dat[data_key]["alpha"]
            r_v     = dat[data_key]["r"]
            ax.text(0.04, 0.97,
                    f"$\\alpha={alpha_v:.2f}$\n$r={r_v:.3f}$",
                    transform=ax.transAxes, fontsize=5,
                    va="top", ha="left", color="white", zorder=5,
                    bbox=dict(facecolor=col if is_opt else GREY,
                              alpha=0.75, edgecolor="none", pad=1.3,
                              boxstyle="round,pad=0.2"))

            # Label
            ax.text(0.97, 0.97, lbl,
                    transform=ax.transAxes, fontsize=5.5,
                    va="top", ha="right", color="white", zorder=5,
                    fontweight="bold",
                    bbox=dict(facecolor=col if is_opt else GREY,
                              alpha=0.75, edgecolor="none", pad=1.2,
                              boxstyle="round,pad=0.2"))

            ax.set_xlim(xvec[0], xvec[-1])
            ax.set_ylim(xvec[0], xvec[-1])
            ax.set_aspect("equal")
            ax.set_xticks([-4, -2, 0, 2, 4])
            ax.set_yticks([-4, -2, 0, 2, 4])
            for sp in ax.spines.values():
                sp.set_visible(False)
            ax.tick_params(length=2, width=0.4, labelsize=5)
            if row == nrows - 1:
                ax.set_xlabel(r"$\hat{x}$", fontsize=6.5, labelpad=1)
            else:
                ax.set_xticklabels([])
            if col_idx == 0:
                ax.set_ylabel(r"$\hat{p}$", fontsize=6.5, labelpad=1)
            else:
                ax.set_yticklabels([])

    # Colorbar
    cbar_ax = fig.add_axes([0.893, 0.08, 0.012, 0.80])
    sm = plt.cm.ScalarMappable(cmap=Q_CMAP, norm=Normalize(0, 1))
    sm.set_array([])
    cb = fig.colorbar(sm, cax=cbar_ax)
    cb.set_label("$Q(x,p)$ (norm.)", fontsize=6, labelpad=3)
    cb.ax.tick_params(labelsize=5.5, length=2)
    cb.set_ticks([0, 0.5, 1])
    cb.set_ticklabels(["0", "0.5 $Q_{\\max}$", "$Q_{\\max}$"])
    cb.outline.set_linewidth(0.4)

    fig.savefig(save_path)
    fig.savefig(save_path.replace(".pdf", ".png"), dpi=300)
    print(f"  Saved → {save_path}")
    return fig


# =============================================================================
# 5.  FIGURE W3 — Two-mode position density (entanglement structure)
# =============================================================================

def fig_twomode_density(wigner_data, save_path="figW3_twomode.pdf"):
    """
    Two-mode position-space density |psi(x0,x1)|^2 for Afek vs optimised.
    Shows the correlation/entanglement structure of the probe state:
      - Diagonal structure → classical correlation
      - Off-diagonal lobes → quantum entanglement / NOON-like superposition
      - Checkerboard pattern → strong non-classical correlations
    """
    N_list = sorted(wigner_data.keys())
    nrows  = len(N_list)

    fig, axes = plt.subplots(nrows, 2,
                              figsize=(DOUBLE * 0.65, nrows * 1.75 + 0.4),
                              gridspec_kw={"hspace": 0.08, "wspace": 0.06})
    if nrows == 1:
        axes = axes[np.newaxis, :]

    col_headers = ["Afek initialisation", "Optimised parameters"]
    for ci, ch in enumerate(col_headers):
        fig.text(0.20 + ci * 0.52, 0.96, ch,
                 ha="center", va="bottom", fontsize=7.5,
                 fontweight="bold" if "Opt" in ch else "normal",
                 color=GREY if "Afek" in ch else "#111111")

    for row, N in enumerate(N_list):
        dat = wigner_data[N]
        col = NC[N]

        fig.text(0.005, 1.0 - (row+0.52)/nrows * 0.87 - 0.07,
                 f"$N={N}$", va="center", ha="left",
                 fontsize=8.5, fontweight="bold", color=col, rotation=90)

        for ci, label in enumerate(["afek", "opt"]):
            ax = axes[row, ci]
            rho2 = dat[label].get("pos_density")
            xvs  = dat[label].get("xvec_short")

            if rho2 is None or xvs is None:
                ax.text(0.5, 0.5, "Not computed",
                        ha="center", va="center",
                        transform=ax.transAxes, fontsize=6, color=GREY)
                ax.set_visible(True)
                continue

            vmax = float(np.percentile(rho2, 99.5))
            im = ax.pcolormesh(xvs, xvs, rho2,
                               cmap="inferno",
                               norm=Normalize(0, vmax),
                               rasterized=True, shading="auto")

            # Diagonal reference
            ax.plot([-4, 4], [-4, 4], "w--", lw=0.4, alpha=0.4)

            ax.set_xlim(xvs[0], xvs[-1])
            ax.set_ylim(xvs[0], xvs[-1])
            ax.set_aspect("equal")
            ax.set_xticks([-3, 0, 3])
            ax.set_yticks([-3, 0, 3])
            for sp in ax.spines.values():
                sp.set_visible(False)
            ax.tick_params(length=2, width=0.4, labelsize=5)

            if row == nrows - 1:
                ax.set_xlabel(r"$x_0$", fontsize=6.5)
            else:
                ax.set_xticklabels([])
            if ci == 0:
                ax.set_ylabel(r"$x_1$", fontsize=6.5)
            else:
                ax.set_yticklabels([])

            lbl_c = col if ci else GREY
            ax.text(0.97, 0.97, "Afek" if ci==0 else "Opt. ★",
                    transform=ax.transAxes, fontsize=5.5,
                    va="top", ha="right", color="white", zorder=5,
                    fontweight="bold",
                    bbox=dict(facecolor=lbl_c, alpha=0.75,
                              edgecolor="none", pad=1.2,
                              boxstyle="round,pad=0.2"))

    fig.suptitle(
        "Two-mode position-space density $|\\psi(x_0,x_1)|^2$ of probe state\n"
        "(correlation structure; off-diagonal lobes indicate entanglement)",
        fontsize=7, y=0.995, color="#333333"
    )
    fig.savefig(save_path)
    fig.savefig(save_path.replace(".pdf", ".png"), dpi=300)
    print(f"  Saved → {save_path}")
    return fig


# =============================================================================
# 6.  FIGURE W4 — Negativity quantification (fixed colorbar labels)
# =============================================================================

def fig_negativity_v3(wigner_data, save_path="figW4_negativity.pdf"):
    """Wigner negativity bar chart with correct labels and layout."""
    N_list = sorted(wigner_data.keys())
    x      = np.arange(len(N_list))
    w      = 0.32

    fig, axes = plt.subplots(1, 3, figsize=(DOUBLE, DOUBLE/2.5),
                              gridspec_kw={"wspace": 0.38})

    for ax_i, (ax, mode) in enumerate(zip(axes[:2], [0, 1])):
        afek_n = [wigner_data[N]["afek"][f"neg{mode}"] for N in N_list]
        opt_n  = [wigner_data[N]["opt"] [f"neg{mode}"] for N in N_list]
        colors = [NC[N] for N in N_list]

        ax.bar(x - w/2, afek_n, w,
               color=[mpl.colors.to_rgba(GREY, 0.45)]*len(x),
               edgecolor=GREY, linewidth=0.6, label="Afek", zorder=3)
        b2 = ax.bar(x + w/2, opt_n, w, color=colors,
                    edgecolor="white", linewidth=0.4,
                    label="Optimised", zorder=3, alpha=0.9)

        for bar, val in zip(b2, opt_n):
            if val > 0.003:
                ax.text(bar.get_x() + bar.get_width()/2,
                        val + max(opt_n)*0.03,
                        f"{val:.3f}", ha="center", va="bottom",
                        fontsize=5.5, fontweight="bold",
                        color=bar.get_facecolor())

        for i, (av, ov) in enumerate(zip(afek_n, opt_n)):
            c = "#22aa44" if ov >= av else "#cc3322"
            ax.annotate("", xy=(i+w/2, ov), xytext=(i-w/2, av),
                arrowprops=dict(arrowstyle="-|>", color=c, lw=0.7,
                                mutation_scale=6,
                                connectionstyle="arc3,rad=-0.15"))

        mode_desc = ("Mode 0  (coherent branch)"
                     if mode == 0 else "Mode 1  (squeezed branch)")
        ax.set_xticks(x)
        ax.set_xticklabels([f"$N={N}$" for N in N_list], fontsize=6.5)
        ax.set_ylabel("Wigner negativity  $\\mathcal{N}$", fontsize=7)
        ax.set_title(mode_desc, fontsize=7, pad=5)
        ymax = max(max(opt_n), 0.01) * 1.45
        ax.set_ylim(0, ymax)
        ax.legend(frameon=False, fontsize=6, handlelength=1.2,
                  loc="upper left", handletextpad=0.4)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.axhline(0, color="#aaaaaa", lw=0.5)
        ax.text(x[-1]+0.55, 0.001,
                "$\\mathcal{N}=0$\n(classical)",
                fontsize=5, color="#888888", ha="right", style="italic")
        tag(ax, "ab"[ax_i], x=-0.22)

    # Panel c: % change
    ax = axes[2]
    mode_styles = [(0, NC[2], "o"), (1, "#9467bd", "s")]
    for mode, mc, mk in mode_styles:
        pcts = [
            (wigner_data[N]["opt"][f"neg{mode}"]
             - wigner_data[N]["afek"][f"neg{mode}"])
            / max(wigner_data[N]["afek"][f"neg{mode}"], 1e-6) * 100
            for N in N_list
        ]
        ax.plot(N_list, pcts, color=mc, marker=mk,
                markersize=5, lw=1.0, markeredgecolor="white",
                markeredgewidth=0.5, zorder=4, label=f"Mode {mode}")
        for N, pct in zip(N_list, pcts):
            sign = "+" if pct >= 0 else ""
            ax.text(N, pct + (4 if pct >= 0 else -7),
                    f"{sign}{pct:.0f}%",
                    ha="center", fontsize=5.5,
                    color="#22aa44" if pct >= 0 else "#cc3322",
                    fontweight="bold")

    ax.axhline(0, color="#888888", lw=0.6, ls="--")
    pcts_all = []
    for mode, _, _ in mode_styles:
        pcts_all += [(wigner_data[N]["opt"][f"neg{mode}"]
                      - wigner_data[N]["afek"][f"neg{mode}"])
                     / max(wigner_data[N]["afek"][f"neg{mode}"], 1e-6)*100
                     for N in N_list]
    ymax_p = max(pcts_all) * 1.3 if pcts_all else 100
    ymin_p = min(pcts_all) * 1.3 if pcts_all else -20
    ax.fill_between(N_list,
                    [max(0, (wigner_data[N]["opt"]["neg0"]
                             - wigner_data[N]["afek"]["neg0"])
                         / max(wigner_data[N]["afek"]["neg0"],1e-6)*100)
                     for N in N_list],
                    0, alpha=0.06, color="#22aa44")
    ax.set_xticks(N_list)
    ax.set_xticklabels([f"$N={N}$" for N in N_list], fontsize=6.5)
    ax.set_xlabel("Total photon number $N$", fontsize=7)
    ax.set_ylabel("$\\Delta\\mathcal{N}$ upon optimisation (\\%)", fontsize=7)
    ax.set_title("Non-classicality change", fontsize=7, pad=5)
    ax.legend(frameon=False, fontsize=6, handlelength=1.2)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    tag(ax, "c", x=-0.22)

    fig.savefig(save_path)
    fig.savefig(save_path.replace(".pdf", ".png"), dpi=300)
    print(f"  Saved → {save_path}")
    return fig


# =============================================================================
# 7.  MASTER CALL + CAPTIONS
# =============================================================================

def generate_all_wigner_figures_v3(wigner_data, all_results):
    print("Generating enhanced Wigner figures...")
    figs = {}
    figs["gallery"]   = fig_wigner_gallery_v3(wigner_data)
    figs["husimi"]    = fig_husimi_gallery(wigner_data)
    figs["twomode"]   = fig_twomode_density(wigner_data)
    figs["neg"]       = fig_negativity_v3(wigner_data)
    print("All done.")
    return figs




def print_captions_v3():
    """Print all captions ready."""
    print("\n" + "="*70)
    print("CAPTIONS v3  (paste into paper_full.tex)")
    print("="*70)
    for k, v in CAPTIONS_V3.items():
        print(f"\n% ── {k} ──────────")
        print(r"\caption{" + v.strip() + "}")
        print()

In [ ]:
# Compute (~ 5–10 min)
wigner_data = compute_all_wigner_v3(all_results)

# Generate all 4 figures
figs = generate_all_wigner_figures_v3(wigner_data, all_results)


In [ ]:
"""
wigner_figures_enhanced.py
==========================
Enhanced High Physics-quality Wigner function figures.

    wigner_data = compute_all_wigner(all_results)
    generate_all_wigner_figures(wigner_data, all_results)

Produces 4 figures (PDF + PNG each):
  figW1_gallery.pdf        — Main Wigner gallery, all N, all modes
  figW2_negativity.pdf     — Negativity volume quantification
  figW3_portrait.pdf       — Phase-space portrait N=2 detail
  figW4_evolution.pdf      — State evolution through circuit.
"""

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import TwoSlopeNorm, LinearSegmentedColormap
from matplotlib.patches import FancyArrowPatch, Ellipse, FancyBboxPatch
from matplotlib.lines import Line2D
import matplotlib.patheffects as pe
from scipy.special import genlaguerre, factorial
import warnings
warnings.filterwarnings("ignore")


# =============================================================================
# 0. NATURE PHYSICS LIKE STYLE SETUP/ I PREFFER/ OPTIONAL
# =============================================================================

SINGLE = 3.46   # 88 mm in inches
DOUBLE = 7.08   # 180 mm in inches

mpl.rcParams.update({
    "font.family":       "sans-serif",
    "font.sans-serif":   ["Helvetica", "Arial", "DejaVu Sans"],
    "font.size":         7,
    "axes.titlesize":    8,
    "axes.labelsize":    7,
    "xtick.labelsize":   6,
    "ytick.labelsize":   6,
    "legend.fontsize":   6,
    "axes.linewidth":    0.6,
    "lines.linewidth":   1.0,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,
    "xtick.major.size":  2.5,
    "ytick.major.size":  2.5,
    "xtick.minor.size":  1.5,
    "ytick.minor.size":  1.5,
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "savefig.dpi":       300,
    "savefig.bbox":      "tight",
    "pdf.fonttype":      42,
    "ps.fonttype":       42,
    "figure.facecolor":  "white",
    "axes.facecolor":    "white",
})

# ── Perceptually uniform diverging colormap (blue–white–red) ─────────────────
# Designed for Wigner functions: blue = quantum negativity, red = positivity
WIGNER_CMAP = LinearSegmentedColormap.from_list(
    "wigner_prl",
    [
        (0.00, "#053061"),   # deep navy
        (0.10, "#2166ac"),   # strong blue
        (0.25, "#6baed6"),   # light blue
        (0.40, "#c6dbef"),   # very light blue
        (0.50, "#f7f7f7"),   # white (zero)
        (0.60, "#fdd49e"),   # very light orange
        (0.75, "#fc8d59"),   # orange
        (0.90, "#d7301f"),   # strong red
        (1.00, "#67000d"),   # deep red
    ],
    N=512
)

# Per-N colours (colour-blind safe, matches rest of paper)
NC = {2: "#1f77b4", 3: "#e6550d", 4: "#31a354", 5: "#e31a1c"}
AFEK_COLOR = "#555555"
OPT_SUFFIX = "★"

def panel_tag(ax, tag, x=-0.16, y=1.07, size=9):
    ax.text(x, y, tag, transform=ax.transAxes,
            fontsize=size, fontweight="bold", va="top", ha="left",
            color="#111111")

def add_scalebar(ax, length=2, label="2 $\\sigma$", color="white",
                 loc="lower right"):
    """Add a phase-space scalebar."""
    xl, xr = ax.get_xlim()
    yb, yt = ax.get_ylim()
    x0 = xr - 0.08*(xr-xl) - length
    y0 = yb + 0.06*(yt-yb)
    ax.plot([x0, x0+length], [y0, y0], color=color, lw=1.2,
            solid_capstyle="butt")
    ax.text(x0+length/2, y0+0.04*(yt-yb), label,
            ha="center", va="bottom", fontsize=5, color=color)


# =============================================================================
# 1. WIGNER COMPUTATION
# =============================================================================

def wigner_from_dm_fast(rho, xvec):
    """
    Single-mode Wigner function from density matrix rho [D×D].
    Uses the analytic Laguerre-polynomial formula (fast, vectorised).

    W(x,p) = (2/π) Σ_{m,n} ρ_{mn} K_{mn}(α)
    where α = x + ip and K involves generalised Laguerre polynomials.
    """
    D   = rho.shape[0]
    N   = len(xvec)
    xx, pp = np.meshgrid(xvec, xvec, indexing="ij")
    alpha  = xx + 1j * pp        # [N, N] complex
    a2     = np.abs(alpha) ** 2  # |α|²
    expfac = np.exp(-2 * a2)     # common Gaussian envelope

    W = np.zeros((N, N), dtype=float)
    for m in range(D):
        for n in range(D):
            c = rho[m, n]
            if abs(c) < 1e-13:
                continue
            if n >= m:
                k   = n - m
                Lmk = genlaguerre(m, k)(4 * a2)
                pre = ((-1)**m
                       * np.sqrt(factorial(m) / factorial(n))
                       * (2 * alpha)**k * expfac * Lmk)
                W += (2 / np.pi) * np.real(c * pre)
            else:
                k   = m - n
                Lnk = genlaguerre(n, k)(4 * a2)
                pre = ((-1)**n
                       * np.sqrt(factorial(n) / factorial(m))
                       * (2 * np.conj(alpha))**k * expfac * Lnk)
                W += (2 / np.pi) * np.real(c * pre)
    return W


def compute_wigner_sf(state, mode, xvec):
    """Compute Wigner via SF built-in, fall back to manual DM method."""
    try:
        W = state.wigner(mode, xvec, xvec)
        return np.array(W, dtype=float)
    except Exception:
        pass
    try:
        dm = np.array(state.reduced_dm(modes=[mode]))
        return wigner_from_dm_fast(dm, xvec)
    except Exception as e:
        print(f"    Warning: Wigner failed ({e}), returning zeros")
        return np.zeros((len(xvec), len(xvec)))


def negativity_volume(W, xvec):
    """Wigner negativity ∫|W|_{W<0} dx dp (normalised units)."""
    dx  = xvec[1] - xvec[0]
    neg = float(np.sum(np.abs(W[W < 0])) * dx**2)
    return neg


def run_probe_fock(params, cutoff):
    """Run probe sub-circuit (prep + BS1 only) on Fock backend."""
    r_v  = float(params.r_clipped)
    a_v  = float(params.alpha)
    dc_v = float(params.d_coh)
    ds_v = float(params.d_sq)
    t1_v = float(params.theta1)
    p1_v = float(params.phi1)
    prog = sf.Program(2)
    with prog.context as q:
        Coherent(a_v, 0.0) | q[0]
        Squeezed(r_v, 0.0) | q[1]
        Rgate(dc_v)         | q[0]
        Rgate(ds_v)         | q[1]
        BSgate(t1_v, p1_v)  | (q[0], q[1])
    eng   = sf.Engine("fock", backend_options={"cutoff_dim": cutoff})
    return eng.run(prog).state


def run_input_fock(params, cutoff):
    """Run input state only (prep, no BS) on Fock backend."""
    r_v  = float(params.r_clipped)
    a_v  = float(params.alpha)
    dc_v = float(params.d_coh)
    ds_v = float(params.d_sq)
    prog = sf.Program(2)
    with prog.context as q:
        Coherent(a_v, 0.0) | q[0]
        Squeezed(r_v, 0.0) | q[1]
        Rgate(dc_v)         | q[0]
        Rgate(ds_v)         | q[1]
    eng   = sf.Engine("fock", backend_options={"cutoff_dim": cutoff})
    return eng.run(prog).state


def compute_all_wigner(all_results, xvec=None, cutoff_override=None):
    """
    Compute Wigner functions for Afek and optimised probe states, all N.

    Returns dict:
        wigner_data[N] = {
            "xvec": array,
            "afek": {"W0": W, "W1": W, "neg0": float, "neg1": float,
                     "W0_in": W, "W1_in": W},
            "opt":  {same structure}
        }
    """
    if xvec is None:
        xvec = np.linspace(-5.5, 5.5, 150)

    data = {}
    for N in sorted(all_results.keys()):
        print(f"\n  N={N} Wigner computation...")
        co  = cutoff_override or max(3*N+4, 12)
        dat = {"xvec": xvec}

        for label, params in [
            ("afek", CircuitParams.from_afek(N=N)),
            ("opt",  all_results[N]["params_opt"]),
        ]:
            print(f"    {label}: probe...", end=" ", flush=True)
            state_probe = run_probe_fock(params, co)
            W0 = compute_wigner_sf(state_probe, 0, xvec)
            W1 = compute_wigner_sf(state_probe, 1, xvec)
            n0 = negativity_volume(W0, xvec)
            n1 = negativity_volume(W1, xvec)
            print(f"neg=({n0:.4f},{n1:.4f})", end="  input...", flush=True)

            state_in  = run_input_fock(params, co)
            W0i = compute_wigner_sf(state_in, 0, xvec)
            W1i = compute_wigner_sf(state_in, 1, xvec)
            print("done")

            dat[label] = {
                "W0": W0, "W1": W1,
                "neg0": n0, "neg1": n1,
                "W0_in": W0i, "W1_in": W1i,
                "r":     float(params.r_clipped),
                "gamma": float(params.gamma),
                "alpha": float(params.alpha),
            }
        data[N] = dat
    return data


# =============================================================================
# 2. SHARED RENDERING HELPERS
# =============================================================================

def render_wigner(ax, W, xvec, color_N=None, show_contour=True,
                  annotate_neg=True, neg_val=None, label_txt=None,
                  show_axes_labels=True, xlabel=True, ylabel=True,
                  vmax_override=None):
    """
    Render one Wigner function panel with Nature-quality styling.

    Features:
      - Symmetric diverging colormap
      - W=0 contour (non-classicality boundary)
      - Negativity annotation box
      - Vacuum circle reference
      - Phase-space origin crosshair (faint)
    """
    N_pts = len(xvec)
    dx    = xvec[1] - xvec[0]

    # Colour scale: symmetric about zero, clipped to 99th percentile
    vmax = vmax_override or float(np.percentile(np.abs(W), 99.5))
    vmax = max(vmax, 1e-6)
    norm  = TwoSlopeNorm(vmin=-vmax, vcenter=0.0, vmax=vmax)

    # Main Wigner surface
    im = ax.pcolormesh(
        xvec, xvec, W.T,
        cmap=WIGNER_CMAP, norm=norm,
        rasterized=True, shading="auto",
        zorder=1
    )

    # Vacuum reference circle (radius = 1/√2, one shot-noise unit)
    theta_c = np.linspace(0, 2*np.pi, 200)
    ax.plot(np.cos(theta_c)/np.sqrt(2),
            np.sin(theta_c)/np.sqrt(2),
            color="white", lw=0.4, ls=":",
            alpha=0.5, zorder=3)

    # Phase-space origin crosshair
    ax.axhline(0, color="white", lw=0.3, alpha=0.3, zorder=2)
    ax.axvline(0, color="white", lw=0.3, alpha=0.3, zorder=2)

    # W=0 contour — the non-classicality boundary
    if show_contour and W.min() < 0:
        try:
            cs = ax.contour(xvec, xvec, W.T,
                            levels=[0.0],
                            colors=["white"],
                            linewidths=[0.7],
                            linestyles=["-"],
                            zorder=4)
        except Exception:
            pass

    # Negativity annotation
    if annotate_neg and neg_val is not None:
        neg_color = "#082567" if neg_val > 0.005 else "#444444"
        ax.text(0.04, 0.96,
                f"$\\mathcal{{N}}={neg_val:.3f}$",
                transform=ax.transAxes, fontsize=5.5,
                va="top", ha="left", color="white", zorder=5,
                bbox=dict(facecolor=neg_color, alpha=0.80,
                          edgecolor="none", pad=1.5,
                          boxstyle="round,pad=0.2"))

    # Top-right label (Afek / Opt)
    if label_txt:
        ax.text(0.97, 0.96, label_txt,
                transform=ax.transAxes, fontsize=5.5,
                va="top", ha="right", color="white", zorder=5,
                fontweight="bold",
                bbox=dict(facecolor="#00000066", alpha=0.7,
                          edgecolor="none", pad=1.2,
                          boxstyle="round,pad=0.2"))

    ax.set_xlim(xvec[0], xvec[-1])
    ax.set_ylim(xvec[0], xvec[-1])
    ax.set_aspect("equal")
    ax.tick_params(labelsize=5.5, length=2, width=0.5)

    if xlabel:
        ax.set_xlabel("$\\hat{x}$  (shot-noise units)", fontsize=6.5,
                      labelpad=2)
    else:
        ax.set_xticklabels([])

    if ylabel:
        ax.set_ylabel("$\\hat{p}$  (shot-noise units)", fontsize=6.5,
                      labelpad=2)
    else:
        ax.set_yticklabels([])

    # Remove spines for clean look
    for spine in ax.spines.values():
        spine.set_visible(False)

    return im, norm, vmax


def add_colorbar(fig, ax, im, norm, label="$W(x,p)$",
                 vmax=None, shrink=0.85, pad=0.02):
    """Add a compact, properly-labelled colorbar."""
    cb = fig.colorbar(im, ax=ax, fraction=0.046*shrink,
                      pad=pad, aspect=20)
    cb.ax.tick_params(labelsize=5, length=2, width=0.4)
    cb.set_label(label, fontsize=5.5, labelpad=2)
    if vmax is not None:
        cb.set_ticks([-vmax, 0, vmax])
        cb.set_ticklabels([f"$-{vmax:.2f}$", "0", f"${vmax:.2f}$"])
    cb.outline.set_linewidth(0.4)
    return cb


# =============================================================================
# FIGURE W1 — Main Wigner Gallery (the centrepiece figure)
# =============================================================================

def fig_wigner_gallery(wigner_data, save_path="figW1_gallery.pdf"):
    """
    Nature-quality Wigner function gallery.

    Layout (4 rows × 5 columns):
      Col 0: Row label
      Col 1: Mode 0 Afek (coherent branch, before mixing)
      Col 2: Mode 0 Opt
      Col 3: Mode 1 Afek (squeezed branch, before mixing)
      Col 4: Mode 1 Opt

    Each panel shows:
      - W(x,p) with symmetric blue–white–red diverging colourmap
      - W=0 contour in white (non-classicality boundary)
      - Vacuum-circle reference (dotted white)
      - Negativity 𝒩 annotation (upper-left)
      - Afek/Opt label (upper-right)

    The probe state is the two-mode state AFTER BS₁ but BEFORE phase
    encoding, so it captures the non-classical resource available for
    sensing.
    """
    N_list = sorted(wigner_data.keys())
    nrows  = len(N_list)

    fig = plt.figure(figsize=(DOUBLE, nrows * 1.72 + 0.55))
    outer_gs = gridspec.GridSpec(
        nrows, 4,
        figure=fig,
        hspace=0.08, wspace=0.06,
        left=0.08, right=0.95,
        top=0.93, bottom=0.05,
    )

    # Column headers
    col_heads = [
        "Mode 0 · Afek init.",
        "Mode 0 · Optimised",
        "Mode 1 · Afek init.",
        "Mode 1 · Optimised",
    ]
    for ci, ch in enumerate(col_heads):
        fig.text(0.115 + ci * 0.215, 0.955, ch,
                 ha="center", va="bottom", fontsize=7,
                 fontweight="bold",
                 color=AFEK_COLOR if "Afek" in ch else "#111111")

    # Vertical separator line between mode-0 and mode-1 groups
    fig.add_artist(mpl.lines.Line2D(
        [0.540, 0.540], [0.04, 0.94],
        transform=fig.transFigure,
        color="#cccccc", lw=0.6, ls="--"
    ))

    # Mode group labels
    fig.text(0.315, 0.972,
             "Mode 0  (coherent branch)",
             ha="center", va="bottom", fontsize=7.5,
             color="#333333", style="italic")
    fig.text(0.755, 0.972,
             "Mode 1  (squeezed branch)",
             ha="center", va="bottom", fontsize=7.5,
             color="#333333", style="italic")

    all_ims = []

    for row, N in enumerate(N_list):
        dat  = wigner_data[N]
        xvec = dat["xvec"]
        col  = NC[N]

        # Shared vmax across all 4 panels of this row for consistent colour
        all_W = [dat["afek"]["W0"], dat["opt"]["W0"],
                 dat["afek"]["W1"], dat["opt"]["W1"]]
        vmax_row = max(float(np.percentile(np.abs(W), 99.5))
                       for W in all_W)

        configs = [
            (dat["afek"]["W0"], dat["afek"]["neg0"], "Afek",  False),
            (dat["opt"] ["W0"], dat["opt"] ["neg0"], "Opt. ★", True),
            (dat["afek"]["W1"], dat["afek"]["neg1"], "Afek",  False),
            (dat["opt"] ["W1"], dat["opt"] ["neg1"], "Opt. ★", True),
        ]

        for col_idx, (W, neg, lbl, is_opt) in enumerate(configs):
            ax = fig.add_subplot(outer_gs[row, col_idx])

            label_color = col if is_opt else AFEK_COLOR
            im, norm, vmax = render_wigner(
                ax, W, xvec,
                neg_val=neg,
                label_txt=lbl,
                vmax_override=vmax_row,
                xlabel=(row == nrows - 1),
                ylabel=(col_idx == 0),
            )
            all_ims.append((ax, im, norm, vmax_row))

            # Improvement arrow between Afek→Opt pairs
            if col_idx in (1, 3):  # Opt panels
                prev_neg = (dat["afek"]["neg0"] if col_idx == 1
                            else dat["afek"]["neg1"])
                delta = neg - prev_neg
                if abs(delta) > 0.001:
                    sign  = "+" if delta > 0 else ""
                    dcolor = "#00aa44" if delta > 0 else "#cc3333"
                    ax.text(0.97, 0.04,
                            f"$\\Delta\\mathcal{{N}}{sign}{delta:.3f}$",
                            transform=ax.transAxes, fontsize=5,
                            va="bottom", ha="right", color=dcolor,
                            zorder=5,
                            bbox=dict(facecolor="white", alpha=0.65,
                                      edgecolor="none", pad=1.0,
                                      boxstyle="round,pad=0.15"))

        # Row label: N and colour band
        ax0 = fig.add_subplot(outer_gs[row, 0])
        fig.text(0.005, (1 - (row + 0.5)/nrows) * 0.88 + 0.06,
                 f"$N={N}$",
                 va="center", ha="left",
                 fontsize=8, fontweight="bold", color=col,
                 rotation=90)

    # One shared colorbar on the right
    sm = plt.cm.ScalarMappable(
        cmap=WIGNER_CMAP,
        norm=TwoSlopeNorm(vmin=-0.32, vcenter=0, vmax=0.32)
    )
    sm.set_array([])
    cbar_ax = fig.add_axes([0.96, 0.10, 0.010, 0.75])
    cb = fig.colorbar(sm, cax=cbar_ax)
    cb.set_label("$W(x,p)$", fontsize=6, labelpad=3)
    cb.ax.tick_params(labelsize=5.5, length=2)
    cb.set_ticks([-0.30, -0.15, 0, 0.15, 0.30])
    cb.outline.set_linewidth(0.4)

    # Non-classicality annotation on colorbar
    cbar_ax.text(2.5, 0.05, "classical\n($W>0$)",
                 transform=cbar_ax.transAxes,
                 fontsize=4.5, ha="left", va="bottom", color="#cc3300")
    cbar_ax.text(2.5, 0.95, "non-classical\n($W<0$)",
                 transform=cbar_ax.transAxes,
                 fontsize=4.5, ha="left", va="top", color="#0033aa")

    fig.savefig(save_path)
    fig.savefig(save_path.replace(".pdf", ".png"), dpi=300)
    print(f"  Saved → {save_path}")
    return fig


# =============================================================================
# FIGURE W2 — Negativity Volume: Quantification
# =============================================================================

def fig_wigner_negativity(wigner_data, all_results,
                           save_path="figW2_negativity.pdf"):
    """
    Three-panel quantification figure:
      a) Negativity 𝒩 for mode 0 (coherent branch)
      b) Negativity 𝒩 for mode 1 (squeezed branch)
      c) Percentage change in 𝒩 upon optimisation

    Shows that optimisation systematically increases Wigner negativity
    at N≥3, confirming genuine enhancement of non-classicality.
    """
    N_list = sorted(wigner_data.keys())
    x      = np.arange(len(N_list))
    w      = 0.32
    colors = [NC[N] for N in N_list]

    fig, axes = plt.subplots(1, 3,
                              figsize=(DOUBLE, DOUBLE/2.4),
                              gridspec_kw={"wspace": 0.38})

    for ax_i, (ax, mode, mode_desc) in enumerate(zip(
        axes[:2],
        [0, 1],
        ["Mode 0  (coherent branch, after BS$_1$)",
         "Mode 1  (squeezed branch, after BS$_1$)"],
    )):
        afek_n = [wigner_data[N]["afek"][f"neg{mode}"] for N in N_list]
        opt_n  = [wigner_data[N]["opt"] [f"neg{mode}"] for N in N_list]

        # Bars
        b1 = ax.bar(x - w/2, afek_n, w,
                    color=[mpl.colors.to_rgba(AFEK_COLOR, 0.55)] * len(x),
                    edgecolor=AFEK_COLOR, linewidth=0.6,
                    label="Afek init.", zorder=3)
        b2 = ax.bar(x + w/2, opt_n, w,
                    color=colors, edgecolor="white",
                    linewidth=0.4, label="Optimised", zorder=3,
                    alpha=0.9)

        # Value labels on top of optimised bars
        for bar, val in zip(b2, opt_n):
            if val > 0.003:
                ax.text(bar.get_x() + bar.get_width()/2,
                        val + max(opt_n) * 0.04,
                        f"{val:.3f}",
                        ha="center", va="bottom", fontsize=5.5,
                        fontweight="bold",
                        color=bar.get_facecolor())

        # Connecting lines Afek→Opt showing direction of change
        for i, (av, ov) in enumerate(zip(afek_n, opt_n)):
            col  = "#22aa44" if ov >= av else "#cc3322"
            ax.annotate("",
                xy=(i + w/2, ov), xytext=(i - w/2, av),
                arrowprops=dict(
                    arrowstyle="-|>",
                    color=col, lw=0.7,
                    mutation_scale=6,
                    connectionstyle="arc3,rad=-0.15",
                ), zorder=4)

        ax.set_xticks(x)
        ax.set_xticklabels([f"$N={N}$" for N in N_list], fontsize=6.5)
        ax.set_ylabel("Wigner negativity  $\\mathcal{N}$", fontsize=7)
        ax.set_title(mode_desc, fontsize=7, pad=5)
        ax.set_ylim(0, max(max(opt_n), 0.01) * 1.40)
        ax.legend(frameon=False, fontsize=6, handlelength=1.2,
                  loc="upper left", handletextpad=0.4)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        panel_tag(ax, "ab"[ax_i], x=-0.20)

        # Classical boundary
        ax.axhline(0, color="#888888", lw=0.5, zorder=1)
        ax.text(x[-1]+0.55, 0.001,
                "classical $\\mathcal{N}=0$",
                fontsize=5, color="#888888", ha="right",
                style="italic")

    # Panel c: percentage change
    ax = axes[2]
    for mi, (mode, mcolor, mk) in enumerate(zip(
        [0, 1], ["#1f77b4", "#9467bd"], ["o", "s"]
    )):
        pcts = [
            (wigner_data[N]["opt"][f"neg{mode}"]
             - wigner_data[N]["afek"][f"neg{mode}"])
            / max(wigner_data[N]["afek"][f"neg{mode}"], 1e-6) * 100
            for N in N_list
        ]
        ax.plot(N_list, pcts, color=mcolor, marker=mk,
                markersize=5, lw=1.0, markeredgecolor="white",
                markeredgewidth=0.5, zorder=4,
                label=f"Mode {mode}")
        for N, pct in zip(N_list, pcts):
            sign = "+" if pct >= 0 else ""
            ax.text(N, pct + (5 if pct >= 0 else -8),
                    f"{sign}{pct:.0f}%",
                    ha="center", fontsize=5.5,
                    color="#22aa44" if pct >= 0 else "#cc3322",
                    fontweight="bold")

    ax.axhline(0, color="#888888", lw=0.6, ls="--")
    ax.fill_between(N_list, 0,
                    [max(0, p) for p in pcts],
                    alpha=0.07, color="#22aa44")
    ax.fill_between(N_list, 0,
                    [min(0, p) for p in pcts],
                    alpha=0.07, color="#cc3322")

    ax.set_xticks(N_list)
    ax.set_xticklabels([f"$N={N}$" for N in N_list], fontsize=6.5)
    ax.set_xlabel("Total photon number $N$", fontsize=7)
    ax.set_ylabel("$\\Delta\\mathcal{N}$ upon optimisation (\\%)",
                  fontsize=7)
    ax.set_title("Non-classicality change", fontsize=7, pad=5)
    ax.legend(frameon=False, fontsize=6, handlelength=1.2)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    panel_tag(ax, "c", x=-0.20)

    fig.savefig(save_path)
    fig.savefig(save_path.replace(".pdf", ".png"), dpi=300)
    print(f"  Saved → {save_path}")
    return fig


# =============================================================================
# FIGURE W3 — Phase-Space Portrait (N=2, high-detail single figure)
# =============================================================================

def fig_phase_portrait(wigner_data, all_results, N=2,
                        save_path="figW3_portrait.pdf"):
    """
    Four-panel detailed phase-space portrait for one N:

      a) Input mode 0 (coherent state |α⟩ after R(d_coh)):
         Gaussian blob displaced in phase space. Afek vs Opt contours overlaid.

      b) Input mode 1 (squeezed vacuum |r⟩ after R(d_sq)):
         Squeezed ellipse. Optimisation rotates and stretches the ellipse.

      c) Probe mode 0 (after BS₁, entangled):
         Non-Gaussian, shows interference fringes and W<0 regions.

      d) Probe mode 1 (after BS₁, entangled):
         Complementary mode of the entangled state.

    Afek state shown as filled heatmap. Optimised shown as overlaid contours.
    """
    dat  = wigner_data[N]
    xvec = dat["xvec"]
    col  = NC[N]

    fig, axes = plt.subplots(
        2, 2, figsize=(DOUBLE, DOUBLE * 0.9),
        gridspec_kw={"hspace": 0.12, "wspace": 0.10}
    )

    panels = [
        (axes[0,0], dat["afek"]["W0_in"], dat["opt"]["W0_in"],
         dat["afek"]["neg0"], dat["opt"]["neg0"],
         "Input: Mode 0\nCoherent state $|\\alpha\\rangle$", "a"),
        (axes[0,1], dat["afek"]["W1_in"], dat["opt"]["W1_in"],
         dat["afek"]["neg1"], dat["opt"]["neg1"],
         "Input: Mode 1\nSqueezed vacuum $|r\\rangle$", "b"),
        (axes[1,0], dat["afek"]["W0"], dat["opt"]["W0"],
         dat["afek"]["neg0"], dat["opt"]["neg0"],
         "Probe: Mode 0\nAfter BS$_1$ (entangled)", "c"),
        (axes[1,1], dat["afek"]["W1"], dat["opt"]["W1"],
         dat["afek"]["neg1"], dat["opt"]["neg1"],
         "Probe: Mode 1\nAfter BS$_1$ (entangled)", "d"),
    ]

    for ax, W_a, W_o, neg_a, neg_o, title, ptag in panels:
        vmax = max(float(np.percentile(np.abs(W_a), 99.5)),
                   float(np.percentile(np.abs(W_o), 99.5)))

        # Background: Afek state (filled heatmap)
        im, norm, _ = render_wigner(ax, W_a, xvec,
                                     neg_val=None,
                                     label_txt=None,
                                     vmax_override=vmax,
                                     xlabel=("1," in ptag or ptag in ("c","d")),
                                     ylabel=(ptag in ("a","c")))

        # Afek contours (dashed white)
        levels = np.linspace(-vmax*0.8, vmax*0.8, 9)
        try:
            ax.contour(xvec, xvec, W_a.T, levels=levels,
                       colors=["white"], linewidths=[0.4],
                       linestyles=["--"], alpha=0.55, zorder=3)
        except Exception:
            pass

        # Optimised contours (solid, coloured)
        try:
            ax.contour(xvec, xvec, W_o.T, levels=levels,
                       colors=[col], linewidths=[0.65],
                       linestyles=["-"], alpha=0.85, zorder=4)
        except Exception:
            pass

        # W=0 boundary for optimised state
        if W_o.min() < 0:
            try:
                ax.contour(xvec, xvec, W_o.T, levels=[0],
                           colors=[col], linewidths=[1.0],
                           linestyles=["-"], alpha=1.0, zorder=5)
            except Exception:
                pass

        # Negativity annotation boxes
        ax.text(0.04, 0.97,
                f"Afek: $\\mathcal{{N}}={neg_a:.3f}$",
                transform=ax.transAxes, fontsize=5.5,
                va="top", ha="left", color="white", zorder=6,
                bbox=dict(facecolor=AFEK_COLOR, alpha=0.75,
                          edgecolor="none", pad=1.5,
                          boxstyle="round,pad=0.2"))
        ax.text(0.04, 0.84,
                f"Opt.: $\\mathcal{{N}}={neg_o:.3f}$",
                transform=ax.transAxes, fontsize=5.5,
                va="top", ha="left", color="white", zorder=6,
                bbox=dict(facecolor=col, alpha=0.85,
                          edgecolor="none", pad=1.5,
                          boxstyle="round,pad=0.2"))

        ax.set_title(title, fontsize=7, pad=4, color="#222222")
        panel_tag(ax, ptag, x=-0.14)

    # Legend
    handles = [
        Line2D([0],[0], color="white", lw=0.6, ls="--",
               label="Afek contours"),
        Line2D([0],[0], color=col, lw=0.8, ls="-",
               label="Optimised contours"),
        Line2D([0],[0], color=col, lw=1.2, ls="-",
               label="$W=0$ boundary (Opt.)"),
    ]
    axes[1,1].legend(handles=handles, frameon=True,
                      fontsize=6, loc="lower right",
                      facecolor="white", framealpha=0.8,
                      edgecolor="#cccccc", handlelength=1.8)

    # Shared colorbar
    fig.subplots_adjust(right=0.88)
    cbar_ax = fig.add_axes([0.90, 0.10, 0.012, 0.78])
    sm = plt.cm.ScalarMappable(
        cmap=WIGNER_CMAP,
        norm=TwoSlopeNorm(vmin=-0.32, vcenter=0, vmax=0.32)
    )
    sm.set_array([])
    cb = fig.colorbar(sm, cax=cbar_ax)
    cb.set_label("$W(x,p)$ [a.u.]", fontsize=6, labelpad=3)
    cb.ax.tick_params(labelsize=5.5, length=2)
    cb.outline.set_linewidth(0.4)

    fig.suptitle(
        f"Phase-space portraits of quantum states  ($N={N}$, probe state)",
        fontsize=8, y=0.995, fontweight="bold", color="#222222"
    )

    fig.savefig(save_path)
    fig.savefig(save_path.replace(".pdf", ".png"), dpi=300)
    print(f"  Saved → {save_path}")
    return fig


# =============================================================================
# FIGURE W4 — State Evolution Through Circuit
# =============================================================================

def fig_state_evolution(wigner_data, all_results, N=2,
                         save_path="figW4_evolution.pdf"):
    """
    State evolution figure — fixed version.

    Fixes vs previous:
      1. No title/header overlap — suptitle removed, col headers placed carefully
      2. theta1 formatted as LaTeX: $\theta_1=\pi/4$
      3. Row gap eliminated — tight hspace
      4. Row labels outside axes, no overlap
      5. Ghost artifact removed — separate axes for arrow col
      6. Per-row vmax so Afek and Opt use their own scale (physics visible)
    """
    dat  = wigner_data[N]
    xvec = dat["xvec"]
    col  = NC[N]

    # ── figure layout: 2 rows × 5 cols (col 1 = BS arrow) ────────────────────
    fig = plt.figure(figsize=(DOUBLE, DOUBLE * 0.68))
    gs  = gridspec.GridSpec(
        2, 5, figure=fig,
        hspace=0.18,    # tight vertical gap
        wspace=0.07,
        left=0.11,      # left margin for row labels
        right=0.91,
        top=0.87,
        bottom=0.10,
        width_ratios=[1, 0.45, 1, 1, 1],  # narrow BS column
    )

    row_info = [
        ("afek", "Afek init.",  AFEK_COLOR),
        ("opt",  "Optimised ★", col),
    ]

    # ── column headers (placed once, above top row) ───────────────────────────
    col_header_x = [0.155, None, 0.445, 0.620, 0.795]
    col_header_labels = [
        "Mode 0  (coherent)",
        None,
        "Mode 1  (squeezed)",
        "Probe M0  (after BS1)",
        "Probe M1  (after BS1)",
    ]
    for cx, cl in zip(col_header_x, col_header_labels):
        if cl:
            fig.text(cx, 0.905, cl, ha="center", va="bottom",
                     fontsize=6.5, fontweight="bold", color="#333333")

    # ── input vs probe separator ──────────────────────────────────────────────
    fig.add_artist(mpl.lines.Line2D(
        [0.365, 0.365], [0.08, 0.89],
        transform=fig.transFigure,
        color="#cccccc", lw=0.7, ls="--"
    ))
    fig.text(0.185, 0.863, "Input states", ###################################corrections#########################################################
             ha="center", va="bottom", fontsize=6,
             color="#888888", style="italic")
    fig.text(0.620, 0.863, "Probe state after BS$_1$",
             ha="center", va="bottom", fontsize=6,
             color="#888888", style="italic")

    # ── data columns per row ──────────────────────────────────────────────────
    data_cols = [
        (0, "W0_in", "neg0"),   # gridspec col, W-key, neg-key
        (2, "W1_in", "neg1"),
        (3, "W0",    "neg0"),
        (4, "W1",    "neg1"),
    ]

    for row, (label, row_lbl, row_col) in enumerate(row_info):

        # Row label — in left margin, no overlap with axes
        fig.text(
            0.005,
            0.78 - row * 0.44,
            row_lbl,
            va="center", ha="left",
            fontsize=7, fontweight="bold",
            color=row_col, rotation=90,
            multialignment="center"
        )

        # Per-row vmax so each row's colour range is meaningful
        W_row = [dat[label][wk] for _, wk, _ in data_cols]
        vmax_row = max(float(np.percentile(np.abs(W), 99.5)) for W in W_row)
        vmax_row = max(vmax_row, 0.03)

        for gs_col, wkey, nkey in data_cols:
            ax  = fig.add_subplot(gs[row, gs_col])
            W   = dat[label][wkey]
            neg = dat[label][nkey]

            norm = TwoSlopeNorm(vmin=-vmax_row, vcenter=0.0, vmax=vmax_row)
            ax.pcolormesh(xvec, xvec, W,
                          cmap=WIGNER_CMAP, norm=norm,
                          rasterized=True, shading="auto", zorder=1)

            # Vacuum reference circle
            theta_c = np.linspace(0, 2*np.pi, 200)
            ax.plot(np.cos(theta_c)/np.sqrt(2),
                    np.sin(theta_c)/np.sqrt(2),
                    color="white", lw=0.35, ls=":", alpha=0.55, zorder=3)

            # W=0 contour
            if W.min() < -0.005:
                try:
                    ax.contour(xvec, xvec, W, levels=[0.0],
                               colors=["white"], linewidths=[0.6],
                               linestyles=["-"], zorder=4)
                except Exception:
                    pass

            # Negativity annotation
            neg_bg = "#08306b" if neg > 0.002 else "#444444"
            ax.text(0.04, 0.96,
                    f"$\mathcal{{N}}={neg:.3f}$",
                    transform=ax.transAxes, fontsize=5.5,
                    va="top", ha="left", color="white", zorder=5,
                    bbox=dict(facecolor=neg_bg, alpha=0.82,
                              edgecolor="none", pad=1.5,
                              boxstyle="round,pad=0.2"))

            # Axes
            ax.set_xlim(xvec[0], xvec[-1])
            ax.set_ylim(xvec[0], xvec[-1])
            ax.set_aspect("equal")
            ax.set_xticks([-4, -2, 0, 2, 4])
            ax.set_yticks([-4, -2, 0, 2, 4])
            for sp in ax.spines.values():
                sp.set_visible(False)
            ax.tick_params(length=2, width=0.4, labelsize=5)

            if row == 1:
                ax.set_xlabel(r"$\hat{x}$  (shot-noise units)",
                              fontsize=6, labelpad=1)
            else:
                ax.set_xticklabels([])

            if gs_col == 0:
                ax.set_ylabel(r"$\hat{p}$  (shot-noise units)",
                              fontsize=6, labelpad=1)
            else:
                ax.set_yticklabels([])

        # ── BS₁ arrow panel (col 1) ───────────────────────────────────────────
        ax_bs = fig.add_subplot(gs[row, 1])
        ax_bs.axis("off")
        ax_bs.set_facecolor("white")  # explicit white bg → no ghost artifact

        # Formatted theta1
        p_cur  = (all_results[N]["params_opt"] if label == "opt"
                  else CircuitParams.from_afek(N=N))
        t1_val = float(p_cur.theta1)
        # Express as fraction of pi if close
        t1_str = (rf"$\theta_1 = \pi/4$"
          if abs(t1_val - np.pi/4) < 0.01
          else rf"$\theta_1 = {t1_val:.3f}$")

        ax_bs.text(0.50, 0.60,
                   "BS$_1$",
                   ha="center", va="center",
                   fontsize=8, color="#333333",
                   fontweight="bold",
                   transform=ax_bs.transAxes)
        ax_bs.annotate("",
                        xy=(0.88, 0.50), xytext=(0.12, 0.50),
                        xycoords="axes fraction",
                        arrowprops=dict(
                            arrowstyle="-|>",
                            color="#333333",
                            lw=1.2,
                            mutation_scale=10,
                        ))
        ax_bs.text(0.50, 0.28,
                   t1_str,
                   ha="center", va="center",
                   fontsize=5.5, color=row_col,
                   transform=ax_bs.transAxes)

    # ── shared colorbar ───────────────────────────────────────────────────────
    # Use full data range for display
    all_W = [dat[lbl][wk]
             for lbl in ["afek","opt"]
             for _, wk, _ in data_cols]
    vmax_cb = max(float(np.percentile(np.abs(W), 99.5)) for W in all_W)

    sm = plt.cm.ScalarMappable(
        cmap=WIGNER_CMAP,
        norm=TwoSlopeNorm(vmin=-vmax_cb, vcenter=0, vmax=vmax_cb)
    )
    sm.set_array([])
    cbar_ax = fig.add_axes([0.925, 0.10, 0.012, 0.74])
    cb = fig.colorbar(sm, cax=cbar_ax)
    cb.set_label("$W(x,p)$", fontsize=6, labelpad=3)
    cb.ax.tick_params(labelsize=5.5, length=2, width=0.4)
    cb.set_ticks([-vmax_cb, 0, vmax_cb])
    cb.set_ticklabels([f"$-{vmax_cb:.2f}$", "0", f"${vmax_cb:.2f}$"])
    cb.outline.set_linewidth(0.4)

    # Colorbar annotations — correct placement
    cbar_ax.text(2.6, 0.97, "non-classical ($W<0$)",
                 transform=cbar_ax.transAxes,
                 fontsize=4.5, ha="left", va="top",
                 color="#08306b")
    cbar_ax.text(2.6, 0.03, "classical ($W>0$)",
                 transform=cbar_ax.transAxes,
                 fontsize=4.5, ha="left", va="bottom",
                 color="#b10026")

    fig.savefig(save_path, facecolor="white", dpi=300)
    fig.savefig(save_path.replace(".pdf", ".png"), dpi=300,
                facecolor="white")
    print(f"  Saved -> {save_path}")
    return fig



# =============================================================================
# MASTER CALL
# =============================================================================

def generate_all_wigner_figures(wigner_data, all_results):
    """Generate all four Wigner figures."""
    print("Generating Wigner figures (Nature-quality)...")
    figs = {}
    figs["gallery"]    = fig_wigner_gallery(wigner_data)
    figs["negativity"] = fig_wigner_negativity(wigner_data, all_results)
    figs["portrait"]   = fig_phase_portrait(wigner_data, all_results, N=2)
    figs["evolution"]  = fig_state_evolution(wigner_data, all_results, N=2)
    print("\nAll Wigner figures saved.")
    return figs



def print_captions():
    """Print all figure captions ready to paste into LaTeX."""
    print("\n" + "="*70)
    print("FIGURE CAPTIONS  (paste directly into paper_full.tex)")
    print("="*70)
    for fig_key, caption in CAPTIONS.items():
        print(f"\n% ── {fig_key} ──")
        print(r"\caption{" + caption.strip() + "}")
        print()

In [ ]:
fig = fig_state_evolution(wigner_data, all_results, N=2)

###################QFI-Calculations########################

In [ ]:
"""
Author : Simanshu Kumar
QFI calculation.
"""

import numpy as np

# ── Step 1: patch run_probe_circuit (unchanged, just redefine for safety) ─────
def run_probe_circuit(params, cutoff):
    """Run state-prep + BS₁ only (NO phase encoding, NO BS₂)."""
    r_v  = float(params.r_clipped)
    a_v  = float(params.alpha)
    dc_v = float(params.d_coh)
    ds_v = float(params.d_sq)
    t1_v = float(params.theta1)
    p1_v = float(params.phi1)

    prog = sf.Program(2)
    with prog.context as q:
        Coherent(a_v, 0.0)          | q[0]
        Squeezed(r_v, 0.0)          | q[1]
        Rgate(dc_v)                 | q[0]
        Rgate(ds_v)                 | q[1]
        BSgate(t1_v, p1_v)          | (q[0], q[1])

    eng   = sf.Engine("fock", backend_options={"cutoff_dim": cutoff})
    state = eng.run(prog).state

    try:
        ket = state.ket()
        if ket is not None:
            return np.array(ket, dtype=complex)
    except Exception:
        pass

    # fallback: dominant eigenvector of density matrix
    try:
        dm = np.array(state.dm(), dtype=complex)
        if dm.ndim == 4:
            dm = dm.reshape(cutoff*cutoff, cutoff*cutoff)
        _, vecs = np.linalg.eigh(dm)
        psi = vecs[:, -1].reshape(cutoff, cutoff)
        idx = np.unravel_index(np.argmax(np.abs(psi)), psi.shape)
        psi *= np.exp(-1j * np.angle(psi[idx]))
        return psi
    except Exception as e:
        print(f"  probe extraction failed: {e}")
        return None


# ── Step 2: FIXED qfi_variance — post-selected N-photon subspace ──────────────
def qfi_variance(params, cutoff):
    """
    QFI = 4·Var(n̂₀) computed in the POST-SELECTED N-photon subspace.

    Projects |ψ⟩ onto {|n₀,n₁⟩ : n₀+n₁=N}, renormalises, then
    computes the variance of n̂₀ on that projected state.

    Guaranteed: F_Q ≤ N².  For ideal NOON: F_Q = N² exactly.

    Returns
    -------
    F_Q           : float   QFI in N-photon sector  (≤ N²)
    n0_mean       : float   ⟨n̂₀⟩ in sector
    n0_var        : float   Var(n̂₀) in sector
    post_sel_prob : float   P(n₀+n₁=N)  [normalisation weight]
    """
    N      = params.N
    ket    = run_probe_circuit(params, cutoff)
    if ket is None:
        return float("nan"), float("nan"), float("nan"), float("nan")

    # Project onto N-photon sector
    proj = np.zeros_like(ket)
    for n0 in range(N + 1):
        n1 = N - n0
        if 0 <= n1 < cutoff:
            proj[n0, n1] = ket[n0, n1]

    post_sel_prob = float(np.sum(np.abs(proj)**2))

    if post_sel_prob < 1e-10:
        return 0.0, 0.0, 0.0, post_sel_prob

    proj /= np.sqrt(post_sel_prob)   # renormalise

    # P(n₀) in projected state — only n₀=0..N are non-zero
    p_n0  = np.array([float(abs(proj[n0, N - n0])**2)
                      for n0 in range(N + 1)])
    n0_v  = np.arange(N + 1, dtype=float)

    n0_mean = float(np.dot(n0_v, p_n0))
    n0_sq   = float(np.dot(n0_v**2, p_n0))
    n0_var  = n0_sq - n0_mean**2
    F_Q     = 4.0 * n0_var

    return F_Q, n0_mean, n0_var, post_sel_prob


# ── Step 3: FIXED cfi_fringe — regularised to match Eq.7 in paper ─────────────
def cfi_fringe(params, N1, N2, cutoff, n_phi=400):
    """Peak CFI using eps=0.05*mean(P) regularisation (same as training)."""
    phi = np.linspace(0, 2*np.pi, n_phi, endpoint=False)
    P   = np.zeros(n_phi)
    for i, ph in enumerate(phi):
        probs, _ = run_circuit(
            params, tf.constant(ph, dtype=tf.float64), cutoff)
        P[i] = float(probs[N1, N2])
    dphi = phi[1] - phi[0]
    dP   = np.gradient(P, dphi)
    eps  = max(0.05 * float(np.mean(P)), 1e-8)   # regularisation
    cfi  = dP**2 / (P + eps)
    return {"cfi_peak": float(np.max(cfi)),
            "rate": float(P.max()),
            "prob": P, "cfi": cfi, "phi": phi}


# ── Step 4: total_cfi — sum over all patterns ─────────────────────────────────
def total_cfi(params, N, cutoff, n_phi=400):
    """Total CFI summed over all coincidence patterns."""
    phi   = np.linspace(0, 2*np.pi, n_phi, endpoint=False)
    dphi  = phi[1] - phi[0]
    total = np.zeros(n_phi)
    pats  = {}
    for N1, N2 in PATTERNS[N]:
        pat = f"|{N1},{N2}>"
        P   = np.zeros(n_phi)
        for i, ph in enumerate(phi):
            probs, _ = run_circuit(
                params, tf.constant(ph, dtype=tf.float64), cutoff)
            P[i] = float(probs[N1, N2])
        dP  = np.gradient(P, dphi)
        eps = max(0.05 * float(np.mean(P)), 1e-8)
        cfi = dP**2 / (P + eps)
        total += cfi
        pats[pat] = {"cfi_peak": float(np.max(cfi)),
                     "rate": float(P.max()), "prob": P, "cfi": cfi}
    return float(np.max(total)), pats, phi


# ── Step 5: quick_qfi_n2 ──────────────────────────────────────────────────────
def quick_qfi_n2(all_results, verbose=True):
    """Fast N=2 QFI check (~30–60 sec). Expected: QFI ≤ 4, η_total 0.2–0.5."""
    N  = 2
    co = 12
    print(f"N=2 QFI check  |  Heisenberg limit N² = {N**2}\n")
    res = {}
    for label, params in [("afek", CircuitParams.from_afek(N=N)),
                           ("opt",  all_results[N]["params_opt"])]:
        F_Q, n0_mean, n0_var, ps_prob = qfi_variance(params, co)
        tcfi, per_pat, _              = total_cfi(params, N, co, n_phi=200)
        eta_total = tcfi / max(F_Q, 1e-10)
        print(f"  N=2 {label.upper()}:")
        print(f"    QFI = 4·Var(n̂₀)|_N = 4·{n0_var:.4f} = {F_Q:.4f}")
        print(f"    QFI / N²            = {F_Q/N**2:.4f}  (≤1.0 guaranteed)")
        print(f"    Post-sel prob       = {ps_prob:.4f}  [P(n₀+n₁={N})]")
        print(f"    Total CFI           = {tcfi:.4f}")
        print(f"    η (total)           = {eta_total:.4f}")
        for pat, d in per_pat.items():
            ep = d["cfi_peak"] / max(F_Q, 1e-10)
            print(f"    {pat}: CFI={d['cfi_peak']:.4f}  "
                  f"rate={d['rate']:.4f}  η_pat={ep:.4f}")
        print()
        res[label] = {"qfi": F_Q, "qfi_fraction": F_Q/N**2,
                      "total_cfi": tcfi, "total_eta": eta_total,
                      "post_sel_prob": ps_prob}
    return res


# ── Quick self-validation ──────────────────────────────────────────────────────
def _validate():
    import numpy as np
    co = 12
    print("Self-validation (NOON states):")
    for N in [2, 3, 4]:
        ket  = np.zeros((co, co), dtype=complex)
        ket[N, 0] = 1/np.sqrt(2)
        ket[0, N] = 1/np.sqrt(2)
        proj = np.zeros_like(ket)
        for n0 in range(N+1):
            proj[n0, N-n0] = ket[n0, N-n0]
        proj /= np.linalg.norm(proj)
        p = np.array([abs(proj[n0,N-n0])**2 for n0 in range(N+1)])
        nv = np.arange(N+1, dtype=float)
        var = np.dot(nv**2, p) - np.dot(nv, p)**2
        qfi = 4*var
        ok  = "✓" if abs(qfi - N**2) < 1e-6 else "✗"
        print(f"  NOON N={N}: QFI={qfi:.4f}  N²={N**2}  {ok}")
    print("  |2,0⟩: QFI =", end=" ")
    ket2 = np.zeros((co,co), dtype=complex); ket2[2,0] = 1.0
    proj2 = np.zeros_like(ket2); proj2[2,0] = ket2[2,0]; proj2[0,2] = ket2[0,2]
    proj2 /= max(np.linalg.norm(proj2), 1e-10)
    p2 = np.array([abs(proj2[n0,2-n0])**2 for n0 in range(3)])
    nv2 = np.arange(3, dtype=float)
    print(f"{4*(np.dot(nv2**2,p2)-np.dot(nv2,p2)**2):.4f}  (expected 0.0)  "
          f"{'✓' if 4*(np.dot(nv2**2,p2)-np.dot(nv2,p2)**2)<1e-6 else '✗'}")

_validate()
print("\nPatch applied. Now run:  r = quick_qfi_n2(all_results)")


In [ ]:
r = quick_qfi_n2(all_results)

In [ ]:
# ============================================================
# Requires: noon_forward_model.py already executed, all_results defined
# ============================================================
import numpy as np

def run_probe_circuit(params, cutoff):
    """Run circuit up to BS1 only — returns 2-mode ket [cutoff, cutoff]."""
    r_v  = float(params.r_clipped)
    a_v  = float(params.alpha)
    dc_v = float(params.d_coh)
    ds_v = float(params.d_sq)
    t1_v = float(params.theta1)
    p1_v = float(params.phi1)

    prog = sf.Program(2)
    with prog.context as q:
        Coherent(a_v, 0.0)          | q[0]
        Squeezed(r_v, 0.0)          | q[1]
        Rgate(dc_v)                 | q[0]
        Rgate(ds_v)                 | q[1]
        BSgate(t1_v, p1_v)          | (q[0], q[1])
    eng   = sf.Engine("fock", backend_options={"cutoff_dim": cutoff})
    state = eng.run(prog).state
    try:
        ket = state.ket()
        if ket is not None:
            return np.array(ket, dtype=complex)
    except Exception:
        pass
    # fallback: dominant eigenvector of density matrix
    dm  = np.array(state.dm(), dtype=complex)
    if dm.ndim == 4:
        dm = dm.reshape(cutoff*cutoff, cutoff*cutoff)
    _, vecs = np.linalg.eigh(dm)
    psi = vecs[:, -1].reshape(cutoff, cutoff)
    idx = np.unravel_index(np.argmax(np.abs(psi)), psi.shape)
    psi *= np.exp(-1j * np.angle(psi[idx]))
    return psi


def qfi_variance(params, cutoff):
    """
    Exact QFI = 4*Var(n̂₀) on the probe state.
    Generator: n̂₀ (phase encoding on mode 0).
    Bounded by N²; equals N² for ideal NOON state.
    """
    ket   = run_probe_circuit(params, cutoff)
    probs = np.abs(ket)**2          # P(n0, n1)
    p_n0  = probs.sum(axis=1)       # marginal over mode 1
    n0    = np.arange(cutoff, dtype=float)
    mean  = float(np.sum(n0 * p_n0))
    sq    = float(np.sum(n0**2 * p_n0))
    var   = sq - mean**2
    # Project onto N-photon subspace for normalised QFI
    N     = params.N
    p_N   = float(sum(probs[k, N-k] for k in range(N+1)))  # post-sel prob
    p_n0_sub = np.array([probs[k, N-k] if k <= N else 0.0
                         for k in range(cutoff)])
    if p_N > 1e-10:
        p_n0_sub /= p_N
    mean_sub = float(np.sum(n0 * p_n0_sub))
    sq_sub   = float(np.sum(n0**2 * p_n0_sub))
    var_sub  = sq_sub - mean_sub**2
    return 4.0*var_sub, mean_sub, var_sub, p_N


def run_all_qfi(all_results, n_phi=400, verbose=True):
    results = {}
    for N in sorted(all_results.keys()):
        co = max(3*N+4, 12)
        results[N] = {}
        if verbose:
            print(f"\n{'='*50}\nN={N}  HL=N²={N**2}")
        for label, params in [("afek", CircuitParams.from_afek(N=N)),
                               ("opt",  all_results[N]["params_opt"])]:
            if verbose:
                print(f"  [{label.upper()}] QFI...", end=" ", flush=True)
            F_Q, n0_mean, n0_var, p_N = qfi_variance(params, co)
            if verbose:
                print(f"F_Q={F_Q:.4f}  F_Q/N²={F_Q/N**2:.4f}  P(post-sel)={p_N:.4f}")
            # CFI per pattern
            phi_arr = np.linspace(0, 2*np.pi, n_phi, endpoint=False)
            dphi    = phi_arr[1] - phi_arr[0]
            pat_data  = {}
            total_cfi = np.zeros(n_phi)
            for N1, N2 in PATTERNS[N]:
                pat = f"|{N1},{N2}>"
                P   = np.array([float(run_circuit(
                                    params,
                                    tf.constant(phi, dtype=tf.float64),
                                    co)[0][N1, N2])
                                for phi in phi_arr])
                dP       = np.gradient(P, dphi)
                cfi_vals = dP**2 / np.maximum(P, 1e-12)
                total_cfi += cfi_vals
                pat_data[pat] = {"cfi_peak": float(np.max(cfi_vals)),
                                 "rate":     float(P.max())}
                if verbose:
                    eta_p = pat_data[pat]["cfi_peak"] / max(F_Q, 1e-10)
                    print(f"    {pat}: CFI={pat_data[pat]['cfi_peak']:.4f}  "
                          f"rate={pat_data[pat]['rate']:.4f}  η={eta_p:.4f}")
            tcfi     = float(np.max(total_cfi))
            eta_tot  = tcfi / max(F_Q, 1e-10)
            if verbose:
                print(f"  Total CFI={tcfi:.4f}  η_total={eta_tot:.4f}")
            results[N][label] = {
                "qfi":          F_Q,
                "qfi_fraction": F_Q / N**2,
                "n0_var":       n0_var,
                "post_sel":     p_N,
                "total_cfi":    tcfi,
                "total_eta":    eta_tot,
                "cfi_patterns": pat_data,
                "efficiency":   {p: d["cfi_peak"]/max(F_Q,1e-10)
                                 for p, d in pat_data.items()},
            }
    return results


def print_qfi_table(qfi_results):
    print(f"\n{'N':>2} {'Label':>6} | {'QFI':>7} {'QFI/N²':>7} {'P_sel':>7} | "
          f"{'TotCFI':>7} {'η_tot':>7} | Pattern  CFI_pk   η_pat")
    print("─"*80)
    for N in sorted(qfi_results.keys()):
        for label in ["afek","opt"]:
            d = qfi_results[N][label]
            first = True
            for pat, pd in d["cfi_patterns"].items():
                eta_p = d["efficiency"][pat]
                if first:
                    print(f"  {N} {label:>6} | {d['qfi']:>7.3f} "
                          f"{d['qfi_fraction']:>7.3f} {d['post_sel']:>7.4f} | "
                          f"{d['total_cfi']:>7.3f} {d['total_eta']:>7.4f} | "
                          f"{pat}  {pd['cfi_peak']:6.4f}  {eta_p:.4f}")
                    first = False
                else:
                    print(f"  {' ':>8} | {' ':>7} {' ':>7} {' ':>7} | "
                          f"{' ':>7} {' ':>7} | "
                          f"{pat}  {pd['cfi_peak']:6.4f}  {eta_p:.4f}")
        print()

#-----For the Latex use guidans----------Will not print with figures
def print_latex_qfi_table(qfi_results):
    lines = [
        r"\begin{table}[t]",
        r"  \centering",
        r"  \caption{\textbf{QFI and measurement efficiency for all $N=2$--$5$.}",
        r"    $\mathcal{F}_Q = 4\,\mathrm{Var}(\hat{n}_0)$ on the post-BS$_1$ probe",
        r"    state (Eq.~\ref{eq:qfi_pure}); exact and bounded by $N^2$.",
        r"    $P_\mathrm{sel}$: post-selection probability $P(n_0+n_1=N)$.",
        r"    $\eta_\Sigma$: total measurement efficiency (all patterns combined).",
        r"    $\eta_\mathrm{pat}$: per-pattern efficiency.}",
        r"  \label{tab:qfi}",
        r"  \small\setlength{\tabcolsep}{3.5pt}",
        r"  \begin{tabular}{clccccccc}",
        r"    \toprule",
        r"    $N$ & Init. & $\mathcal{F}_Q$ & $\mathcal{F}_Q/N^2$"
        r" & $P_\mathrm{sel}$ & $F_\Sigma$ & $\eta_\Sigma$"
        r" & Pattern & $\eta_\mathrm{pat}$ \\",
        r"    \midrule",
    ]
    for N in sorted(qfi_results.keys()):
        for label, lbl in [("afek","Afek"),("opt","Opt.")]:
            d    = qfi_results[N][label]
            pats = list(d["cfi_patterns"].keys())
            for i, pat in enumerate(pats):
                eta_p = d["efficiency"][pat]
                col   = ("darkgreen" if eta_p >= 0.4 else
                         "darkorange" if eta_p >= 0.2 else "darkred")
                eta_c = ("darkgreen" if d["total_eta"] >= 0.4 else
                         "darkorange" if d["total_eta"] >= 0.2 else "darkred")
                if i == 0:
                    nr   = len(pats)
                    nN   = f"\\multirow{{{nr*2}}}{{*}}{{{N}}}" if label=="afek" and i==0 else ""
                    lines.append(
                        f"    {nN} & {lbl}"
                        f" & {d['qfi']:.3f} & {d['qfi_fraction']:.3f}"
                        f" & {d['post_sel']:.4f}"
                        f" & {d['total_cfi']:.3f}"
                        f" & \\textcolor{{{eta_c}}}{{{d['total_eta']:.3f}}}"
                        f" & ${pat}$"
                        f" & \\textcolor{{{col}}}{{{eta_p:.3f}}} \\\\")
                else:
                    lines.append(
                        f"     &  &  &  &  &  & "
                        f" & ${pat}$"
                        f" & \\textcolor{{{col}}}{{{eta_p:.3f}}} \\\\")
        lines.append(r"    \midrule")
    lines += [r"    \bottomrule", r"  \end{tabular}", r"\end{table}"]
    print("\n".join(lines))


def print_qfi_section(qfi_results):
    n2o  = qfi_results.get(2,{}).get("opt",{})
    qf2  = n2o.get("qfi_fraction", float("nan"))
    eta2 = n2o.get("total_eta",    float("nan"))
    all_qf  = [qfi_results[N]["opt"]["qfi_fraction"]
               for N in qfi_results if not np.isnan(qfi_results[N]["opt"]["qfi_fraction"])]
    all_eta = [qfi_results[N]["opt"]["total_eta"]
               for N in qfi_results if not np.isnan(qfi_results[N]["opt"]["total_eta"])]
    qf_r  = f"{min(all_qf):.2f}--{max(all_qf):.2f}" if all_qf else "?"
    eta_r = f"{min(all_eta):.2f}--{max(all_eta):.2f}" if all_eta else "?"
    print(f"""
KEY NUMBERS FOR §7.2:
  N=2 QFI/N² (Opt) = {qf2:.4f}  → probe reaches {qf2*100:.1f}% of Heisenberg limit
  N=2 η_total(Opt) = {eta2:.4f}  → {eta2*100:.0f}% measurement efficiency (2-pattern scheme)
  QFI/N² range (all N, Opt): {qf_r}
  η_total range  (all N, Opt): {eta_r}
""")

print("✓ QFI functions loaded: run_probe_circuit, qfi_variance,")
print("  run_all_qfi, print_qfi_table, print_latex_qfi_table, print_qfi_section")

In [ ]:
qfi_results = run_all_qfi(all_results)

In [ ]:
quick_qfi_n2(all_results)

In [ ]:
for N in [3, 4]:
    params_afek = CircuitParams.from_afek(N=N)
    params_opt  = all_results[N]["params_opt"]
    co = max(3*N+4, 12)
    for label, params in [("afek", params_afek), ("opt", params_opt)]:
        F_Q, n0_mean, n0_var, p_N = qfi_variance(params, co)
        print(f"N={N} {label}: QFI={F_Q:.4f}, QFI/N²={F_Q/N**2:.4f}, P_sel={p_N:.4f}")

####################QFI-Plot#####################

In [ ]:
"""
fig_qfi.py
==========
Panel (a): QFI/N² — probe quality vs Heisenberg limit, Afek vs Opt
Panel (b): η_Σ × P_sel — useful measurement events per pulse (log scale)

Output: fig_qfi.pdf  and  fig_qfi.png  (300 dpi)
"""

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.patches import Patch

# ── Style ─────────────────────────────────────────────────────────────────────
try:
    mpl.rcParams.update(NATURE_RC)
except NameError:
    mpl.rcParams.update({
        "font.family":        "sans-serif",
        "font.sans-serif":    ["Helvetica", "Arial", "DejaVu Sans"],
        "font.size":          7,
        "axes.labelsize":     7.5,
        "axes.titlesize":     8,
        "xtick.labelsize":    6.5,
        "ytick.labelsize":    6.5,
        "axes.linewidth":     0.6,
        "xtick.major.width":  0.6,
        "ytick.major.width":  0.6,
        "axes.spines.top":    False,
        "axes.spines.right":  False,
        "legend.fontsize":    6.5,
        "legend.frameon":     False,
        "figure.dpi":         150,
        "savefig.dpi":        300,
        "pdf.fonttype":       42,
        "ps.fonttype":        42,
    })

DOUBLE = 7.08   # inches — full two-column width
N_COLORS = {2: "#1f77b4", 3: "#e6550d", 4: "#31a354", 5: "#e31a1c"}
AFEK_COLOR  = "#aaaaaa"
AFEK_HATCH  = ""
OPT_ALPHA   = 0.88


# =============================================================================
# DATA  — populated from qfi_results dict OR hardcoded fallback
# =============================================================================

def extract_data(qfi_results):
    """Extract arrays from run_all_qfi() output dict."""
    N_list    = sorted(qfi_results.keys())
    qf_afek   = [qfi_results[N]["afek"]["qfi_fraction"]  for N in N_list]
    qf_opt    = [qfi_results[N]["opt"]["qfi_fraction"]   for N in N_list]
    etaP_afek = [qfi_results[N]["afek"]["total_eta"] *
                 qfi_results[N]["afek"]["post_sel"]       for N in N_list]
    etaP_opt  = [qfi_results[N]["opt"]["total_eta"]  *
                 qfi_results[N]["opt"]["post_sel"]        for N in N_list]
    return N_list, qf_afek, qf_opt, etaP_afek, etaP_opt


# Hardcoded values — use if qfi_results not available
HARDCODED = {
    "N_list":    [2,      3,      4,      5     ],
    "qf_afek":   [0.7932, 0.5841, 0.5259, 0.3597],
    "qf_opt":    [0.8169, 0.4859, 0.4916, 0.5759],
    # η_total × P_sel
    "etaP_afek": [0.0795*0.0700, 0.031*0.0188, 0.025*0.0162, 0.0074*0.0074],
    "etaP_opt":  [0.2057*0.2159, 0.357*0.0562, 0.326*0.1637, 0.0863*0.0449],
    # improvement factors for annotation
    "factors":   [8, 35, 133, 71],
}


# =============================================================================
# FIGURE
# =============================================================================

def make_qfi_figure(qfi_results=None, save_path="fig_qfi.pdf"):
    """
    Generate and save the two-panel QFI figure.

    Parameters
    ----------
    qfi_results : dict from run_all_qfi(), or None to use hardcoded values
    save_path   : output filename (.pdf and .png both saved)
    """

    # ── Load data ─────────────────────────────────────────────────────────────
    if qfi_results is not None:
        N_list, qf_afek, qf_opt, etaP_afek, etaP_opt = extract_data(qfi_results)
        factors = [round(etaP_opt[i]/etaP_afek[i]) for i in range(len(N_list))]
    else:
        print("  [fig_qfi] using hardcoded values (qfi_results not provided)")
        d        = HARDCODED
        N_list   = d["N_list"]
        qf_afek  = d["qf_afek"]
        qf_opt   = d["qf_opt"]
        etaP_afek = d["etaP_afek"]
        etaP_opt  = d["etaP_opt"]
        factors   = d["factors"]

    x     = np.arange(len(N_list))
    w     = 0.36          # bar width
    cols  = [N_COLORS[N] for N in N_list]

    # ── Figure layout ─────────────────────────────────────────────────────────
    fig, axes = plt.subplots(
        1, 2,
        figsize=(DOUBLE, DOUBLE * 0.42),
        gridspec_kw={"wspace": 0.38},
    )

    # ══════════════════════════════════════════════════════════════════════════
    # Panel (a): QFI / N²  — probe quality
    # ══════════════════════════════════════════════════════════════════════════
    ax = axes[0]

    bars_a = ax.bar(x - w/2, qf_afek, w,
                    color=AFEK_COLOR, edgecolor="white",
                    linewidth=0.4, label="Afek init.", zorder=3)
    # Add thin coloured top edge on Afek bars to identify N
    for bar, col in zip(bars_a, cols):
        bar.set_edgecolor(col)
        bar.set_linewidth(1.0)

    bars_o = ax.bar(x + w/2, qf_opt, w,
                    color=cols, edgecolor="white",
                    linewidth=0.4, alpha=OPT_ALPHA, label="Optimised", zorder=3)

    # Heisenberg limit
    ax.axhline(1.0, color="#222222", lw=0.8, ls="--", zorder=2)
    ax.text(len(N_list) - 0.55, 1.015, "Heisenberg limit ($= 1$)",
            ha="right", va="bottom", fontsize=5.5, color="#222222")

    # Annotate change arrows for each N
    for i, N in enumerate(N_list):
        dq = qf_opt[i] - qf_afek[i]
        sign = "+" if dq >= 0 else "−"
        color = "#31a354" if dq >= 0 else "#d62728"
        ymax = max(qf_afek[i], qf_opt[i])
        ax.annotate(
            f"{sign}{abs(dq)*100:.0f}pp",
            xy=(x[i], ymax + 0.01),
            ha="center", va="bottom",
            fontsize=5, color=color, fontweight="bold",
        )

    ax.set_xticks(x)
    ax.set_xticklabels([f"$N={N}$" for N in N_list])
    ax.set_ylabel(r"Probe quality $\mathcal{F}_Q / N^2$", fontsize=7.5)
    ax.set_ylim(0, 1.18)
    ax.set_yticks([0, 0.25, 0.50, 0.75, 1.00])
    ax.yaxis.set_major_formatter(ticker.FormatStrFormatter("%.2f"))

    # Shade quantum-enhanced region (above SNL ≈ 0.25 for coherent state)
    ax.axhspan(0, 0.26, alpha=0.04, color="#888888", zorder=0)
    ax.text(0.02, 0.05, "classical\nregion",
            transform=ax.transAxes, fontsize=5, color="#888888",
            va="bottom", style="italic")

    # Legend
    legend_handles = (
        [Patch(facecolor=AFEK_COLOR, edgecolor="#555555", linewidth=0.8, label="Afek init. (all $N$)")]
        + [Patch(facecolor=N_COLORS[N], edgecolor="none", alpha=OPT_ALPHA,
                 label=f"$N={N}$  Opt.")
           for N in N_list]
    )
    # legend moved to below figure

    ax.set_title("Probe quality vs Heisenberg limit", fontsize=8, pad=5)
    ax.text(-0.18, 1.06, "a", transform=ax.transAxes,
            fontsize=10, fontweight="bold", va="top")

    # ══════════════════════════════════════════════════════════════════════════
    # Panel (b): η_Σ × P_sel  — useful events per pulse (log scale)
    # ══════════════════════════════════════════════════════════════════════════
    ax = axes[1]

    bars_a2 = ax.bar(x - w/2, etaP_afek, w,
                     color=AFEK_COLOR, edgecolor="white",
                     linewidth=0.4, label="Afek init.", zorder=3)
    for bar, col in zip(bars_a2, cols):
        bar.set_edgecolor(col)
        bar.set_linewidth(1.0)

    bars_o2 = ax.bar(x + w/2, etaP_opt, w,
                     color=cols, edgecolor="white",
                     linewidth=0.4, alpha=OPT_ALPHA, label="Optimised", zorder=3)

    ax.set_yscale("log")

    # Improvement factor annotations above each opt bar
    for i, (fac, yopt) in enumerate(zip(factors, etaP_opt)):
        ax.text(x[i] + w/2, yopt * 1.1,  
                f"$\\times{fac}$",
                ha="center", va="bottom",
                fontsize=6, color=cols[i], fontweight="bold")

    # Practical threshold line: ~1 event per 1000 pulses (10 kHz laser → 10/s)
    ax.axhline(1e-3, color="#666666", lw=0.6, ls=":",  zorder=2)
    ax.text(len(N_list) - 0.55, 1.3e-3,
            "1 event / 1000 pulses",
            ha="right", va="bottom", fontsize=5, color="#666666")

    ax.set_xticks(x)
    ax.set_xticklabels([f"$N={N}$" for N in N_list])
    ax.set_ylabel(
        r"Useful events / pulse   $\eta_\Sigma \times P_\mathrm{sel}$",
        fontsize=7.5,
    )
    ax.yaxis.set_major_formatter(
        ticker.FuncFormatter(lambda y, _: f"$10^{{{int(np.log10(y))}}}$"
                             if y > 0 and y == 10**int(np.log10(y))
                             else f"{y:.0e}".replace("e-0","e-").replace("e+0",""))
    )

    # legend moved to below figure

    ax.set_title("Useful measurement events per pulse", fontsize=8, pad=5)
    ax.text(-0.18, 1.06, "b", transform=ax.transAxes,
            fontsize=10, fontweight="bold", va="top")

    # ── Save ──────────────────────────────────────────────────────────────────
    # ── Single shared legend below both panels ───────────────────────────────
    fig.legend(
        handles=legend_handles,
        loc="lower center",
        bbox_to_anchor=(0.5, -0.03),
        ncol=len(legend_handles),        # all items in one horizontal row
        handlelength=0.9,
        handletextpad=0.4,
        columnspacing=1.0,
        borderpad=0.4,
        frameon=False,
        fontsize=6.5,
    )
    fig.tight_layout(pad=0.6)
    fig.subplots_adjust(bottom=0.13)     # make room for legend below
    fig.savefig(save_path, bbox_inches="tight")
    png_path = save_path.replace(".pdf", ".png")
    fig.savefig(png_path, dpi=300, bbox_inches="tight")
    print(f"  Saved → {save_path}")
    print(f"  Saved → {png_path}")
    plt.show()
    return fig


# =============================================================================
# MAIN CALL
# =============================================================================

if __name__ == "__main__":
    # Called directly — use hardcoded values
    fig = make_qfi_figure(qfi_results=None, save_path="fig_qfi.pdf")
 
else:
    # Called from notebook — try to use qfi_results if it exists
    try:
        fig = make_qfi_figure(qfi_results=qfi_results, save_path="fig_qfi.pdf")
    except NameError:
        print("qfi_results not found — using hardcoded values")
        fig = make_qfi_figure(qfi_results=None, save_path="fig_qfi.pdf")
    